# Quantum-Enabled Watermarking with TF-QKD and Siamese ConvNeXt — Complete Pipeline (Phases 0–7)

One notebook, run top to bottom, starting from Phase 0. Phase 0 (data preparation and Siamese training) runs automatically on the first run, when the prepared images and the trained encoder are not yet in `outputs/`.

| Phase | What it does | Main outputs |
|---|---|---|
| 0 | Siamese ConvNeXt trained on attack-matched pairs | `authentication_encoder.pt`, threshold |
| 1 | Simulated TF-QKD key (BB84-style post-processing) | `K_TF_final_key.npy` |
| 2 | Key derivation (BLAKE2b) and AES-256-GCM watermark encryption | `W_enc_bits.npy`, tag |
| 3 | K_TF-only block selection, adaptive DCT embedding with the constants fixed in CONFIG | watermarked images, embedding maps |
| 4 | 19 benign/geometric attacks + 4 forgeries, ground-truth masks | attacked images, `Masks/` |
| 5 | Extraction (reference-assisted for rotation/cropping) and decryption | NC/BER per image and attack |
| 6 | Image-level authentication and region-level tamper localization | decisions, tamper maps, localization metrics |
| 7 | Security, ablation, four-watermark test, image mapping, comparison, manuscript numbers, figure exports | tables, `manuscript_numbers.json`, figure images |

**Order matters.** Run the CONFIG cell first. The embedding constants and the localization setting are fixed in CONFIG.
**Data pools.** Phases 3–6 use `outputs/PHASE0_PREPARED/test/authentic`. The optional sweep (3.8) uses `val/authentic` only.
**Reporting.** Every number in the paper comes from the CSV/JSON files this notebook writes; none is typed in by hand.

In [ ]:
# =============================================================================
# CONFIG — the only cell you normally edit
# =============================================================================
import json, random
from pathlib import Path
import numpy as np

# ---- all paths are relative to the folder that contains this notebook ----
ROOT        = Path.cwd()
DATASET_DIR = ROOT / "data" / "CASIA2"          # download CASIA 2.0 here (not included in the repository)
OUT         = ROOT / "outputs"                  # every file written by this notebook goes here
PREPARED    = OUT / "PHASE0_PREPARED"           # prepared train/val/test images
PHASE0_OUT  = OUT / "PHASE0_RESULTS"            # trained Siamese encoder
RESULTS     = OUT / "PROPOSED_SCHEME_RESULTS"
RESULTS.mkdir(parents=True, exist_ok=True)

WATERMARK_PATH   = ROOT / "watermarks" / "watermark_binary.png"      # main watermark (already binary, 32x32)
WATERMARK_PATH_2 = ROOT / "watermarks" / "watermark_binary_2.png"    # second watermark for the 30-image test in 7.3b (a synthetic pattern is used if the file is missing)
WATERMARK_ID     = "main"
GLOBAL_SEED      = 42

# ---- embedding constants (the values reported in the manuscript) ----
EMBEDDING_THRESHOLD = 28.0     # base threshold in  T = 28.0 * (0.90 + 0.65 * rho)
EMBED_MARGIN        = 15.0     # m  in  Delta = T + m + min(0.5|c1-c2|, cap)
DIFF_CAP            = 18.0     # cap
REDUNDANCY          = 3        # R  (R can be 1..4: 1024 bits * R must not exceed the 4096 blocks)

# ---- sweep / selection targets ----
TARGET_PSNR      = 41.0
MIN_CLEAN_NC     = 0.99
MIN_ATTACKED_NC  = 0.90
MAX_BENIGN_FALSE_FLAG = 0.05
FROZEN_PARAMS_FILE = RESULTS / "PHASE3_SWEEP" / "frozen_embedding_params.json"

# ---- Phase 0 pair construction ----
# RUN_MODE picks MAX_IMAGES automatically -- no manual flipping between runs:
#   "debug"  -> 200 images,  fast sanity-check run, numbers NOT for the manuscript
#   "final"  -> None (full held-out test pool), the run whose numbers go into every table
# Phase 0 (data preparation + Siamese training) runs automatically when the prepared images or the trained encoder are not found in outputs/.
RUN_PHASE0 = not ((PREPARED / "test" / "authentic").exists() and (PHASE0_OUT / "deployment" / "authentication_encoder.pt").exists())
RUN_MODE = "final"            # "final" = full held-out test pool (all tables); "debug" = 200-image quick check only
MAX_IMAGES = {"debug": 200, "final": None}[RUN_MODE]
GENUINE_DIFF_FRACTION = 0.0    # share of genuine pairs made of two DIFFERENT images (pipeline 0.2a). 0 = off (see Phase 0 note)

# ---- key generation ----
# "reproducible": fixed master seed 42 -> the SAME K_TF on every run (needed to reproduce the paper's numbers).
#                 Anyone with this code can regenerate that key, so it is a TEST key only.
# "fresh"       : master seed drawn from the OS random source (secrets) -> a new, unpredictable K_TF.
KEY_MODE = "reproducible"

# ---- Phase 6 ----
LOC_ALPHA, LOC_MIN_RATE = 1e-8, 0.4   # localization setting reported in the manuscript
VAL_FRACTION = 0.30            # identities used to calibrate thresholds; the rest are the held-out test identities

print("ROOT:", ROOT)

In [ ]:
# =============================================================================
# Shared imports and helpers (used by several phases)
# =============================================================================
import os, io, re, math, time, json, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import convolve2d
from skimage.metrics import structural_similarity as _ssim_fn

_noprint = lambda *a, **k: None
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300})

random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED)

def seed_everything(seed=GLOBAL_SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

def rgb2bgr(a): return np.ascontiguousarray(a[:, :, ::-1])
bgr2rgb = rgb2bgr

## Shared attack library
The 19 benign/geometric attacks and the 4 forgery functions, defined once because Phase 0 (impostor pairs) and Phase 4 (attack simulation) both use them.

In [ ]:
# =============================================================================
# PHASE 4 : ATTACK SIMULATION
# CELL 2 : Attack Functions
# =============================================================================

# ------------------------------------------------------------------
# JPEG Compression
# ------------------------------------------------------------------

def jpeg_compression(img, quality=50):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, enc = cv2.imencode(".jpg", img, encode_param)
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)


# ------------------------------------------------------------------
# Gaussian Noise
# ------------------------------------------------------------------

def gaussian_noise(img, sigma=10, rng=None):
    rng = rng if rng is not None else np.random.default_rng()
    noise = rng.normal(0, sigma, img.shape)
    out = img.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)


# ------------------------------------------------------------------
# Salt & Pepper Noise
# ------------------------------------------------------------------

def salt_pepper_noise(img, amount=0.02, rng=None):

    rng = rng if rng is not None else np.random.default_rng()
    out = img.copy()

    num_salt = int(amount * img.size * 0.5)
    coords = [rng.integers(0, i, num_salt) for i in img.shape[:2]]
    out[coords[0], coords[1]] = 255

    num_pepper = int(amount * img.size * 0.5)
    coords = [rng.integers(0, i, num_pepper) for i in img.shape[:2]]
    out[coords[0], coords[1]] = 0

    return out


# ------------------------------------------------------------------
# Speckle Noise
# ------------------------------------------------------------------

def speckle_noise(img, rng=None):

    rng = rng if rng is not None else np.random.default_rng()
    noise = rng.standard_normal(img.shape)

    out = img.astype(np.float32)
    out = out + out * noise * 0.15

    return np.clip(out, 0, 255).astype(np.uint8)


# ------------------------------------------------------------------
# Gaussian Blur
# ------------------------------------------------------------------

def gaussian_blur(img):
    return cv2.GaussianBlur(img, (5,5), 1.0)


# ------------------------------------------------------------------
# Median Filter
# ------------------------------------------------------------------

def median_filter(img):
    return cv2.medianBlur(img, 3)


# ------------------------------------------------------------------
# Mean Filter
# ------------------------------------------------------------------

def mean_filter(img):
    return cv2.blur(img, (3,3))


# ------------------------------------------------------------------
# Bilateral Filter
# ------------------------------------------------------------------

def bilateral_filter(img):
    return cv2.bilateralFilter(img, 9, 75, 75)


# ------------------------------------------------------------------
# Sharpening
# ------------------------------------------------------------------

def sharpening(img):

    kernel = np.array([
        [0,-1,0],
        [-1,5,-1],
        [0,-1,0]
    ])

    return cv2.filter2D(img, -1, kernel)


# ------------------------------------------------------------------
# Unsharp Masking
# ------------------------------------------------------------------

def unsharp_mask(img):

    blur = cv2.GaussianBlur(img,(5,5),1.0)

    return cv2.addWeighted(img,1.5,blur,-0.5,0)


# ------------------------------------------------------------------
# Histogram Equalization
# ------------------------------------------------------------------

def histogram_equalization(img):

    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

    ycrcb[:,:,0] = cv2.equalizeHist(ycrcb[:,:,0])

    return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)


# ------------------------------------------------------------------
# CLAHE
# ------------------------------------------------------------------

def clahe(img):

    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    ycrcb[:,:,0] = clahe.apply(ycrcb[:,:,0])

    return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)


# ------------------------------------------------------------------
# Rotation
# ------------------------------------------------------------------

def rotation(img, angle=5):

    h,w = img.shape[:2]

    M = cv2.getRotationMatrix2D(
        (w//2,h//2),
        angle,
        1.0
    )

    return cv2.warpAffine(img,M,(w,h))


# ------------------------------------------------------------------
# Cropping
# ------------------------------------------------------------------

def cropping(img, crop_ratio=0.10):

    h,w = img.shape[:2]

    dx = int(w*crop_ratio)
    dy = int(h*crop_ratio)

    crop = img[
        dy:h-dy,
        dx:w-dx
    ]

    return cv2.resize(
        crop,
        (w,h),
        interpolation=cv2.INTER_LINEAR
    )


# ------------------------------------------------------------------
# Resize
# ------------------------------------------------------------------

def resize_attack(img, scale=0.75):

    h,w = img.shape[:2]

    small = cv2.resize(
        img,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_AREA
    )

    return cv2.resize(
        small,
        (w,h),
        interpolation=cv2.INTER_LINEAR
    )


# ------------------------------------------------------------------
# Brightness
# ------------------------------------------------------------------

def brightness(img, beta=25):

    out = img.astype(np.int16)

    out = out + beta

    return np.clip(out,0,255).astype(np.uint8)


# ------------------------------------------------------------------
# Contrast
# ------------------------------------------------------------------

def contrast(img, alpha=1.25):

    out = img.astype(np.float32)

    out = out * alpha

    return np.clip(out,0,255).astype(np.uint8)


# ------------------------------------------------------------------
# Gamma
# ------------------------------------------------------------------

def gamma_correction(img, gamma=1.2):

    invGamma = 1.0/gamma

    table = np.array([
        ((i/255.0)**invGamma)*255
        for i in np.arange(256)
    ]).astype("uint8")

    return cv2.LUT(img,table)


# ------------------------------------------------------------------
# Motion Blur
# ------------------------------------------------------------------

def motion_blur(img, kernel_size=9):

    kernel = np.zeros((kernel_size,kernel_size))

    kernel[int((kernel_size-1)/2),:] = np.ones(kernel_size)

    kernel = kernel/kernel_size

    return cv2.filter2D(img,-1,kernel)

In [ ]:
# =============================================================================
# PHASE 4 : ATTACK SIMULATION
# CELL 2B (NEW) : Malicious Content-Editing Attack Functions
# =============================================================================
# Each function returns (attacked_img, mask), where mask is a single-
# channel uint8 array the same H x W as img: 255 where pixels were
# edited, 0 elsewhere. All region sizes/positions are drawn from the
# seeded `rng` passed in, for the same reproducibility guarantee as
# the existing 19 attacks.
# =============================================================================

def _random_region(h, w, rng, area_ratio_range=(0.10, 0.15)):
    """Pick a random rectangular region covering roughly 10-15% of
    the image area, sized independently of aspect ratio so the region
    is not always square."""
    area_ratio = rng.uniform(*area_ratio_range)
    region_area = area_ratio * h * w

    aspect = rng.uniform(0.6, 1.6)
    rh = int(np.sqrt(region_area * aspect))
    rw = int(np.sqrt(region_area / aspect))

    rh = min(max(rh, 8), h - 1)
    rw = min(max(rw, 8), w - 1)

    y0 = int(rng.integers(0, h - rh))
    x0 = int(rng.integers(0, w - rw))

    return y0, x0, rh, rw


def copy_move_forgery(img, rng):
    """Copies a region from elsewhere in the SAME image and pastes it
    into a different, non-overlapping location -- the classic
    copy-move forgery pattern."""
    h, w = img.shape[:2]
    out = img.copy()
    mask = np.zeros((h, w), dtype=np.uint8)

    src_y, src_x, rh, rw = _random_region(h, w, rng)
    patch = img[src_y:src_y + rh, src_x:src_x + rw].copy()

    # Find a destination that does not overlap the source region;
    # retry a bounded number of times, fall back to whatever the last
    # draw gives if unlucky (still non-degenerate, just possibly close).
    for _ in range(20):
        dst_y = int(rng.integers(0, h - rh))
        dst_x = int(rng.integers(0, w - rw))
        overlap = not (
            dst_x + rw <= src_x or dst_x >= src_x + rw or
            dst_y + rh <= src_y or dst_y >= src_y + rh
        )
        if not overlap:
            break

    out[dst_y:dst_y + rh, dst_x:dst_x + rw] = patch
    mask[dst_y:dst_y + rh, dst_x:dst_x + rw] = 255

    return out, mask


def image_splicing(img, source_img, rng):
    """Pastes a region from a DIFFERENT source image directly (hard
    paste, no blending) into the target image -- simulates inserting
    foreign content."""
    h, w = img.shape[:2]
    out = img.copy()
    mask = np.zeros((h, w), dtype=np.uint8)

    sh, sw = source_img.shape[:2]
    rh = min(int(0.12 * h), sh - 1)
    rw = min(int(0.12 * w), sw - 1)
    rh, rw = max(rh, 8), max(rw, 8)

    src_y = int(rng.integers(0, sh - rh))
    src_x = int(rng.integers(0, sw - rw))
    patch = source_img[src_y:src_y + rh, src_x:src_x + rw]

    dst_y = int(rng.integers(0, h - rh))
    dst_x = int(rng.integers(0, w - rw))

    out[dst_y:dst_y + rh, dst_x:dst_x + rw] = patch
    mask[dst_y:dst_y + rh, dst_x:dst_x + rw] = 255

    return out, mask


def object_removal(img, rng):
    """Removes a region and fills it using inpainting, simulating
    deletion of content from the scene."""
    h, w = img.shape[:2]
    y0, x0, rh, rw = _random_region(h, w, rng, area_ratio_range=(0.06, 0.10))

    mask = np.zeros((h, w), dtype=np.uint8)
    mask[y0:y0 + rh, x0:x0 + rw] = 255

    out = cv2.inpaint(img, mask, inpaintRadius=7, flags=cv2.INPAINT_TELEA)

    return out, mask


def region_replacement(img, source_img, rng):
    """Replaces a region with content from another source image, using
    a feathered (alpha-blended) edge rather than image_splicing's hard
    paste -- kept as a distinct attack type since the two stress the
    watermark/forensic signals differently (a soft edge is harder for
    blind forensic analysis to catch, but the embedded coefficients
    inside the region are disturbed the same way either way)."""
    h, w = img.shape[:2]
    out = img.copy().astype(np.float32)
    mask = np.zeros((h, w), dtype=np.uint8)

    sh, sw = source_img.shape[:2]
    rh = min(int(0.13 * h), sh - 1)
    rw = min(int(0.13 * w), sw - 1)
    rh, rw = max(rh, 8), max(rw, 8)

    src_y = int(rng.integers(0, sh - rh))
    src_x = int(rng.integers(0, sw - rw))
    patch = source_img[src_y:src_y + rh, src_x:src_x + rw].astype(np.float32)

    dst_y = int(rng.integers(0, h - rh))
    dst_x = int(rng.integers(0, w - rw))

    # Feathered alpha mask: 1.0 in the interior, tapering to 0 at the
    # region border, so the blend is smooth rather than a hard edge.
    alpha = np.ones((rh, rw), dtype=np.float32)
    feather = max(2, min(rh, rw) // 8)
    for i in range(feather):
        v = (i + 1) / feather
        alpha[i, :] *= v
        alpha[-(i + 1), :] *= v
        alpha[:, i] *= v
        alpha[:, -(i + 1)] *= v
    alpha3 = alpha[:, :, None]

    region = out[dst_y:dst_y + rh, dst_x:dst_x + rw]
    out[dst_y:dst_y + rh, dst_x:dst_x + rw] = (
        alpha3 * patch + (1 - alpha3) * region
    )

    mask[dst_y:dst_y + rh, dst_x:dst_x + rw] = 255

    return np.clip(out, 0, 255).astype(np.uint8), mask

# PHASE 0 — Siamese ConvNeXt authentication model

## Steps 0.1–0.2 — dataset preparation and preprocessing

In [ ]:
if RUN_PHASE0:
    # =============================================================================
    # STEP 0.1 (CORRECTED) : Dataset Organization and Group-Aware Splitting
    #   0.1.1 Collect authentic image dataset
    #   0.1.2 Collect corresponding tampered image dataset
    #   0.1.3 Remove corrupted and duplicate image samples
    #   0.1.4 Organize the dataset into a unified directory structure
    #   0.1.5 Split into Training / Validation / Testing subsets
    #   0.1.6 Validate dataset integrity and class distribution
    # =============================================================================

    import hashlib
    import shutil
    from pathlib import Path
    from collections import Counter, defaultdict
    import re

    from PIL import Image
    import numpy as np
    from sklearn.model_selection import train_test_split

    # ------------------------------------------------------------
    # Configuration (paths come from the CONFIG cell)
    # ------------------------------------------------------------
    DATASET_ROOT = DATASET_DIR
    OUTPUT_ROOT  = PREPARED

    AUTHENTIC_FOLDER_NAME = "Au"
    TAMPERED_FOLDER_NAME  = "Tp"

    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    IMG_SIZE = 512  # unified prepared-image size

    TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
    SPLIT_SEED = 42

    assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-6, \
        "Train/val/test ratios must sum to 1.0"

    # ------------------------------------------------------------
    # Substep 0.1.1 / 0.1.2 : Collect authentic + tampered image paths
    # ------------------------------------------------------------
    auth_dir = DATASET_ROOT / AUTHENTIC_FOLDER_NAME
    tamp_dir = DATASET_ROOT / TAMPERED_FOLDER_NAME

    if not auth_dir.exists():
        raise FileNotFoundError(f"Authentic folder not found at {auth_dir}.")
    if not tamp_dir.exists():
        raise FileNotFoundError(f"Tampered folder not found at {tamp_dir}.")

    def collect_image_paths(folder):
        return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXTS)

    authentic_paths_raw = collect_image_paths(auth_dir)
    tampered_paths_raw  = collect_image_paths(tamp_dir)

    print("Substep 0.1.1 — Authentic images collected :", len(authentic_paths_raw))
    print("Substep 0.1.2 — Tampered images collected  :", len(tampered_paths_raw))

    # ------------------------------------------------------------
    # Substep 0.1.3 : Remove corrupted and duplicate image samples
    # ------------------------------------------------------------
    def is_readable_image(path):
        try:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                im.convert("RGB").load()
            return True
        except Exception:
            return False

    def file_md5(path, chunk_size=1 << 20):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(chunk_size), b""):
                h.update(chunk)
        return h.hexdigest()

    def clean_image_list(paths, label):
        corrupted, seen_hashes, duplicates, kept = [], {}, [], []
        for p in paths:
            if not is_readable_image(p):
                corrupted.append(p)
                continue
            digest = file_md5(p)
            if digest in seen_hashes:
                duplicates.append(p)
                continue
            seen_hashes[digest] = p
            kept.append(p)
        print(f"  [{label}] corrupted removed : {len(corrupted)}")
        print(f"  [{label}] duplicates removed: {len(duplicates)}")
        print(f"  [{label}] kept             : {len(kept)}")
        return kept, corrupted, duplicates

    print("\nSubstep 0.1.3 — Scanning for corrupted / duplicate files "
          "(this reads every file once; can take a while on large datasets)...")
    authentic_paths, _, _ = clean_image_list(authentic_paths_raw, "authentic")
    tampered_paths, _, _  = clean_image_list(tampered_paths_raw, "tampered")
else:
    print("STEP 0.1: skipped (RUN_PHASE0 = False); using the saved Phase 0 results.")


In [ ]:
# =============================================================================
# STEP 0.2 : Image Preprocessing and Augmentation
#   0.2.1 Load the training image dataset
#   0.2.2 Resize all images to the ConvNeXt input resolution
#   0.2.3 Normalize image pixel values
#   0.2.4 Apply data augmentation for robust feature learning
#   0.2.5 Convert images into tensor representation
#   0.2.6 Generate mini-batches for model training
# =============================================================================

import torchvision.transforms as T

CONVNEXT_INPUT_SIZE = 224          # Substep 0.2.2
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # Substep 0.2.3
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Substeps 0.2.2 - 0.2.5, training-time (includes augmentation, 0.2.4)
train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((CONVNEXT_INPUT_SIZE, CONVNEXT_INPUT_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    T.RandomRotation(degrees=5),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Substeps 0.2.2, 0.2.3, 0.2.5, eval-time (no augmentation — augmentation is
# a training-only regularizer; applying it at eval time would make
# validation/test metrics non-deterministic and is intentionally avoided)
eval_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((CONVNEXT_INPUT_SIZE, CONVNEXT_INPUT_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

BATCH_SIZE = 16   # Substep 0.2.6 (mini-batch generation is wired in Step 0.3
                   # via DataLoader, using this batch size)

print("Step 0.2 complete.")
print("  Input resolution :", CONVNEXT_INPUT_SIZE)
print("  Batch size        :", BATCH_SIZE)
print("  Train transform   :", train_transform)
print("  Eval transform    :", eval_transform)

## Phase 0 — Siamese ConvNeXt (attack-matched)
**Step 0.3 pair construction**
- genuine pairs: an authentic image and the same image after a benign operation (JPEG, blur, noise, brightness, resize, mild rotation);
- impostor pairs: an authentic image and the same image after one of the four forgeries used in Phase 4 (copy-move, splicing, object removal, region replacement);
- validation pairs are generated once and kept fixed; every random source is seeded.

Two *different* authentic images can be added as genuine pairs through `GENUINE_DIFF_FRACTION`; it is off by default, because at deployment the model only compares an image with its own reference.

In [ ]:
if RUN_PHASE0:
    # =============================================================================
    # STEP 0.3 : Attack-matched Siamese pair construction
    # =============================================================================
    import random
    import numpy as np
    import cv2
    import torch
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image

    GENUINE_LABEL, IMPOSTOR_LABEL = 0.0, 1.0
    BENIGN_OPS = ["jpeg", "gaussian_blur", "gaussian_noise", "rotation", "brightness", "median_blur", "resize"]
    FORGERY_OPS = ["copy_move", "splicing", "object_removal", "region_replacement"]

    def apply_benign(img, rng):
        op = BENIGN_OPS[int(rng.integers(0, len(BENIGN_OPS)))]
        out = img.copy()
        if op == "jpeg":
            q = int(rng.integers(30, 90))
            ok, enc = cv2.imencode(".jpg", cv2.cvtColor(out, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, q])
            out = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        elif op == "gaussian_blur":
            k = int(rng.choice([3, 5, 7])); out = cv2.GaussianBlur(out, (k, k), 0)
        elif op == "gaussian_noise":
            out = np.clip(out.astype(np.float32) + rng.normal(0, float(rng.uniform(5, 15)), out.shape), 0, 255).astype(np.uint8)
        elif op == "rotation":
            h, w = out.shape[:2]
            M = cv2.getRotationMatrix2D((w / 2, h / 2), float(rng.uniform(-5, 5)), 1.0)
            out = cv2.warpAffine(out, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        elif op == "brightness":
            out = np.clip(out.astype(np.float32) + float(rng.uniform(-20, 20)), 0, 255).astype(np.uint8)
        elif op == "median_blur":
            out = cv2.medianBlur(out, int(rng.choice([3, 5])))
        else:  # resize
            s = float(rng.uniform(0.6, 0.9)); h, w = out.shape[:2]
            out = cv2.resize(cv2.resize(out, None, fx=s, fy=s, interpolation=cv2.INTER_AREA), (w, h), interpolation=cv2.INTER_LINEAR)
        return out

    def apply_forgery(img, source, rng):
        op = FORGERY_OPS[int(rng.integers(0, len(FORGERY_OPS)))]
        if op == "copy_move":          out, _ = copy_move_forgery(img, rng)
        elif op == "splicing":         out, _ = image_splicing(img, source, rng)
        elif op == "object_removal":   out, _ = object_removal(img, rng)
        else:                          out, _ = region_replacement(img, source, rng)
        return out

    class SiamesePairDatasetV3(Dataset):
        """genuine (label 0): authentic vs benign-degraded SELF   [optionally a fraction: two different authentic images]
           impostor (label 1): authentic vs forged SELF (the four Phase 4 forgeries).
           fixed=True -> item i is always the same pair (used for validation); fixed=False -> a fresh pair every epoch."""
        def __init__(self, authentic_dir, transform, pairs_per_epoch=2000, seed=42, fixed=False, genuine_diff_fraction=0.0):
            self.paths = sorted(Path(authentic_dir).glob("*.png"))
            if len(self.paths) < 2:
                raise ValueError(f"Need at least 2 authentic images in {authentic_dir}")
            self.transform, self.n, self.seed, self.fixed = transform, pairs_per_epoch, seed, fixed
            self.rng = np.random.default_rng(seed)
            sched_rng = random.Random(seed)
            n_imp = self.n // 2
            n_gen = self.n - n_imp
            n_diff = int(round(n_gen * genuine_diff_fraction))
            self.schedule = ["impostor"] * n_imp + ["genuine_diff"] * n_diff + ["genuine_benign"] * (n_gen - n_diff)
            sched_rng.shuffle(self.schedule)
            self.linked_authentic_paths = self.paths           # (kept so the Phase 0 validation printout still works)
            self.tampered_paths = []

        def __len__(self): return self.n

        def _load(self, p): return np.array(Image.open(p).convert("RGB"))

        def __getitem__(self, idx):
            rng = np.random.default_rng([self.seed, idx]) if self.fixed else self.rng
            kind = self.schedule[idx]
            i = int(rng.integers(0, len(self.paths)))
            a = self._load(self.paths[i])
            if kind == "genuine_diff":
                j = (i + 1 + int(rng.integers(0, len(self.paths) - 1))) % len(self.paths)
                b, label = self._load(self.paths[j]), GENUINE_LABEL
            elif kind == "genuine_benign":
                b, label = apply_benign(a, rng), GENUINE_LABEL
            else:
                j = (i + 1 + int(rng.integers(0, len(self.paths) - 1))) % len(self.paths)
                b, label = apply_forgery(a, self._load(self.paths[j]), rng), IMPOSTOR_LABEL
            return self.transform(a), self.transform(b), torch.tensor(label, dtype=torch.float32), kind

    def pair_collate(batch):
        return (torch.stack([b[0] for b in batch]), torch.stack([b[1] for b in batch]),
                torch.stack([b[2] for b in batch]), [b[3] for b in batch])

    train_dataset = SiamesePairDatasetV3(OUTPUT_ROOT / "train/authentic", train_transform, 2000, seed=42, fixed=False,
                                         genuine_diff_fraction=GENUINE_DIFF_FRACTION)
    val_dataset   = SiamesePairDatasetV3(OUTPUT_ROOT / "val/authentic", eval_transform, 500, seed=123, fixed=True,
                                         genuine_diff_fraction=GENUINE_DIFF_FRACTION)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, collate_fn=pair_collate)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=pair_collate)

    from collections import Counter
    for name, ds in (("train", train_dataset), ("val", val_dataset)):
        c = Counter(ds.schedule)
        print(f"  [{name}] pairs={len(ds)}  genuine_benign={c['genuine_benign']}  genuine_diff={c['genuine_diff']}  impostor={c['impostor']}")
    print("Step 0.3 complete (attack-matched pairs).")
else:
    print("STEP 0.3: skipped (RUN_PHASE0 = False); using the saved Phase 0 results.")


## Steps 0.4–0.8 — model, training, threshold, export, evaluation

In [ ]:
if RUN_PHASE0:
    # =============================================================================
    # STEP 0.4 : Siamese ConvNeXt Model Initialization
    #   0.4.1 Load the ImageNet-pretrained ConvNeXt backbone
    #   0.4.2 Construct the dual-branch Siamese architecture
    #   0.4.3 Share network parameters between both branches
    #   0.4.4 Initialize the projection head for feature embedding
    #   0.4.5 Configure the embedding dimension
    #   0.4.6 Initialize model hyperparameters
    # =============================================================================

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

    siamese_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", siamese_device)

    # Substep 0.4.6 : model hyperparameters, collected in one place so they can
    # be serialized as-is in Step 0.7 (deployment config export)
    MODEL_CONFIG = {
        "embedding_dim": 128,          # Substep 0.4.5
        "contrastive_margin": 1.0,
        "learning_rate": 3e-4,
        "weight_decay": 1e-4,
        "num_epochs": 30,
        "batch_size": BATCH_SIZE,
        "lr_scheduler": "cosine_annealing",
        "lr_eta_min": 1e-6,
        "optimizer": "AdamW",
        "backbone": "convnext_tiny (ImageNet pretrained)",
        "input_size": CONVNEXT_INPUT_SIZE,
        "seed": 42,
    }

    class SiameseConvNeXt(nn.Module):
        """Dual-branch Siamese network with a single SHARED ConvNeXt-Tiny
        backbone (Substeps 0.4.1-0.4.3) and a learned projection head
        (Substep 0.4.4) producing an L2-normalized embedding of configurable
        dimension (Substep 0.4.5)."""
        def __init__(self, embedding_dim=128):
            super().__init__()
            weights = ConvNeXt_Tiny_Weights.DEFAULT          # 0.4.1
            base = convnext_tiny(weights=weights)
            self.backbone = base.features                     # shared encoder (0.4.3)
            self.pool = base.avgpool
            self.projection = nn.Sequential(                  # projection head (0.4.4)
                nn.Linear(768, 256),
                nn.ReLU(inplace=True),
                nn.Linear(256, embedding_dim),                # embedding dim (0.4.5)
            )

        def forward_one(self, x):
            x = self.backbone(x)
            x = self.pool(x)
            x = torch.flatten(x, 1)
            x = self.projection(x)
            return F.normalize(x, p=2, dim=1)

        def forward(self, x1, x2):                            # dual-branch (0.4.2)
            return self.forward_one(x1), self.forward_one(x2)

    torch.manual_seed(MODEL_CONFIG["seed"])
    siamese_model = SiameseConvNeXt(embedding_dim=MODEL_CONFIG["embedding_dim"]).to(siamese_device)

    n_params = sum(p.numel() for p in siamese_model.parameters())
    print("Step 0.4 complete.")
    print("  Model config:", MODEL_CONFIG)
    print("  Total parameters:", f"{n_params:,}")
else:
    print("STEP 0.4: skipped (RUN_PHASE0 = False); using the saved Phase 0 results.")


In [ ]:
if RUN_PHASE0:
    # =============================================================================
    # STEP 0.5 : Siamese ConvNeXt Model Training
    #   0.5.1 Perform forward propagation through the Siamese ConvNeXt network
    #   0.5.2 Extract deep semantic feature embeddings
    #   0.5.3 Compute the embedding distance between image pairs
    #   0.5.4 Optimize the embedding space using Contrastive Loss
    #   0.5.5 Update model parameters using the AdamW optimizer
    #   0.5.6 Adjust the learning rate using the scheduler
    #   0.5.7 Save the best-performing model during training
    # =============================================================================

    import time
    import torch

    METRICS_ROOT = OUTPUT_ROOT.parent / "PHASE0_RESULTS" / "training"
    METRICS_ROOT.mkdir(parents=True, exist_ok=True)
    BEST_CHECKPOINT_PATH = METRICS_ROOT / "siamese_convnext_BEST.pt"
    LAST_CHECKPOINT_PATH = METRICS_ROOT / "siamese_convnext_LAST.pt"

    class ContrastiveLoss(nn.Module):
        """Substep 0.5.4 — contrastive loss on Euclidean distance between
        L2-normalized embeddings (bounded in [0, 2])."""
        def __init__(self, margin=1.0):
            super().__init__()
            self.margin = margin

        def forward(self, e1, e2, label):
            dist = F.pairwise_distance(e1, e2)                       # 0.5.3
            loss_genuine  = (1 - label) * dist.pow(2)
            loss_impostor = label * torch.clamp(self.margin - dist, min=0.0).pow(2)
            return 0.5 * (loss_genuine + loss_impostor).mean(), dist

    criterion = ContrastiveLoss(margin=MODEL_CONFIG["contrastive_margin"])
    optimizer = torch.optim.AdamW(                                    # 0.5.5
        siamese_model.parameters(),
        lr=MODEL_CONFIG["learning_rate"],
        weight_decay=MODEL_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(           # 0.5.6
        optimizer, T_max=MODEL_CONFIG["num_epochs"], eta_min=MODEL_CONFIG["lr_eta_min"],
    )

    def run_epoch(loader, train_mode):
        siamese_model.train(mode=train_mode)
        running_loss = 0.0
        n_samples = 0
        with torch.set_grad_enabled(train_mode):
            for img_a, img_b, label, _kinds in loader:
                img_a, img_b, label = img_a.to(siamese_device), img_b.to(siamese_device), label.to(siamese_device)
                if train_mode:
                    optimizer.zero_grad()
                e1, e2 = siamese_model(img_a, img_b)                  # 0.5.1 forward prop, 0.5.2 embeddings
                loss, _ = criterion(e1, e2, label)
                if train_mode:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(siamese_model.parameters(), max_norm=1.0)
                    optimizer.step()
                running_loss += loss.item() * img_a.size(0)
                n_samples += img_a.size(0)
        return running_loss / n_samples

    train_loss_history, val_loss_history = [], []
    best_val_loss = float("inf")
    best_epoch = -1
    training_start_time = time.perf_counter()

    for epoch in range(1, MODEL_CONFIG["num_epochs"] + 1):
        epoch_train_loss = run_epoch(train_loader, train_mode=True)
        scheduler.step()
        epoch_val_loss = run_epoch(val_loader, train_mode=False)

        train_loss_history.append(epoch_train_loss)
        val_loss_history.append(epoch_val_loss)

        # Substep 0.5.7 : checkpoint the model whenever validation loss improves
        improved = epoch_val_loss < best_val_loss
        if improved:
            best_val_loss = epoch_val_loss
            best_epoch = epoch
            torch.save({
                "epoch": epoch,
                "model_state_dict": siamese_model.state_dict(),
                "val_loss": best_val_loss,
                "config": MODEL_CONFIG,
            }, BEST_CHECKPOINT_PATH)

        current_lr = scheduler.get_last_lr()[0]
        flag = "  <- best so far" if improved else ""
        print(f"Epoch {epoch}/{MODEL_CONFIG['num_epochs']}  "
              f"train={epoch_train_loss:.4f}  val={epoch_val_loss:.4f}  lr={current_lr:.2e}{flag}")

    training_time_sec = time.perf_counter() - training_start_time

    torch.save({
        "epoch": MODEL_CONFIG["num_epochs"],
        "model_state_dict": siamese_model.state_dict(),
        "val_loss": epoch_val_loss,
        "config": MODEL_CONFIG,
    }, LAST_CHECKPOINT_PATH)

    print("\nStep 0.5 complete.")
    print(f"  Best epoch          : {best_epoch}  (val_loss={best_val_loss:.4f})")
    print(f"  Best checkpoint     : {BEST_CHECKPOINT_PATH}")
    print(f"  Final-epoch checkpoint (for reference): {LAST_CHECKPOINT_PATH}")
    print(f"  Total training time : {training_time_sec:.1f} sec")
else:
    print("STEP 0.5: skipped (RUN_PHASE0 = False); using the saved Phase 0 results.")


In [ ]:
if RUN_PHASE0:
    # =============================================================================
    # STEP 0.6 : Model Validation and Threshold Optimization
    #   0.6.1 Generate validation feature embeddings
    #   0.6.2 Compute pairwise embedding distances
    #   0.6.3 Determine the optimal authentication threshold
    #   0.6.4 Validate authentication performance
    #   0.6.5 Select the best-performing authentication model
    # =============================================================================

    import numpy as np
    import torch
    from sklearn.metrics import accuracy_score, f1_score

    # Substep 0.6.5 : explicitly select the BEST checkpoint saved in Step 0.5
    # (not just whatever is left in memory after the training loop ends)
    best_ckpt = torch.load(BEST_CHECKPOINT_PATH, map_location=siamese_device)
    siamese_model.load_state_dict(best_ckpt["model_state_dict"])
    siamese_model.eval()
    print(f"Substep 0.6.5 — Loaded best checkpoint from epoch {best_ckpt['epoch']} "
          f"(val_loss={best_ckpt['val_loss']:.4f}) for threshold selection and evaluation.")

    # Substeps 0.6.1 / 0.6.2 : embeddings + pairwise distances on the validation set
    all_distances, all_labels = [], []
    with torch.no_grad():
        for img_a, img_b, label, _kinds in val_loader:
            img_a, img_b = img_a.to(siamese_device), img_b.to(siamese_device)
            e1, e2 = siamese_model(img_a, img_b)
            dist = F.pairwise_distance(e1, e2)
            all_distances.extend(dist.cpu().numpy().tolist())
            all_labels.extend(label.numpy().tolist())

    all_distances = np.array(all_distances)
    all_labels = np.array(all_labels)

    # Substep 0.6.3 : optimal threshold = point minimizing |FAR - FRR| (EER point)
    thresholds = np.linspace(all_distances.min(), all_distances.max(), 400)
    best_threshold, best_eer_gap, best_eer_value = thresholds[0], 1.0, 1.0
    for t in thresholds:
        preds = (all_distances >= t).astype(int)
        far = np.mean((preds == 1) & (all_labels == 0))
        frr = np.mean((preds == 0) & (all_labels == 1))
        gap = abs(far - frr)
        if gap < best_eer_gap:
            best_eer_gap, best_threshold = gap, t
            best_eer_value = (far + frr) / 2.0  # actual EER: average of FAR/FRR at their closest point

    siamese_auth_threshold = float(best_threshold)

    # Substep 0.6.4 : validate authentication performance at that threshold
    final_preds = (all_distances >= siamese_auth_threshold).astype(int)
    val_accuracy = accuracy_score(all_labels, final_preds)
    val_f1 = f1_score(all_labels, final_preds, zero_division=0)

    print("Step 0.6 complete.")
    print(f"  Optimal threshold (EER point) : {siamese_auth_threshold:.4f}")
    print(f"  Validation accuracy @ threshold: {val_accuracy:.4f}")
    print(f"  Validation F1 @ threshold      : {val_f1:.4f}")
else:
    print("STEP 0.6: skipped (RUN_PHASE0 = False); using the saved Phase 0 results.")


In [ ]:
if RUN_PHASE0:
    # =============================================================================
    # STEP 0.7 : Authentication Model Deployment Preparation
    #   0.7.1 Extract the shared ConvNeXt feature encoder
    #   0.7.2 Preserve the learned feature extraction parameters
    #   0.7.3 Export the authentication encoder for deployment
    #   0.7.4 Save the optimized authentication threshold
    #   0.7.5 Save the model configuration and hyperparameters
    # =============================================================================

    import json
    import torch

    DEPLOY_ROOT = OUTPUT_ROOT.parent / "PHASE0_RESULTS" / "deployment"
    DEPLOY_ROOT.mkdir(parents=True, exist_ok=True)

    # Substeps 0.7.1 / 0.7.2 : extract just the shared encoder (backbone + pool +
    # projection head) — NOT the full Siamese wrapper — since a deployed
    # authenticator only ever needs forward_one(), never the paired forward().
    encoder_state = {
        "backbone": siamese_model.backbone.state_dict(),
        "pool": siamese_model.pool.state_dict(),
        "projection": siamese_model.projection.state_dict(),
    }

    # Substep 0.7.3 : export as a standalone deployment artifact
    encoder_export_path = DEPLOY_ROOT / "authentication_encoder.pt"
    torch.save({
        "encoder_state": encoder_state,
        "embedding_dim": MODEL_CONFIG["embedding_dim"],
        "source_checkpoint_epoch": best_ckpt["epoch"],
    }, encoder_export_path)

    # Substep 0.7.4 : save the optimized threshold as its own artifact
    threshold_export_path = DEPLOY_ROOT / "auth_threshold.json"
    with open(threshold_export_path, "w") as f:
        json.dump({
            "siamese_auth_threshold": siamese_auth_threshold,
            "eer_gap_at_threshold": float(best_eer_gap),
            "selected_from_epoch": best_ckpt["epoch"],
        }, f, indent=2)

    # Substep 0.7.5 : save full model configuration / hyperparameters
    config_export_path = DEPLOY_ROOT / "model_config.json"
    with open(config_export_path, "w") as f:
        json.dump(MODEL_CONFIG, f, indent=2)

    print("Step 0.7 complete. Deployment artifacts written to:", DEPLOY_ROOT)
    print("  Encoder   :", encoder_export_path.name)
    print("  Threshold :", threshold_export_path.name)
    print("  Config    :", config_export_path.name)
else:
    print("STEP 0.7: skipped (RUN_PHASE0 = False); using the saved Phase 0 results.")


In [ ]:
if RUN_PHASE0:
    # =============================================================================
    # STEP 0.8 : Authentication Model Performance Evaluation
    #   0.8.1 Accuracy, Precision, Recall, F1-Score
    #   0.8.2 ROC-AUC, PR-AUC, MCC, Balanced Accuracy, Cohen's Kappa, FAR, FRR, EER
    #   0.8.3 Confusion Matrix, ROC Curve, Precision-Recall Curve
    #   0.8.4 Inference time, model parameters, FLOPs, memory utilization
    #   0.8.5 Training and validation performance plots
    #   0.8.6 Save trained encoder, evaluation metrics, plots, performance reports
    # =============================================================================

    import time
    import json
    import numpy as np
    import pandas as pd
    import torch
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_auc_score, average_precision_score, matthews_corrcoef,
        cohen_kappa_score, roc_curve, precision_recall_curve, confusion_matrix,
    )

    EVAL_ROOT = OUTPUT_ROOT.parent / "PHASE0_RESULTS" / "evaluation"
    EVAL_ROOT.mkdir(parents=True, exist_ok=True)

    # Substep 0.8.1 + most of 0.8.2 (reuse the val distances/labels/threshold from Step 0.6)
    acc  = accuracy_score(all_labels, final_preds)
    prec = precision_score(all_labels, final_preds, zero_division=0)
    rec  = recall_score(all_labels, final_preds, zero_division=0)
    f1   = f1_score(all_labels, final_preds, zero_division=0)

    roc_auc = roc_auc_score(all_labels, all_distances)
    pr_auc  = average_precision_score(all_labels, all_distances)
    mcc     = matthews_corrcoef(all_labels, final_preds)
    kappa   = cohen_kappa_score(all_labels, final_preds)   # <-- Cohen's Kappa

    far = np.mean((final_preds == 1) & (all_labels == 0))
    frr = np.mean((final_preds == 0) & (all_labels == 1))
    tpr = np.mean((final_preds == 1) & (all_labels == 1)) / max(np.mean(all_labels == 1), 1e-8)
    tnr = np.mean((final_preds == 0) & (all_labels == 0)) / max(np.mean(all_labels == 0), 1e-8)
    balanced_acc = 0.5 * (tpr + tnr)
    eer = float(best_eer_value)

    cm = confusion_matrix(all_labels, final_preds)

    # ------------------------------------------------------------
    # Substep 0.8.4 : inference time (per-pair latency), params, FLOPs, memory
    # ------------------------------------------------------------
    param_count = sum(p.numel() for p in siamese_model.parameters())

    try:
        from thop import profile
        dummy = torch.randn(1, 3, CONVNEXT_INPUT_SIZE, CONVNEXT_INPUT_SIZE).to(siamese_device)
        flops, _ = profile(siamese_model.forward_one, inputs=(dummy,), verbose=False)
    except Exception as e:
        flops = None
        print("FLOPs computation skipped (thop not available):", e)

    # per-pair inference latency, averaged over N warmed-up forward passes
    siamese_model.eval()
    N_TIMING_RUNS = 50
    dummy_a = torch.randn(1, 3, CONVNEXT_INPUT_SIZE, CONVNEXT_INPUT_SIZE).to(siamese_device)
    dummy_b = torch.randn(1, 3, CONVNEXT_INPUT_SIZE, CONVNEXT_INPUT_SIZE).to(siamese_device)
    with torch.no_grad():
        for _ in range(5):  # warm-up, not timed
            siamese_model(dummy_a, dummy_b)
        if siamese_device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(N_TIMING_RUNS):
            siamese_model(dummy_a, dummy_b)
        if siamese_device.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()
    inference_time_ms_per_pair = ((t1 - t0) / N_TIMING_RUNS) * 1000.0

    if siamese_device.type == "cuda":
        memory_utilization_mb = torch.cuda.max_memory_allocated(siamese_device) / (1024 ** 2)
        memory_metric_name = "peak_gpu_memory_mb"
    else:
        try:
            import psutil, os
            memory_utilization_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
            memory_metric_name = "process_rss_mb"
        except ImportError:
            memory_utilization_mb = None
            memory_metric_name = "unavailable (psutil not installed)"

    siamese_evaluation_metrics = {
        "accuracy": float(acc), "precision": float(prec), "recall": float(rec), "f1_score": float(f1),
        "roc_auc": float(roc_auc), "pr_auc": float(pr_auc), "mcc": float(mcc),
        "balanced_accuracy": float(balanced_acc), "cohens_kappa": float(kappa),
        "far": float(far), "frr": float(frr), "eer": eer,
        "optimal_threshold": siamese_auth_threshold,
        "parameter_count": int(param_count),
        "flops": float(flops) if flops is not None else None,
        "inference_time_ms_per_pair": float(inference_time_ms_per_pair),
        memory_metric_name: float(memory_utilization_mb) if memory_utilization_mb is not None else None,
        "training_time_sec": float(training_time_sec),
        "selected_epoch": int(best_ckpt["epoch"]),
    }

    pd.DataFrame([siamese_evaluation_metrics]).to_csv(EVAL_ROOT / "phase0_8_evaluation_metrics.csv", index=False)
    with open(EVAL_ROOT / "phase0_8_evaluation_metrics.json", "w") as f:
        json.dump(siamese_evaluation_metrics, f, indent=2)

    # ------------------------------------------------------------
    # Substep 0.8.3 : Confusion Matrix, ROC Curve, Precision-Recall Curve
    # Substep 0.8.5 : Training / validation performance plots
    # ------------------------------------------------------------
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    axes[0].plot(train_loss_history, label="train loss")
    axes[0].plot(val_loss_history, label="val loss")
    axes[0].axvline(best_ckpt["epoch"] - 1, color="green", linestyle="--", label="best epoch")
    axes[0].set_title("Training / Validation Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    fpr, tpr_curve, _ = roc_curve(all_labels, all_distances)
    axes[1].plot(fpr, tpr_curve)
    axes[1].plot([0, 1], [0, 1], "--", color="gray")
    axes[1].set_title(f"ROC Curve (AUC={roc_auc:.3f})")
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")

    pr_precision, pr_recall, _ = precision_recall_curve(all_labels, all_distances)
    axes[2].plot(pr_recall, pr_precision)
    axes[2].set_title(f"Precision-Recall Curve (AP={pr_auc:.3f})")
    axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")

    axes[3].imshow(cm, cmap="Blues")
    axes[3].set_title("Confusion Matrix (val pairs)")
    axes[3].set_xticks([0, 1]); axes[3].set_xticklabels(["Genuine", "Impostor"])
    axes[3].set_yticks([0, 1]); axes[3].set_yticklabels(["Genuine", "Impostor"])
    for i in range(2):
        for j in range(2):
            axes[3].text(j, i, cm[i, j], ha="center", va="center")
    axes[3].set_xlabel("Predicted"); axes[3].set_ylabel("True")

    plt.tight_layout()
    plt.savefig(EVAL_ROOT / "phase0_8_evaluation_plots.png", dpi=150)
    plt.show()

    # ------------------------------------------------------------
    # Substep 0.8.6 : save encoder + metrics + plots + a written report
    # (encoder/threshold/config were already exported in Step 0.7; this writes
    # the human-readable performance report to close out Phase 0)
    # ------------------------------------------------------------
    report_path = EVAL_ROOT / "phase0_performance_report.txt"
    with open(report_path, "w") as f:
        f.write("PHASE 0 — Siamese ConvNeXt Authentication Model: Performance Report\n")
        f.write("=" * 75 + "\n\n")
        f.write(f"Selected checkpoint epoch : {best_ckpt['epoch']}\n")
        f.write(f"Training time             : {training_time_sec:.1f} sec\n\n")
        for k, v in siamese_evaluation_metrics.items():
            f.write(f"{k:32s}: {v}\n")

    print("Step 0.8 complete. Evaluation artifacts written to:", EVAL_ROOT)
    for k, v in siamese_evaluation_metrics.items():
        print(f"  {k:32s}: {v}")
    print("\nPHASE 0 COMPLETE.")
else:
    print("STEP 0.8: skipped (RUN_PHASE0 = False); using the saved Phase 0 results.")


# PHASE 1 — Simulated TF-QKD key generation
*Simulated BB84-style sifting, QBER estimate, Cascade-style reconciliation (modelled by its leakage) and Toeplitz privacy amplification. This is not a physical TF-QKD link and the paper must say so.*

In [ ]:
# =============================================================================
# STEP 1 : TF-QKD Session Initialization
#   1.1 Initialize the simulated TF-QKD process (fixed master seed)
#   1.2 Configure Alice and Bob communication entities
#   1.3 Generate secure quantum session parameters
#   1.4 Establish logical synchronization frames between Alice and Bob
#   1.5 Assign Frame IDs / Slot IDs, tag ground-truth validity
# =============================================================================

import random
import numpy as np

# ------------------------------------------------------------
# Substep 1.3 : session parameters — explicit, not left implicit
# ------------------------------------------------------------
DEFAULT_SESSION_PARAMS = {
    "num_frames": 4000,               # rounds per session attempt
    "slots_per_frame": 64,            # time-bin slots modeled per frame
    "frame_match_probability": 0.85,  # relay interference click probability
    "channel_error_rate": 0.03,       # bit-flip probability on a real click
    "dark_count_probability": 0.01,   # spurious click with no real signal
    "sync_slip_probability": 0.02,    # probability Bob's frame record desyncs
}

QBER_ACCEPT_THRESHOLD = 0.11   # BB84-FAMILY security bound (NOT derived from
                                # TF-QKD's own security proof — see the scope
                                # note above). Used here as a practical,
                                # standard acceptance criterion for the
                                # BB84-style post-processing chain that
                                # follows the TF-QKD-inspired frame/click
                                # simulation. Substep 3.2 value.
FINAL_KEY_BITS = 256
MAX_RETRIES = 5                # shared budget for 3.3 (QBER retry) and
                                # 3.6 (accumulate-to-256-bits retry)

def init_tfqkd_session(seed, params=None):
    """Substeps 1.1 / 1.2 : seeded environment + Alice/Bob entities.
    Alice and Bob each get an independent local RNG stream derived from
    the same master seed, so the whole session is fully reproducible."""
    params = dict(DEFAULT_SESSION_PARAMS if params is None else params)
    master_rng = random.Random(seed)
    alice_rng = random.Random(master_rng.randrange(2**32))
    bob_rng   = random.Random(master_rng.randrange(2**32))
    params["seed"] = seed
    return alice_rng, bob_rng, params

RECON_MODEL_NOTES = (
    "Frame matching: independent Bernoulli trial per frame, P(match) = "
    "frame_match_probability. Relay click: a frame registers a click if "
    "EITHER a real (matched) interference event occurs OR a spurious "
    "dark-count event occurs on an unmatched frame — i.e. click = "
    "real_click OR dark_count, evaluated independently per frame, never "
    "per-slot or per-detector. Channel error: independent per-frame "
    "Bernoulli BIT-FLIP (not phase-flip) applied only to real-click "
    "rounds, P(flip) = channel_error_rate. Dark count: independent "
    "per-frame Bernoulli, P = dark_count_probability, sampled only when "
    "no real click occurred on that frame."
)

# Substep 1.5 : explicit ground-truth validity enum, assigned to EVERY
# frame at creation time (not just discarded ones), as an exact label
# rather than an inferred boolean:
#   "matched"    -> real relay click, frames aligned            (kept)
#   "dark_count" -> spurious click only, no real interference   (discard)
#   "no_click"   -> neither a real nor a spurious click occurred(discard)
#   "desynced"   -> Frame/Slot ID misaligned between Alice/Bob  (discard)
# Priority order below: desync is checked first because a misaligned
# round is unusable regardless of what the click state was.
VALIDITY_MATCHED    = "matched"
VALIDITY_DARK_COUNT = "dark_count"
VALIDITY_NO_CLICK   = "no_click"
VALIDITY_DESYNCED   = "desynced"

SIMULATION_ASSUMPTIONS = (
    "Explicit simulation assumptions (Substep 1.3): "
    "(1) bit-flip errors are independent per frame (no burst/correlated "
    "error model); (2) dark counts are independent per frame; "
    "(3) the classical channel used for all classical communication in "
    "this simulation (basis disclosure for sifting, QBER sample "
    "disclosure, Cascade parity exchange, Toeplitz seed exchange) is "
    "assumed authenticated and noiseless — its own security is NOT "
    "modeled here, only the quantum-channel/detection side is; "
    "(4) detector imperfection (efficiency loss, saturation) is folded "
    "entirely into frame_match_probability as a single lumped parameter, "
    "not modeled as a separate physical effect; (5) NO active "
    "eavesdropper is executed — the QBER acceptance threshold is used as "
    "a standard theoretical security BOUND (per BB84-family analysis), "
    "not the output of an attack simulation; (6) single quantum channel, "
    "no multi-user/multiplexing effects modeled."
)

def generate_synchronization_frames(alice_rng, bob_rng, params):
    """Substeps 1.4 / 1.5 : build synchronization frames.

    Synchronization model: both Frame_ID (the shared
    round counter) and Slot_ID (Alice's chosen time-bin, 0..slots_per_frame-1)
    are treated as jointly agreed between Alice and Bob under normal
    operation via the same classical synchronization channel referenced in
    SIMULATION_ASSUMPTIONS. Clock drift / misalignment is the ONLY
    desynchronization mechanism modeled: with probability
    sync_slip_probability, Bob's copy of BOTH IDs is perturbed
    independently of Alice's (Frame_ID shifted by +1, Slot_ID redrawn) —
    more complex drift dynamics (e.g. drift accumulating across a session)
    are explicitly out of scope.

    Each frame models one TF-QKD round: Alice and Bob each pick a basis
    and (for Alice) a bit; a relay interference click either happens
    (frame_match_probability) or a spurious dark-count click happens
    instead. See RECON_MODEL_NOTES for the exact sampling model of each of
    these. Ground-truth validity is tagged NOW (round_validity) so Step 2's
    discard logic checks a concrete label rather than re-deriving 'invalid'
    after the fact.
    """
    frames = []
    for frame_id in range(params["num_frames"]):
        slot_id = alice_rng.randrange(params["slots_per_frame"])

        alice_basis = alice_rng.choice(["Z", "X"])
        alice_bit   = alice_rng.randint(0, 1)
        bob_basis   = bob_rng.choice(["Z", "X"])

        real_click = alice_rng.random() < params["frame_match_probability"]
        dark_count = (not real_click) and (alice_rng.random() < params["dark_count_probability"])
        desynced   = bob_rng.random() < params["sync_slip_probability"]

        # Desync perturbs BOTH Bob's Frame_ID and Slot_ID (Substep 1.5)
        bob_frame_id = frame_id + (1 if desynced else 0)
        bob_slot_id = (bob_rng.randrange(params["slots_per_frame"])
                       if desynced else slot_id)

        if desynced:
            round_validity = VALIDITY_DESYNCED
        elif real_click:
            round_validity = VALIDITY_MATCHED
        elif dark_count:
            round_validity = VALIDITY_DARK_COUNT
        else:
            round_validity = VALIDITY_NO_CLICK

        frames.append({
            "frame_id": frame_id,
            "bob_frame_id": bob_frame_id,
            "slot_id": slot_id,
            "bob_slot_id": bob_slot_id,
            "alice_basis": alice_basis,
            "alice_bit": alice_bit,
            "bob_basis": bob_basis,
            "real_click": real_click,
            "dark_count": dark_count,
            "desynced": desynced,
            "round_validity": round_validity,   # explicit enum, every frame
        })
    return frames

print("Step 1 complete.")
print("  Session parameters:", DEFAULT_SESSION_PARAMS)
print(f"  QBER acceptance threshold : {QBER_ACCEPT_THRESHOLD}")
print(f"  Target final key length   : {FINAL_KEY_BITS} bits")
print(f"  Max retries (shared budget for Steps 3.3 / 3.6): {MAX_RETRIES}")

In [ ]:
# =============================================================================
# STEP 2 : Simulated Quantum Key Exchange and Synchronization
#   2.1 Simulate TF-QKD quantum key exchange within assigned frames
#   2.2 Post-alignment of Alice and Bob transmission rounds
#   2.3 Discard unmatched or invalid transmission rounds
#   2.4 Key sifting on synchronized transmission rounds
#   2.5 Generate the sifted secret key
# =============================================================================

def simulate_quantum_exchange(frames, alice_rng, params):
    """Substep 2.1 : for each frame, derive Bob's observed bit.
      - real click, no dark count -> Bob's bit = Alice's bit, flipped
        with probability channel_error_rate
      - dark-count-only click     -> Bob's bit is spurious/random,
        uncorrelated with Alice's bit
      - no click at all           -> no observation (round unusable)
    """
    for f in frames:
        if f["real_click"]:
            f["bob_bit"] = f["alice_bit"] if alice_rng.random() >= params["channel_error_rate"] \
                           else 1 - f["alice_bit"]
        elif f["dark_count"]:
            f["bob_bit"] = alice_rng.randint(0, 1)  # spurious, uncorrelated
        else:
            f["bob_bit"] = None
    return frames

def postalign_and_discard(frames):
    """Substeps 2.2 / 2.3 : align by Frame ID AND Slot ID (both IDs, not just Frame ID,
    since Substep 1.5 synchronizes both), against the explicit round_validity label
    assigned at frame creation (Substep 1.5) rather than re-deriving the
    condition here. A frame is kept only if round_validity ==
    VALIDITY_MATCHED AND both its Frame ID and Slot ID actually align;
    everything else is discarded with its validity label as the reason.
    """
    kept, discarded = [], []
    for f in frames:
        ids_aligned = (f["frame_id"] == f["bob_frame_id"] and f["slot_id"] == f["bob_slot_id"])
        if f["round_validity"] == VALIDITY_MATCHED and ids_aligned:
            kept.append(f)
        else:
            # covers desynced / dark_count / no_click, and the edge case
            # of a "matched" frame whose IDs slipped for an unrelated reason
            f["discard_reason"] = (f["round_validity"]
                                    if f["round_validity"] != VALIDITY_MATCHED
                                    else VALIDITY_DESYNCED)
            discarded.append(f)
    return kept, discarded

def sift_key(aligned_frames):
    """Substep 2.4 / 2.5 : keep only basis-matched rounds; concatenate
    Alice's bits from those rounds into the sifted key. (Alice's and
    Bob's bits agree here except for the channel-error-flipped ones,
    which QBER estimation in Step 3 is specifically designed to catch.)"""
    sifted = [f for f in aligned_frames if f["alice_basis"] == f["bob_basis"]]
    sifted_bits_alice = [f["alice_bit"] for f in sifted]
    sifted_bits_bob   = [f["bob_bit"] for f in sifted]
    return sifted, sifted_bits_alice, sifted_bits_bob

def run_step2(alice_rng, bob_rng, params, seed):
    frames = generate_synchronization_frames(alice_rng, bob_rng, params)
    frames = simulate_quantum_exchange(frames, alice_rng, params)
    aligned, discarded = postalign_and_discard(frames)
    sifted, bits_a, bits_b = sift_key(aligned)
    return {
        "seed": seed, "params": params,
        "frames_total": len(frames), "aligned": len(aligned),
        "discarded": len(discarded), "sifted_length": len(sifted),
        "discard_reasons": discarded, "sifted_bits_alice": bits_a,
        "sifted_bits_bob": bits_b,
    }

_demo_alice_rng, _demo_bob_rng, _demo_params = init_tfqkd_session(seed=42)
_demo_result = run_step2(_demo_alice_rng, _demo_bob_rng, _demo_params, seed=42)

print("Step 2 complete (single-session demo, seed=42).")
print(f"  Frames generated : {_demo_result['frames_total']}")
print(f"  Discarded        : {_demo_result['discarded']}")
from collections import Counter
print(f"    by reason      : {Counter(d['discard_reason'] for d in _demo_result['discard_reasons'])}")
print(f"  Aligned (kept)   : {_demo_result['aligned']}")
print(f"  Sifted key length: {_demo_result['sifted_length']} bits")

In [ ]:
# =============================================================================
# STEP 3 : Secret Key Verification and Generation
#   3.1 Compute QBER                       3.5 Privacy amplification
#   3.2 Verify against acceptance threshold 3.6 Generate final 256-bit key
#   3.3 Discard + retry if QBER too high    3.7 Validate randomness/entropy
#   3.4 Error reconciliation                3.8 Store all metadata
# =============================================================================

import math
import json
import datetime as _dt
import numpy as np

QBER_DISCLOSE_FRACTION = 0.15   # fraction of sifted key sacrificed to
                                 # estimate QBER (standard QKD practice —
                                 # never estimate QBER from key-forming bits)
CASCADE_EFFICIENCY_F = 1.16     # typical Cascade reconciliation inefficiency
                                 # factor relative to the Shannon (Slepian-
                                 # Wolf) bound — literature-standard value

def compute_qber(bits_alice, bits_bob, rng, disclose_fraction=QBER_DISCLOSE_FRACTION):
    """Substep 3.1 : QBER from a randomly disclosed, held-out subset.
    Returns (qber, remaining_alice_bits, remaining_bob_bits, n_disclosed)."""
    n = len(bits_alice)
    n_disclosed = max(1, int(n * disclose_fraction))
    idx = list(range(n))
    rng.shuffle(idx)
    disclosed_idx = set(idx[:n_disclosed])

    errors = sum(1 for i in disclosed_idx if bits_alice[i] != bits_bob[i])
    qber = errors / n_disclosed

    keep_idx = [i for i in idx if i not in disclosed_idx]
    remaining_alice = [bits_alice[i] for i in keep_idx]
    remaining_bob   = [bits_bob[i] for i in keep_idx]
    return qber, remaining_alice, remaining_bob, n_disclosed

def binary_entropy(p):
    if p <= 0.0 or p >= 1.0:
        return 0.0
    return -p * math.log2(p) - (1 - p) * math.log2(1 - p)

def reconcile(bits_alice, bits_bob, qber):
    """Substep 3.4 : simulated Cascade reconciliation — OUTCOME-level
    model, not a pass-by-pass implementation. We do NOT run literal
    multi-pass Cascade (no fixed/adaptive pass count applies here, since
    there are no passes to count); instead we model the two things
    downstream steps actually need: (a) Bob's bits get corrected to match
    Alice's (Cascade's job), and (b) a known number of bits are leaked to
    the public channel as parity information during that process,
    estimated via the literature-standard f_EC * n * h2(QBER) formula
    (f_EC = CASCADE_EFFICIENCY_F, the standard inefficiency factor
    relative to the Shannon/Slepian-Wolf bound). That leaked-bit count is
    what Step 3.5 must strip out. If a literal pass-by-pass Cascade
    trace is later required, this function is the one to replace.
    """
    n = len(bits_alice)
    leaked_bits = min(n, math.ceil(CASCADE_EFFICIENCY_F * n * binary_entropy(qber)))
    corrected_bits = list(bits_alice)  # Cascade's job is exactly to make
                                        # Bob's key match Alice's; the
                                        # reconciled key is Alice's bits
    return corrected_bits, leaked_bits

def privacy_amplify(bits, leaked_bits, rng, security_margin_bits=32):
    """Substep 3.5 : Toeplitz-matrix universal hashing.
    Compress the reconciled key by (QBER-disclosed bits already removed
    in 3.1) + (leaked_bits from 3.4) + a fixed security margin, via
    multiplication by a Toeplitz matrix mod 2 — the standard universal-
    hash construction for QKD privacy amplification. The Toeplitz matrix
    is generated from `rng`, which the caller derives deterministically
    from the session seed (seed + 2) — so the matrix is reproducible per
    session, not drawn from unseeded system randomness.
    """
    n = len(bits)
    out_len = max(0, n - leaked_bits - security_margin_bits)
    if out_len == 0:
        return [], 0
    # Toeplitz matrix T (out_len x n): T[i, j] = t[i - j + n - 1], where t holds
    # n + out_len - 1 seeded random bits, so every diagonal is constant and every
    # output bit depends on all n input bits.
    t = np.array([rng.randint(0, 1) for _ in range(n + out_len - 1)], dtype=np.uint8)
    bits_arr = np.array(bits, dtype=np.int32)
    idx = np.arange(out_len)[:, None] - np.arange(n)[None, :] + n - 1
    T = t[idx].astype(np.int32)
    out_bits = (T @ bits_arr) % 2
    return out_bits.astype(np.uint8).tolist(), out_len

def shannon_entropy_bits(bits):
    bits = np.asarray(bits).flatten()
    p1 = np.mean(bits); p0 = 1.0 - p1
    ent = 0.0
    for p in (p0, p1):
        if p > 0:
            ent -= p * np.log2(p)
    return float(ent)

def nist_monobit_test(bits):
    """Substep 3.7a : basic randomness assessment — NIST SP 800-22
    Frequency (Monobit) subtest only. At 256 bits this is NOT a claim of
    full NIST SP 800-22 validation (the full 15-test suite needs far more
    bits to be statistically meaningful) — it is one specific, named,
    cheap subtest, reported as such."""
    x = np.array([1 if b == 1 else -1 for b in bits])
    s_obs = abs(np.sum(x)) / np.sqrt(len(x))
    from math import erfc, sqrt
    p_value = erfc(s_obs / sqrt(2))
    return {"statistic": float(s_obs), "p_value": float(p_value), "pass": p_value >= 0.01}

def nist_runs_test(bits):
    """Substep 3.7b : basic randomness assessment — NIST SP 800-22 Runs
    subtest only, same caveat as nist_monobit_test above."""
    n = len(bits)
    pi = sum(bits) / n
    if abs(pi - 0.5) >= (2 / math.sqrt(n)):
        return {"statistic": None, "p_value": 0.0, "pass": False,
                "note": "prerequisite frequency test failed"}
    v_obs = 1 + sum(1 for i in range(1, n) if bits[i] != bits[i - 1])
    from math import erfc, sqrt
    num = abs(v_obs - 2 * n * pi * (1 - pi))
    den = 2 * math.sqrt(2 * n) * pi * (1 - pi)
    p_value = erfc(num / den) if den > 0 else 0.0
    return {"statistic": float(v_obs), "p_value": float(p_value), "pass": p_value >= 0.01}

def run_single_session(seed, attempt_number):
    """One full attempt at Steps 1-3.5. attempt_number is the caller's
    retry counter (Substep 3.3/3.6 driver), recorded here so each session_log entry is traceable to which retry it
    was, not just which seed. Returns accepted=False with a reason if
    QBER exceeds threshold (caller retries), otherwise the amplified
    key bits + full metadata for that session."""
    session_timestamp = _dt.datetime.now().isoformat()
    alice_rng, bob_rng, params = init_tfqkd_session(seed)
    step2_out = run_step2(alice_rng, bob_rng, params, seed)

    if step2_out["sifted_length"] < 64:
        return {"accepted": False, "reason": "sifted_key_too_short", "seed": seed,
                "attempt_number": attempt_number, "timestamp": session_timestamp,
                "step2": step2_out}

    qber_rng = random.Random(seed + 1)
    qber, rem_a, rem_b, n_disclosed = compute_qber(
        step2_out["sifted_bits_alice"], step2_out["sifted_bits_bob"], qber_rng
    )

    session_record = {
        "seed": seed, "attempt_number": attempt_number, "timestamp": session_timestamp,
        "qber": qber, "n_disclosed_for_qber": n_disclosed,
        "sifted_length": step2_out["sifted_length"],
        "frames_total": step2_out["frames_total"],
        "discarded": step2_out["discarded"],
        "params": params,
    }

    # Substep 3.2 / 3.3 : accept/discard against the threshold
    if qber > QBER_ACCEPT_THRESHOLD:
        session_record["accepted"] = False
        session_record["reason"] = f"qber_exceeds_threshold ({qber:.4f} > {QBER_ACCEPT_THRESHOLD})"
        return session_record

    recon_rng = random.Random(seed + 2)
    corrected_bits, leaked_bits = reconcile(rem_a, rem_b, qber)
    amplified_bits, out_len = privacy_amplify(corrected_bits, leaked_bits, recon_rng)

    session_record.update({
        "accepted": True,
        "reconciliation_leaked_bits": leaked_bits,
        "amplified_key_length": out_len,
        "amplified_bits": amplified_bits,
    })
    return session_record

def generate_tfqkd_key(master_seed=42, target_bits=FINAL_KEY_BITS, max_retries=MAX_RETRIES):
    """Top-level driver implementing Substeps 3.3 and 3.6's retry /
    accumulation logic on top of run_single_session."""
    accumulated_bits = []
    session_log = []
    seed_cursor = master_seed
    attempts = 0

    while len(accumulated_bits) < target_bits and attempts < max_retries:
        attempts += 1
        result = run_single_session(seed_cursor, attempt_number=attempts)
        session_log.append(result)
        if result.get("accepted"):
            accumulated_bits.extend(result["amplified_bits"])
        seed_cursor += 1  # fresh seed per Substep 3.3 retry

    if len(accumulated_bits) < target_bits:
        return {
            "success": False,
            "reason": f"could not reach {target_bits} bits within {max_retries} attempts "
                      f"(got {len(accumulated_bits)} bits)",
            "session_log": session_log,
        }

    final_key = accumulated_bits[:target_bits]

    # Substep 3.7 : randomness validation on the FINAL key only
    entropy = shannon_entropy_bits(final_key)
    monobit = nist_monobit_test(final_key)
    runs = nist_runs_test(final_key)

    return {
        "success": True,
        "final_key_bits": final_key,
        "final_key_length": len(final_key),
        "attempts_used": attempts,
        "sessions_contributed": sum(1 for s in session_log if s.get("accepted")),
        "entropy_bits_per_bit": entropy,
        "nist_monobit": monobit,
        "nist_runs": runs,
        "session_log": session_log,
    }

print("Step 3 functions defined: compute_qber, reconcile, privacy_amplify, "
      "shannon_entropy_bits, nist_monobit_test, nist_runs_test, "
      "generate_tfqkd_key (top-level driver with retry/accumulation).")

In [ ]:
# =============================================================================
# EXECUTION : run the full Phase 1 pipeline end-to-end and persist outputs
# Substep 3.8 : store synchronization metadata, session metadata, and
# key-generation parameters
# =============================================================================

from pathlib import Path
import secrets

PHASE1_RESULT_ROOT = RESULTS / "TF_QKD_RESULTS"
PHASE1_RESULT_ROOT.mkdir(parents=True, exist_ok=True)

if KEY_MODE == "reproducible":
    MASTER_SEED = 42      # fixed: reproduces the paper's key; NOT secret (anyone with the code can regenerate it)
elif KEY_MODE == "fresh":
    MASTER_SEED = secrets.randbits(63)   # unpredictable seed from the OS random source
else:
    raise ValueError("KEY_MODE must be 'reproducible' or 'fresh'")
print(f"KEY_MODE = {KEY_MODE}" + ("  (fixed test key, seed 42)" if KEY_MODE == "reproducible" else "  (fresh key)"))
result = generate_tfqkd_key(master_seed=MASTER_SEED, target_bits=FINAL_KEY_BITS, max_retries=MAX_RETRIES)

if not result["success"]:
    raise RuntimeError("TF-QKD key generation failed: " + result["reason"])

K_TF = np.array(result["final_key_bits"], dtype=np.uint8)  # final 256-bit key, downstream-compatible name

# ------------------------------------------------------------
# Persist: key bits + full metadata record (Substep 3.8)
# ------------------------------------------------------------
np.save(PHASE1_RESULT_ROOT / "K_TF_final_key.npy", K_TF)

metadata_record = {
    "generated_at": __import__("datetime").datetime.now().isoformat(),
    "scope_note": ("Synchronization-aware, TF-QKD-inspired SIMULATION using "
                    "BB84-style post-processing (basis sifting, BB84-family "
                    "QBER bound, Cascade-style reconciliation, Toeplitz "
                    "privacy amplification). Not a physical quantum channel; "
                    "not a faithful full TF-QKD protocol implementation."),
    "randomness_assessment_note": ("Basic randomness assessment using "
                    "Frequency (Monobit) and Runs subtests only — not a "
                    "claim of full NIST SP 800-22 suite validation."),
    "key_mode": KEY_MODE,
    "master_seed": MASTER_SEED if KEY_MODE == "reproducible" else "not stored (fresh key)",
    "target_key_bits": FINAL_KEY_BITS,
    "qber_accept_threshold": QBER_ACCEPT_THRESHOLD,
    "max_retries": MAX_RETRIES,
    "attempts_used": result["attempts_used"],
    "sessions_contributed": result["sessions_contributed"],
    "final_key_length": result["final_key_length"],
    "entropy_bits_per_bit": result["entropy_bits_per_bit"],
    "nist_monobit_test": result["nist_monobit"],
    "nist_runs_test": result["nist_runs"],
    "cascade_efficiency_factor": CASCADE_EFFICIENCY_F,
    "qber_disclose_fraction": QBER_DISCLOSE_FRACTION,
    "session_log": [
        {k: v for k, v in s.items() if k not in ("amplified_bits",)}
        # exclude raw key material from the human-readable log, but keep
        # every session's QBER / accept-reject / discard-reason data
        for s in result["session_log"]
    ],
}

with open(PHASE1_RESULT_ROOT / "phase1_key_generation_metadata.json", "w") as f:
    json.dump(metadata_record, f, indent=2, default=str)

print("=" * 78)
print("PHASE 1 COMPLETE — Simulated TF-QKD 256-bit key generated")
print("=" * 78)
print(f"  Attempts used            : {result['attempts_used']} / {MAX_RETRIES}")
print(f"  Sessions contributing bits: {result['sessions_contributed']}")
print(f"  Final key length          : {result['final_key_length']} bits")
print(f"  Shannon entropy           : {result['entropy_bits_per_bit']:.4f} bits/bit (ideal 1.0)")
print(f"  NIST Monobit test         : p={result['nist_monobit']['p_value']:.4f}  "
      f"pass={result['nist_monobit']['pass']}")
print(f"  NIST Runs test            : p={result['nist_runs']['p_value']:.4f}  "
      f"pass={result['nist_runs']['pass']}")
print(f"\n  Key saved to    : {PHASE1_RESULT_ROOT / 'K_TF_final_key.npy'}")
print(f"  Metadata saved to: {PHASE1_RESULT_ROOT / 'phase1_key_generation_metadata.json'}")

# PHASE 2 — Key derivation and watermark encryption

## Step 4 — Binary watermark preparation

In [ ]:
# =============================================================================
# STEP 4 : BINARY WATERMARK PREPARATION
# =============================================================================

import numpy as np
from PIL import Image
from pathlib import Path

# WATERMARK_PATH comes from CONFIG
WATERMARK_SIZE = 32

if not WATERMARK_PATH.exists():
    raise FileNotFoundError(f"Binary watermark not found:\n{WATERMARK_PATH}")

raw = Image.open(WATERMARK_PATH)

if raw.mode not in {"L", "1"}:
    raise ValueError(
        "Invalid watermark format.\n"
        f"Detected image mode: {raw.mode}\n\n"
        "The input watermark must be a genuine single-channel binary image."
    )

watermark_array = np.asarray(raw)

if watermark_array.ndim != 2:
    raise ValueError(
        "Binary watermark must be a 2-D single-channel image.\n"
        f"Detected shape: {watermark_array.shape}"
    )

if watermark_array.size == 0:
    raise ValueError("Binary watermark is empty.")

unique_values = np.unique(watermark_array)
allowed_values = {0, 1, 255}

if not set(unique_values.tolist()).issubset(allowed_values):
    raise ValueError(
        "The supplied watermark is not binary.\n"
        f"Detected unique values: {unique_values.tolist()}\n\n"
        "Expected binary values represented as {0,1} or {0,255}."
    )

binary_wm = (watermark_array > 0).astype(np.uint8)

normalized_values = np.unique(binary_wm)
if not set(normalized_values.tolist()).issubset({0, 1}):
    raise ValueError(
        "Binary normalization failed.\n"
        f"Detected values: {normalized_values.tolist()}"
    )

# -----------------------------------------------------------------------
# Preserve the original (pre-standardization) binary watermark separately
# from the resized 32x32 working copy, so both are available on disk.
# -----------------------------------------------------------------------
original_shape = binary_wm.shape
W_original = binary_wm.copy()

if original_shape != (WATERMARK_SIZE, WATERMARK_SIZE):
    binary_pil = Image.fromarray((binary_wm * 255).astype(np.uint8))
    binary_pil = binary_pil.resize(
        (WATERMARK_SIZE, WATERMARK_SIZE), Image.Resampling.NEAREST
    )
    binary_wm = (np.asarray(binary_pil) > 0).astype(np.uint8)

if binary_wm.shape != (WATERMARK_SIZE, WATERMARK_SIZE):
    raise ValueError(
        "Watermark dimension standardization failed.\n"
        f"Current shape: {binary_wm.shape}\n"
        f"Expected: {(WATERMARK_SIZE, WATERMARK_SIZE)}"
    )

final_unique_values = np.unique(binary_wm)
if not set(final_unique_values.tolist()).issubset({0, 1}):
    raise ValueError(
        "Final watermark is not binary.\n"
        f"Detected values: {final_unique_values.tolist()}"
    )

def watermark_to_bitstream(binary_wm):
    if binary_wm.ndim != 2:
        raise ValueError("Binary watermark must be two-dimensional.")
    if not set(np.unique(binary_wm).tolist()).issubset({0, 1}):
        raise ValueError("Bitstream conversion requires values {0,1}.")
    return binary_wm.flatten(order="C").astype(np.uint8)

W = binary_wm
W_bitstream = watermark_to_bitstream(W)

expected_bits = WATERMARK_SIZE * WATERMARK_SIZE
if len(W_bitstream) != expected_bits:
    raise ValueError(
        "Unexpected watermark bitstream length.\n"
        f"Expected : {expected_bits}\n"
        f"Received : {len(W_bitstream)}"
    )

watermark_metadata = {
    "source_path": str(WATERMARK_PATH),
    "input_mode": raw.mode,
    "input_representation": "binary",
    "original_shape": list(original_shape),
    "final_shape": list(binary_wm.shape),
    "binary_values": [0, 1],
    "resize_method": (
        "NEAREST" if original_shape != binary_wm.shape else "none (already target size)"
    ),
    "grayscale_conversion": "none",
    "thresholding": "none",
    "bit_order": "row-major (C order)",
    "total_bits": int(len(W_bitstream)),
}

print("=" * 80)
print("STEP 4 : BINARY WATERMARK PREPARATION")
print("=" * 80)
print("Watermark path       :", WATERMARK_PATH)
print("Input mode           :", raw.mode)
print("Original shape       :", original_shape)
print("Final shape          :", binary_wm.shape)
print("Unique values        :", np.unique(binary_wm))
print("Bitstream length     :", len(W_bitstream), "bits")
print("0 bits               :", int(np.sum(W_bitstream == 0)))
print("1 bits               :", int(np.sum(W_bitstream == 1)))
print("=" * 80)
print("W, W_original, and W_bitstream are now defined in this kernel.")
print("STEP 4 COMPLETE")
print("=" * 80)

## Steps 5-6: key derivation and AES-256-GCM watermark encryption
- `K_AES` (32 bytes) is derived from `K_TF` with BLAKE2b; the 12-byte GCM nonce is derived from `K_TF` **and the persistent image ID**, so every image gets its own nonce and its own ciphertext/tag.
- The image ID is passed to AES-GCM as **associated data**: changing the ID, the ciphertext or the tag makes authentication fail.
- The actual per-image encryption happens in Stage A (one image ID per host image). This cell only checks the construction on a demo ID.
- For robustness measurements under attack, the receiver also decrypts through the same keystream **without the tag check** (`ctr_decrypt_bits`); one corrupted ciphertext bit corrupts exactly one plaintext bit. The tag is checked separately (`tag_ok`).


In [ ]:
# PURE-LIB : AES-256-GCM watermark encryption, one nonce per image, image ID bound as associated data
import hashlib
import numpy as np
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.exceptions import InvalidTag

def derive_wm_crypto(K_TF_bytes, image_id):
    """K_AES (32 B) = BLAKE2b(K_TF, key=K_TF, person='WM-AES-KEY')          -- same for all images
       Nonce (12 B) = BLAKE2b(image_id, key=K_TF, person='WM-GCM-NONCE')    -- different for every image ID
       AAD          = 'WM-IMAGE-ID:' + image_id                             -- authenticated, not encrypted
    One watermark per image ID: embedding a second, different watermark under the SAME image ID would reuse the nonce."""
    if image_id is None or str(image_id) == "":
        raise ValueError("derive_wm_crypto needs the persistent image ID (nonce and AAD are bound to it)")
    iid = str(image_id).encode("utf-8")
    K_AES = hashlib.blake2b(K_TF_bytes, key=K_TF_bytes, digest_size=32, person=b"WM-AES-KEY").digest()
    nonce = hashlib.blake2b(iid, key=K_TF_bytes, digest_size=12, person=b"WM-GCM-NONCE").digest()
    return {"key": K_AES, "K_AES": K_AES, "nonce": nonce, "aad": b"WM-IMAGE-ID:" + iid, "image_id": str(image_id)}

def gcm_encrypt_bits(bits, ctx):
    bits = np.asarray(bits, dtype=np.uint8).ravel()
    pt = np.packbits(bits).tobytes()
    ct_tag = AESGCM(ctx["K_AES"]).encrypt(ctx["nonce"], pt, ctx["aad"])
    ct, tag = ct_tag[:-16], ct_tag[-16:]
    ct_bits = np.unpackbits(np.frombuffer(ct, dtype=np.uint8))[:bits.size]
    return ct_bits.astype(np.uint8), tag

def gcm_decrypt_verify(ct_bits, tag, ctx):
    ct_bits = np.asarray(ct_bits, dtype=np.uint8).ravel()
    ct = np.packbits(ct_bits).tobytes()
    try:
        pt = AESGCM(ctx["K_AES"]).decrypt(ctx["nonce"], ct + tag, ctx["aad"])
    except InvalidTag:
        raise ValueError("MAC tag verification failed (ciphertext, tag or image ID changed)")
    pt_bits = np.unpackbits(np.frombuffer(pt, dtype=np.uint8))[:ct_bits.size]
    return pt_bits.astype(np.uint8)

def ctr_decrypt_bits(ct_bits, ctx):
    """Decryption WITHOUT the tag check (same GCM keystream), used only to measure NC/BER after attacks."""
    ct_bits = np.asarray(ct_bits, dtype=np.uint8).ravel()
    ct = np.packbits(ct_bits).tobytes()
    full_nonce = ctx["nonce"] + (2).to_bytes(4, "big")   # GCM keystream starts at counter 2
    enc = Cipher(algorithms.AES(ctx["K_AES"]), modes.CTR(full_nonce)).encryptor()
    pt = enc.update(ct) + enc.finalize()
    pt_bits = np.unpackbits(np.frombuffer(pt, dtype=np.uint8))[:ct_bits.size]
    return pt_bits.astype(np.uint8)

def encrypt_watermark_for_image(W_plain_bits, K_TF_bytes, image_id):
    """Per-image encryption: returns (encrypted bits, 16-byte tag, ctx)."""
    ctx = derive_wm_crypto(K_TF_bytes, image_id)
    enc_bits, tag = gcm_encrypt_bits(W_plain_bits, ctx)
    return enc_bits, tag, ctx


In [ ]:
# =============================================================================
# STEPS 5-6 : key derivation + AES-256-GCM checks (per-image encryption is done in Stage A)
# =============================================================================
if "W_bitstream" not in globals():
    raise RuntimeError("Run Step 4 (watermark preparation) first.")

K_TF = np.load(RESULTS / "TF_QKD_RESULTS" / "K_TF_final_key.npy").astype(np.uint8)
assert K_TF.size == 256 and set(np.unique(K_TF).tolist()) <= {0, 1}, "K_TF must be 256 binary values"
K_TF_bits  = K_TF
K_TF_bytes = np.packbits(K_TF).tobytes()
WM_PLAIN_BITS = W_bitstream.astype(np.uint8)

# demo image IDs, used only for the construction checks below
_DEMO_ID, _DEMO_ID_2 = "selfcheck-demo-image-1", "selfcheck-demo-image-2"
W_enc_bits, GCM_TAG, _ctx = encrypt_watermark_for_image(WM_PLAIN_BITS, K_TF_bytes, _DEMO_ID)
W_enc = W_enc_bits.reshape(32, 32)       # demo ciphertext; only its SIZE is used later (1024 bits)

# check 1: authenticated decryption reproduces the watermark bit-exactly
assert (gcm_decrypt_verify(W_enc_bits, GCM_TAG, _ctx) == WM_PLAIN_BITS).all()
# check 2: decryption without the tag check agrees on an uncorrupted ciphertext
assert (ctr_decrypt_bits(W_enc_bits, _ctx) == WM_PLAIN_BITS).all()
# check 3: one flipped ciphertext bit -> authentication fails; decryption without tag check differs in exactly that bit
_bad = W_enc_bits.copy(); _bad[7] ^= 1
try:
    gcm_decrypt_verify(_bad, GCM_TAG, _ctx); raise RuntimeError("GCM authentication should have failed")
except ValueError:
    pass
assert int((ctr_decrypt_bits(_bad, _ctx) != WM_PLAIN_BITS).sum()) == 1
# check 4: a different image ID gives a different nonce and a different ciphertext
_enc2, _tag2, _ctx2 = encrypt_watermark_for_image(WM_PLAIN_BITS, K_TF_bytes, _DEMO_ID_2)
assert _ctx2["nonce"] != _ctx["nonce"] and (_enc2 != W_enc_bits).any()
# check 5: correct ciphertext + tag but a CHANGED image ID -> authentication fails (ID is bound as associated data)
try:
    gcm_decrypt_verify(W_enc_bits, GCM_TAG, derive_wm_crypto(K_TF_bytes, _DEMO_ID_2)); raise RuntimeError("should have failed")
except ValueError:
    pass

WM_DIR = RESULTS / "watermark_encryption"
WM_DIR.mkdir(parents=True, exist_ok=True)
np.save(WM_DIR / "W_original.npy", W_original)
np.save(WM_DIR / "W_bitstream.npy", WM_PLAIN_BITS)
json.dump({
    "cipher": "AES-256-GCM (authenticated encryption)",
    "key_derivation": "K_AES (32 B) = BLAKE2b(K_TF, key=K_TF, person='WM-AES-KEY')",
    "nonce_derivation": "Nonce (12 B) = BLAKE2b(image_id, key=K_TF, person='WM-GCM-NONCE'): one nonce per image ID",
    "associated_data": "'WM-IMAGE-ID:' + image_id (authenticated, not encrypted)",
    "per_image": "each host image is encrypted separately in Stage A; ciphertext and tag are stored in the embedding map / ID registry",
    "tag_bytes": 16, "payload_bits": int(WM_PLAIN_BITS.size),
    "robustness_decryption": "same keystream applied without the tag check, so a corrupted ciphertext still decrypts (bit-local)",
}, open(WM_DIR / "phase2_encryption_metadata.json", "w"), indent=2)

print("=" * 78)
print("PHASE 2 COMPLETE (construction checks)")
print(f"  watermark bits {WM_PLAIN_BITS.size}  |  encrypted bits {W_enc_bits.size}")
print("  checks passed: authenticated round trip; decryption without tag == authenticated decryption;")
print("  1-bit ciphertext error -> tag fails; different image ID -> different nonce/ciphertext; changed image ID -> tag fails")
print("  per-image encryption happens in Stage A")


# PHASE 3 — Adaptive watermark embedding
K_TF-only secret block selection (no content ranking, so it is identical for the original and every attacked copy) followed by content-adaptive DCT embedding in the Y channel.

In [ ]:
# =============================================================================
# SETUP : paths, folders, and deterministic-execution configuration
# =============================================================================

import os
import uuid
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import convnext_tiny

from scipy.fftpack import dct, idct
from skimage.metrics import structural_similarity as ssim

torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except TypeError:
    torch.use_deterministic_algorithms(True)
    
prepared_root = PREPARED

INPUT_FOLDER = prepared_root / "test" / "authentic"

result_root = RESULTS
folder_result_dir = result_root / "folder_level_embedding_authentic"
watermarked_dir = folder_result_dir / "watermarked_images"
difference_dir = folder_result_dir / "difference_images"
selected_block_dir = folder_result_dir / "selected_block_visuals"
embedding_map_dir = result_root / "PHASE3_EMBEDDING_MAPS"

for d in (folder_result_dir, watermarked_dir, difference_dir, selected_block_dir, embedding_map_dir):
    d.mkdir(parents=True, exist_ok=True)

print("Input folder     :", INPUT_FOLDER)
print("Embedding maps to:", embedding_map_dir)
print("Results to       :", folder_result_dir)

# MAX_IMAGES comes from CONFIG

# EMBEDDING_THRESHOLD and REDUNDANCY come from CONFIG

if "K_TF" not in globals():
    ktf_path = result_root / "TF_QKD_RESULTS" / "K_TF_final_key.npy"
    if ktf_path.exists():
        K_TF = np.load(ktf_path)
        print("Loaded K_TF from:", ktf_path)
    else:
        raise ValueError(
            f"K_TF not found in memory or on disk at {ktf_path}. "
            f"Run Phase 1 (TF-QKD key generation) first."
        )
if "W_enc" not in globals():
    encrypted_path = result_root / "watermark_encryption" / "W_enc_bits.npy"
    if encrypted_path.exists():
        W_enc = np.load(encrypted_path).reshape(32, 32)
        print("Loaded W_enc from:", encrypted_path)
    else:
        raise ValueError("W_enc not found. Run Phase 2 (watermark encryption) first.")
if W_enc.size != 32 * 32:
    raise ValueError(f"W_enc must contain 1024 bits (32x32). Got {W_enc.size}.")

if "WM_PLAIN_BITS" not in globals():
    WM_PLAIN_BITS = np.load(result_root / "watermark_encryption" / "W_bitstream.npy").astype(np.uint8)
    print("Loaded the plaintext watermark bitstream from disk")
print("Encrypted watermark bits per image:", W_enc.size)

# K_TF as produced by Phase 1 is a NumPy 0/1 bit array, not raw bytes --
# pack it once here (same convention Phase 2 already uses).
K_TF_bytes = np.packbits(np.asarray(K_TF).astype(np.uint8)).tobytes()

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
image_paths = sorted(p for p in INPUT_FOLDER.rglob("*") if p.suffix.lower() in IMG_EXTS)
if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]
if len(image_paths) == 0:
    raise ValueError("No images found in the given folder.")
print("Total images found:", len(image_paths))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# =============================================================================
# Trained Siamese Encoder (Phase 0)
# Spatial feature map for embedding-strength modulation ONLY.
# Not used for block selection.
# =============================================================================

PHASE0_ENCODER_PATH = PHASE0_OUT / "deployment" / "authentication_encoder.pt"

if not PHASE0_ENCODER_PATH.exists():
    raise FileNotFoundError(
        f"Trained Siamese authentication encoder not found at "
        f"{PHASE0_ENCODER_PATH}. "
        f"Run Phase 0 (Step 0.7 deployment export) first."
    )


# =============================================================================
# Trained Siamese Encoder
# =============================================================================

class TrainedSiameseEncoder(nn.Module):
    """
    Only the spatial (pre-pool) feature map is used downstream.
    The pooled/projected embedding is retained only for compatibility.
    """

    def __init__(self, embedding_dim=128):
        super().__init__()

        base = convnext_tiny(weights=None)

        self.backbone = base.features
        self.pool = base.avgpool

        self.projection = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, embedding_dim),
        )

    def forward(self, x):
        spatial_map = self.backbone(x)
        return spatial_map


# =============================================================================
# Load Trained Encoder
# =============================================================================

_encoder_checkpoint = torch.load(
    PHASE0_ENCODER_PATH,
    map_location=device
)

_embedding_dim = _encoder_checkpoint.get("embedding_dim", 128)

feature_extractor = TrainedSiameseEncoder(
    embedding_dim=_embedding_dim
).to(device)

feature_extractor.backbone.load_state_dict(
    _encoder_checkpoint["encoder_state"]["backbone"]
)

feature_extractor.eval()

print(
    f"Loaded TRAINED Siamese authentication encoder from Phase 0 "
    f"(source checkpoint epoch "
    f"{_encoder_checkpoint.get('source_checkpoint_epoch', '?')}) "
    f"-- spatial feature map only, used for embedding-strength modulation."
)


# =============================================================================
# ConvNeXt Preprocessing
# =============================================================================

convnext_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


# =============================================================================
# Extract Spatial Feature Map
# =============================================================================

def extract_spatial_map(img_rgb):

    x = convnext_transform(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        spatial_map = feature_extractor(x)

    return spatial_map.squeeze(0).cpu().numpy()


# =============================================================================
# Compute Semantic Block Scores
# =============================================================================

def compute_semantic_block_scores(spatial_map, blocks_per_row):
    """
    Computes the semantic activation score for each image block.

    These scores are used ONLY for adaptive embedding-strength
    modulation. They are NOT used for block selection.
    """

    energy_map = np.sqrt(
        np.sum(
            spatial_map.astype(np.float32) ** 2,
            axis=0
        )
    )

    energy_img = Image.fromarray(
        energy_map.astype(np.float32)
    )

    blocks_per_row_img = energy_img.resize(
        (blocks_per_row, blocks_per_row),
        Image.Resampling.BILINEAR
    )

    return np.array(
        blocks_per_row_img,
        dtype=np.float32
    ).flatten()


# =============================================================================
# Dummy Verification
# =============================================================================

print("Ready. Spatial map shape check on a dummy input:")

with torch.no_grad():

    _smap = feature_extractor(
        torch.randn(1, 3, 224, 224).to(device)
    )

print("  spatial_map:", tuple(_smap.shape))

In [ ]:
# =============================================================================
# K_TF-only block selection (attack-immune by construction)
# =============================================================================

def blake2b_digest(data, digest_size=32, person=b"BLKSEL"):
    return hashlib.blake2b(data, digest_size=digest_size, person=person).digest()

def generate_seed_from_key_and_identifier(K_TF_bytes, image_identifier: str):
    """The ONLY inputs are K_TF and a stable per-image identifier. That
    identifier MUST be a persistent Image ID (a random id generated once
    by the sender and stored in transmission metadata -- see Stage A),
    NOT the filename: filenames can change in transit/storage/renaming,
    but a stored Image ID cannot, since it is never recomputed from the
    file. NO image content is involved either, so this seed is identical
    for the original and every attacked variant of the same image,
    regardless of any filename change."""
    seed_bytes = blake2b_digest(
        K_TF_bytes + image_identifier.encode("utf-8") + b"KTF_ONLY_BLOCK_SELECTION",
        digest_size=32, person=b"SELSEED",
    )
    seed_int = int.from_bytes(seed_bytes[:8], "big")
    return seed_int, seed_bytes

def _keyed_uint32_stream(seed_bytes):
    """Keyed pseudo-random stream: BLAKE2b in counter mode, keyed with the full 256-bit selection seed."""
    ctr = 0
    while True:
        blk = hashlib.blake2b(ctr.to_bytes(8, "big"), key=seed_bytes, digest_size=64, person=b"BLKPERM").digest()
        for j in range(0, 64, 4):
            yield int.from_bytes(blk[j:j + 4], "big")
        ctr += 1

def keyed_sample_without_replacement(seed_bytes, n_total, k):
    """Partial Fisher-Yates shuffle driven by the keyed BLAKE2b stream (rejection sampling, so no modulo bias).
    Replaces numpy's PCG64 (non-cryptographic, 64-bit seed): the block map now depends on all 256 seed bits."""
    if k > n_total:
        raise ValueError("k must not exceed n_total")
    stream = _keyed_uint32_stream(seed_bytes)
    idx = list(range(n_total))
    for i in range(k):
        m = n_total - i
        limit = (2 ** 32 // m) * m
        r = next(stream)
        while r >= limit:
            r = next(stream)
        j = i + (r % m)
        idx[i], idx[j] = idx[j], idx[i]
    return idx[:k]

def select_blocks_tfqkd(K_TF_bytes, image_identifier, image_size=512, block_size=8, n_blocks=1024):
    """WHICH blocks: pure K_TF+identifier-seeded uniform random choice
    over the full block grid. No content ranking, no reliability score --
    that's the point. `image_identifier` must be the persistent Image ID
    from transmission metadata (see Stage A), not the filename. A
    receiver with only K_TF and that Image ID reproduces this exact set
    regardless of what attack the image went through, AND regardless of
    any filename change."""
    blocks_per_row = image_size // block_size
    total_blocks = blocks_per_row * blocks_per_row

    _, seed_bytes = generate_seed_from_key_and_identifier(K_TF_bytes, image_identifier)
    chosen_indices = keyed_sample_without_replacement(seed_bytes, total_blocks, n_blocks)
    selected_blocks = []
    for idx in chosen_indices:
        row_block, col_block = divmod(int(idx), blocks_per_row)
        selected_blocks.append({
            "index": int(idx), "x": col_block * block_size, "y": row_block * block_size,
            "row_block": row_block, "col_block": col_block,
        })
    return selected_blocks, seed_bytes

print("K_TF-only block selection ready (no image-content dependency).")

In [ ]:
# =============================================================================
# DCT embedding machinery
# =============================================================================

def rgb_to_ycbcr_np(img_rgb):
    img = img_rgb.astype(np.float32)
    R, G, B = img[:, :, 0], img[:, :, 1], img[:, :, 2]
    Y  = 0.299 * R + 0.587 * G + 0.114 * B
    Cb = 128.0 - 0.168736 * R - 0.331264 * G + 0.5 * B
    Cr = 128.0 + 0.5 * R - 0.418688 * G - 0.081312 * B
    return np.clip(np.stack([Y, Cb, Cr], axis=2), 0, 255)       # float: no truncation here

def ycbcr_to_rgb_np(img_ycbcr):
    img = img_ycbcr.astype(np.float32)
    Y, Cb, Cr = img[:, :, 0], img[:, :, 1] - 128.0, img[:, :, 2] - 128.0
    R = Y + 1.402 * Cr
    G = Y - 0.344136 * Cb - 0.714136 * Cr
    B = Y + 1.772 * Cb
    return np.rint(np.clip(np.stack([R, G, B], axis=2), 0, 255)).astype(np.uint8)

def apply_luma_delta(rgb, Y_orig, Y_new):
    """Write the luminance change back into RGB with ONE rounding step.
    Adding the same d to R, G and B changes Y by exactly d (0.299+0.587+0.114 = 1) and leaves Cb and Cr
    unchanged (their coefficients sum to 0), so the chroma channels are untouched and no colour
    round-trip error is introduced."""
    d = np.asarray(Y_new, dtype=np.float64) - np.asarray(Y_orig, dtype=np.float64)
    return np.rint(np.clip(rgb.astype(np.float64) + d[:, :, None], 0, 255)).astype(np.uint8)

def dct2(block):
    return dct(dct(block.T, norm="ortho").T, norm="ortho")

def idct2(block):
    return idct(idct(block.T, norm="ortho").T, norm="ortho")

def embed_bit_by_coefficient_comparison(C, bit, pos1=(3, 2), pos2=(2, 3), threshold=8.0, margin=25.0, diff_cap=DIFF_CAP):
    c1, c2 = float(C[pos1]), float(C[pos2])
    target = threshold + margin
    original_diff = abs(c1 - c2)
    effective_target = max(target, target + min(0.50 * original_diff, diff_cap))
    avg = (c1 + c2) / 2.0
    if bit == 1:
        if (c1 - c2) < effective_target:
            C[pos1] = avg + effective_target / 2.0
            C[pos2] = avg - effective_target / 2.0
    else:
        if (c2 - c1) < effective_target:
            C[pos1] = avg - effective_target / 2.0
            C[pos2] = avg + effective_target / 2.0
    return C

def compute_adaptive_threshold(block, C, semantic_norm=0.0, base_threshold=None,
                                min_scale=0.90, max_scale=1.55,
                                w_var=0.25, w_energy=0.45, w_sem=0.30):
    if base_threshold is None:
        base_threshold = EMBEDDING_THRESHOLD
    variance = np.var(block)
    variance_norm = variance / (variance + 500.0)

    C_energy = C.copy(); C_energy[0, 0] = 0
    energy = np.sum(C_energy ** 2)
    energy_norm = energy / (energy + 100000.0)

    robustness_score = w_var * variance_norm + w_energy * energy_norm + w_sem * semantic_norm
    scale = min_scale + (max_scale - min_scale) * robustness_score
    return base_threshold * scale

def embed_encrypted_watermark_in_y_channel(Y, W_enc, selected_blocks, adaptive_thresholds,
                                            block_size=8, pos1=(2, 1), pos2=(1, 2),
                                            margin=12.0, redundancy=3, diff_cap=None):
    diff_cap = DIFF_CAP if diff_cap is None else diff_cap
    Y_w = Y.copy().astype(np.float32)
    wm_bits = np.repeat(W_enc.flatten().astype(np.uint8), redundancy)
    for i, bit in enumerate(wm_bits):
        b = selected_blocks[i]
        x, y = b["x"], b["y"]
        block = Y_w[y:y + block_size, x:x + block_size].copy()
        C = dct2(block)
        threshold = adaptive_thresholds[i]
        C_mod = embed_bit_by_coefficient_comparison(
            C=C, bit=int(bit), pos1=pos1, pos2=pos2,
            threshold=threshold, margin=margin, diff_cap=diff_cap
        )
        block_w = np.clip(idct2(C_mod), 0, 255)
        Y_w[y:y + block_size, x:x + block_size] = block_w
    return np.clip(Y_w, 0, 255)          # float; rounding happens once, in apply_luma_delta

def load_host_image(image_path, size=512):
    img = Image.open(image_path).convert("RGB").resize((size, size), Image.Resampling.LANCZOS)
    return np.array(img, dtype=np.uint8)

def compute_mse(img1, img2):
    return float(np.mean((img1.astype(np.float32) - img2.astype(np.float32)) ** 2))

def compute_psnr(img1, img2):
    mse_val = compute_mse(img1, img2)
    return float("inf") if mse_val == 0 else float(20 * np.log10(255.0 / np.sqrt(mse_val)))

def compute_ssim_rgb(img1, img2):
    return float(ssim(img1, img2, channel_axis=-1, data_range=255))

def compute_entropy_gray(img_rgb):
    gray = np.array(Image.fromarray(img_rgb).convert("L"), dtype=np.uint8)
    hist, _ = np.histogram(gray.flatten(), bins=256, range=(0, 256))
    prob = hist / np.sum(hist); prob = prob[prob > 0]
    return float(-np.sum(prob * np.log2(prob)))

BLOCK_SIZE = 8
POS1, POS2 = (2, 1), (1, 2)
POS1B, POS2B = (3, 1), (1, 3)
# EMBED_MARGIN comes from CONFIG

print("DCT embedding functions ready.")

## Evaluation library
Pure functions without file access: vectorized embedding/extraction, metrics, region-level tamper localization and the image-level decision rule. Used by the sweep (3.8), Phases 5–7 and the ablation.

In [ ]:
# PURE-LIB : vectorized embedding / extraction / metrics / region-level tamper localization
import numpy as np
from scipy.fft import dctn, idctn
from scipy.stats import binom

BLOCK = 8
NBLK = 64                      # 512 / 8 blocks per row
POS_A, POS_B = (2, 1), (1, 2)  # the single coefficient pair used by the scheme

def blocks_rc(blocks):
    rows = np.fromiter((b["row_block"] for b in blocks), dtype=np.int64, count=len(blocks))
    cols = np.fromiter((b["col_block"] for b in blocks), dtype=np.int64, count=len(blocks))
    return rows, cols

def _grid(Y):      # (512,512) -> (64,64,8,8)
    return np.ascontiguousarray(Y).reshape(NBLK, BLOCK, NBLK, BLOCK).transpose(0, 2, 1, 3)

def gather_blocks(Y, rows, cols):
    return _grid(Y)[rows, cols].astype(np.float64)

def scatter_blocks(Y, rows, cols, blk):
    G = _grid(Y.astype(np.float64)).copy()
    G[rows, cols] = blk
    return G.transpose(0, 2, 1, 3).reshape(512, 512)

def rgb_to_y_float(rgb):
    r = rgb.astype(np.float64)
    return 0.299 * r[:, :, 0] + 0.587 * r[:, :, 1] + 0.114 * r[:, :, 2]

def adaptive_thresholds_vec(blk, coeffs, sem_norm, base, w=(0.25, 0.45, 0.30), s_min=0.90, s_max=1.55):
    """Vectorized Eq. 1 + Eq. 2:  rho = w_v*v + w_e*e + w_s*s ;  T = base*(s_min + (s_max-s_min)*rho)"""
    var = blk.reshape(len(blk), -1).var(axis=1)
    v = var / (var + 500.0)
    C = coeffs.copy(); C[:, 0, 0] = 0.0
    en = (C ** 2).sum(axis=(1, 2))
    e = en / (en + 1e5)
    rho = w[0] * v + w[1] * e + w[2] * np.asarray(sem_norm, dtype=np.float64)
    return base * (s_min + (s_max - s_min) * rho)

def embed_vec(Y, bits, rows, cols, thresholds, margin, cap, pos1=POS_A, pos2=POS_B):
    """Eq. 4-6, vectorized over all selected blocks. Returns the modified luminance (float)."""
    B = gather_blocks(Y, rows, cols)
    C = dctn(B, type=2, norm="ortho", axes=(1, 2))
    c1 = C[:, pos1[0], pos1[1]].copy(); c2 = C[:, pos2[0], pos2[1]].copy()
    eff = thresholds + margin + np.minimum(0.5 * np.abs(c1 - c2), cap)       # Delta (Eq. 4)
    avg = 0.5 * (c1 + c2)
    b = np.asarray(bits).astype(bool)
    up = b & ((c1 - c2) < eff)                                               # bit 1 needs c1 >> c2
    dn = (~b) & ((c2 - c1) < eff)                                            # bit 0 needs c2 >> c1
    n1 = np.where(up, avg + eff / 2, np.where(dn, avg - eff / 2, c1))
    n2 = np.where(up, avg - eff / 2, np.where(dn, avg + eff / 2, c2))
    C[:, pos1[0], pos1[1]] = n1; C[:, pos2[0], pos2[1]] = n2
    Bw = np.clip(idctn(C, type=2, norm="ortho", axes=(1, 2)), 0, 255)
    return scatter_blocks(Y, rows, cols, Bw)

def thresholds_for_image(Y, rows, cols, sem_norm, base):
    B = gather_blocks(Y, rows, cols)
    C = dctn(B, type=2, norm="ortho", axes=(1, 2))
    return adaptive_thresholds_vec(B, C, sem_norm, base)

def extract_raw_bits(rgb, rows, cols, pos1=POS_A, pos2=POS_B):
    """bit = 1[c1 - c2 >= 0] for every selected block (the raw, un-voted bits)."""
    C = dctn(gather_blocks(rgb_to_y_float(rgb), rows, cols), type=2, norm="ortho", axes=(1, 2))
    return ((C[:, pos1[0], pos1[1]] - C[:, pos2[0], pos2[1]]) >= 0).astype(np.uint8)

def majority_vote_bits(raw, R):
    return (np.asarray(raw).reshape(-1, R).sum(axis=1) >= (R / 2.0)).astype(np.uint8)

def ncc_bits(a, b):
    a = np.asarray(a, dtype=np.float64).ravel(); b = np.asarray(b, dtype=np.float64).ravel()
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / d) if d > 0 else 0.0

def ber_bits(a, b):
    return float(np.mean(np.asarray(a).ravel() != np.asarray(b).ravel()))

# ---------------- region-level tamper localization (Phase 6, step 6.4) ----------------
CELL_PX = 16      # output resolution: 16 x 16-pixel cells (32 x 32 grid)
WIN_BLK = 4       # test window = 4 x 4 blocks = 32 x 32 pixels, stride 2 blocks

def block_error_grids(blocks, errors):
    S = np.zeros((NBLK, NBLK)); E = np.zeros((NBLK, NBLK))
    for b, e in zip(blocks, errors):
        S[b["row_block"], b["col_block"]] = 1.0
        E[b["row_block"], b["col_block"]] = float(e)
    return S, E

def _window_sums(G, w=WIN_BLK, step=2):
    I = np.pad(G.cumsum(0).cumsum(1), ((1, 0), (1, 0)))
    n = (NBLK - w) // step + 1
    idx = np.arange(n) * step
    r0, c0 = idx[:, None], idx[None, :]
    return I[r0 + w, c0 + w] - I[r0, c0 + w] - I[r0 + w, c0] + I[r0, c0]

def localize_tampering(blocks, raw_bits, expected_bits, alpha=1e-4, min_rate=0.30, min_blocks=6, floor=0.02):
    """Scan-statistic localization. Slide a 32x32-pixel window (stride 16) over the image; in each window compare the number of
    wrong raw bits with the image's own baseline error rate (one-sided binomial test). A 16x16 cell is flagged when at least
    half of the windows covering it are flagged. Returns (cell_flags[32x32], window_rate, window_pvalue, baseline_rate)."""
    errors = (np.asarray(raw_bits) != np.asarray(expected_bits)).astype(np.uint8)
    S, E = block_error_grids(blocks, errors)
    n = _window_sums(S); k = _window_sums(E)
    valid = n >= min_blocks
    rate = np.divide(k, n, out=np.zeros_like(k), where=n > 0)
    p0 = max(float(np.median(rate[valid])), floor) if valid.any() else floor
    pval = np.ones_like(rate)
    pval[valid] = binom.sf(k[valid] - 1, n[valid].astype(int), p0)
    win_flag = valid & (pval < alpha) & (rate >= min_rate)
    nw = win_flag.shape[0]; ncell = NBLK // 2
    votes = np.zeros((ncell, ncell)); cover = np.zeros((ncell, ncell))
    for dr in (0, 1):
        for dc in (0, 1):
            # cell index c is covered by window c - dr (window index w covers cells w and w+1)
            r = np.arange(ncell) - dr; c = np.arange(ncell) - dc
            rv = (r >= 0) & (r < nw); cv = (c >= 0) & (c < nw)
            sub = np.zeros((ncell, ncell)); ok = np.zeros((ncell, ncell))
            sub[np.ix_(rv, cv)] = win_flag[np.ix_(r[rv], c[cv])]
            ok[np.ix_(rv, cv)] = 1.0
            votes += sub; cover += ok
    cells = (votes >= np.maximum(1.0, cover / 2.0)) & (cover > 0)
    return cells, rate, pval, p0

def region_ground_truth(mask, cell_px=CELL_PX, min_frac=0.25):
    n = mask.shape[0] // cell_px
    g = (mask > 0).reshape(n, cell_px, n, cell_px).mean(axis=(1, 3))
    return g >= min_frac

def flagged_to_mask(cells, cell_px=CELL_PX):
    return np.kron(cells.astype(np.uint8), np.ones((cell_px, cell_px), dtype=np.uint8)) * 255

def prf_iou(pred, gt):
    pred = np.asarray(pred, bool); gt = np.asarray(gt, bool)
    tp = int((pred & gt).sum()); fp = int((pred & ~gt).sum()); fn = int((~pred & gt).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    iou = tp / (tp + fp + fn) if tp + fp + fn else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=p, recall=r, f1=f1, iou=iou)

# ---------------- three-way
# image-level decision: AUTHENTIC / LOCALIZED_TAMPERING / GLOBAL_MODIFICATION.
# The binomial-test localization above (cells, from localize_tampering) only
# answers "which 16x16 cells look wrong". On its own it cannot tell a real
# local forgery apart from a benign GLOBAL attack (JPEG/blur/noise/gamma),
# which also perturbs the raw bits but EVERYWHERE, not in one region. This
# adds that missing distinction, using the background (non-flagged-cell)
# block error rate B alongside the flagged cell fraction F.
# ---------------------------------------------------------------------------

def classify_tamper_level(blocks, raw_bits, expected_bits, cells,
                           cell_px=CELL_PX, block_px=BLOCK,
                           global_flag_frac=0.50, global_bg_rate=0.03,
                           authentic_flag_frac=0.0025):
    """blocks/raw_bits/expected_bits: same inputs as localize_tampering.
    cells: the (32,32) boolean grid localize_tampering already returned.
    Returns the image-level verdict plus the two statistics it was based on."""
    errors = (np.asarray(raw_bits) != np.asarray(expected_bits)).astype(np.uint8)
    S, E = block_error_grids(blocks, errors)          # (64,64) selected-mask / error-mask
    blocks_per_cell = max(1, cell_px // block_px)      # 16 // 8 = 2
    flagged_block_grid = np.kron(
        cells.astype(np.uint8),
        np.ones((blocks_per_cell, blocks_per_cell), dtype=np.uint8),
    ).astype(bool)
    bg_mask = (S > 0) & (~flagged_block_grid)
    bg_selected = float(S[bg_mask].sum())
    B = float(E[bg_mask].sum() / bg_selected) if bg_selected > 0 else 0.0
    F = float(cells.mean())
    if F > global_flag_frac or B > global_bg_rate:
        level = "GLOBAL_MODIFICATION"
    elif F < authentic_flag_frac:
        level = "AUTHENTIC"
    else:
        level = "LOCALIZED_TAMPERING"
    return dict(level=level, flagged_fraction=F, background_error_rate=B)


### Worked numerical example
The embedding rule (Eq. 4–6) applied to two scalar coefficients, with every intermediate value printed.

In [ ]:
# =============================================================================
# Worked numerical example of
# the embedding rule (Eq. 4-6 in embed_vec above), using the constants
# reported in the manuscript from CONFIG, so the arithmetic can be checked by hand.
# =============================================================================

def worked_example(c1, c2, T_b, bit, margin=EMBED_MARGIN, cap=DIFF_CAP):
    """Mirrors embed_vec's per-block rule exactly (same Delta formula),
    but on two scalar coefficients, with every intermediate value printed."""
    original_diff = abs(c1 - c2)
    delta = T_b + margin + min(0.5 * original_diff, cap)  # Eq. 4
    avg = 0.5 * (c1 + c2)
    if bit == 1:
        if (c1 - c2) < delta:
            new_c1, new_c2 = avg + delta / 2.0, avg - delta / 2.0
            changed = True
        else:
            new_c1, new_c2, changed = c1, c2, False
    else:
        if (c2 - c1) < delta:
            new_c1, new_c2 = avg - delta / 2.0, avg + delta / 2.0
            changed = True
        else:
            new_c1, new_c2, changed = c1, c2, False
    return dict(c1=c1, c2=c2, T_b=T_b, bit=bit, margin=margin, cap=cap,
                original_diff=original_diff, delta=delta, avg=avg,
                new_c1=new_c1, new_c2=new_c2, block_was_modified=changed,
                extracted_bit_after=int((new_c1 - new_c2) >= 0))

T_EX = EMBEDDING_THRESHOLD * (0.90 + 0.65 * 0.0)     # Eq. 2 with rho = 0 (range: 0.90*T0 .. 1.55*T0)
print(f"Worked example -- embedding bit 1 into a block whose adaptive threshold T_b = {T_EX:.3f} (T0 = {EMBEDDING_THRESHOLD:g}, rho = 0),")
print("with original DCT coefficients c1 = 10.0, c2 = 4.0:\n")
ex1 = worked_example(c1=10.0, c2=4.0, T_b=T_EX, bit=1)
for k, v in ex1.items():
    print(f"  {k:22s} = {v}")

print("\nSame block, but embedding bit 0 instead (c1/c2 must flip sign):\n")
ex2 = worked_example(c1=10.0, c2=4.0, T_b=T_EX, bit=0)
for k, v in ex2.items():
    print(f"  {k:22s} = {v}")

print("\nA block that already satisfies the target gap needs NO modification "
      "(c1 - c2 already >= Delta for bit 1):\n")
ex3 = worked_example(c1=60.0, c2=10.0, T_b=T_EX, bit=1)
for k, v in ex3.items():
    print(f"  {k:22s} = {v}")

worked_examples_df = pd.DataFrame([ex1, ex2, ex3])
_we_dir = RESULTS / "PHASE3_SWEEP"
_we_dir.mkdir(parents=True, exist_ok=True)
worked_examples_df.to_csv(_we_dir / "worked_embedding_example.csv", index=False)
print("\nSaved:", _we_dir / "worked_embedding_example.csv")


In [ ]:
# =============================================================================
# semantic scores for the selected blocks of one image (same normalization as Stage A)
# =============================================================================
def semantic_norm_selected(rgb, selected_blocks):
    spatial = extract_spatial_map(rgb)
    sc = compute_semantic_block_scores(spatial, 512 // BLOCK_SIZE)
    sc = (sc - sc.min()) / (sc.max() - sc.min() + 1e-8)
    return np.array([sc[b["index"]] for b in selected_blocks], dtype=np.float64)

def watermark_image(rgb, bits_R, rows, cols, thresholds, margin, cap):
    """full embedding path used everywhere outside Stage B: Y modified, chroma untouched, one rounding step"""
    Y = rgb_to_ycbcr_np(rgb)[:, :, 0].astype(np.float64)
    Yw = embed_vec(Y, bits_R, rows, cols, thresholds, margin, cap)
    return apply_luma_delta(rgb, Y, Yw)

def image_thresholds(rgb, rows, cols, sem_norm, base):
    Y = rgb_to_ycbcr_np(rgb)[:, :, 0].astype(np.float64)
    return thresholds_for_image(Y, rows, cols, sem_norm, base)

def psnr_rgb(a, b):
    m = float(np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2))
    return float("inf") if m == 0 else 10 * math.log10(255.0 ** 2 / m)

def ssim_rgb(a, b):
    return float(_ssim_fn(a, b, channel_axis=-1, data_range=255))

# in-memory versions of the Phase 4 attacks, operating on RGB arrays (Phase 4 functions expect BGR)
def _via_bgr(fn, *a, **k):
    return lambda im, rng=None: bgr2rgb(fn(rgb2bgr(im), *a, **k))
ATTACKS_MEM = {
    "JPEG_Q70":       lambda im, rng: bgr2rgb(jpeg_compression(rgb2bgr(im), quality=70)),
    "JPEG_Q50":       lambda im, rng: bgr2rgb(jpeg_compression(rgb2bgr(im), quality=50)),
    "Gaussian_Noise": lambda im, rng: bgr2rgb(gaussian_noise(rgb2bgr(im), sigma=10, rng=rng)),
    "Gaussian_Blur":  lambda im, rng: bgr2rgb(gaussian_blur(rgb2bgr(im))),
    "Median_Filter":  lambda im, rng: bgr2rgb(median_filter(rgb2bgr(im))),
    "Brightness":     lambda im, rng: bgr2rgb(brightness(rgb2bgr(im), beta=25)),
}

def evaluate_embedding(rgb_list, params, W_plain, K_bytes, wm_id=None, sem_fn=None, attacks=None, encrypt=True,
                       adaptive=True, do_loc=True, do_adv=False, adv_key_bytes=b"\x00" * 32, seed=GLOBAL_SEED, id_prefix="eval",
                       loc_alpha=1e-4, loc_min_rate=0.30):
    """Embed -> attack -> extract on a list of RGB host images (all in memory).
       params = dict(T0, margin, cap, R).  Returns means of PSNR/SSIM, clean + attacked NC/BER (CTR-decrypted),
       splice-localization P/R/F1 under JPEG70 and the false-flag rate on un-tampered JPEG70 images.
       encrypt=False  -> the plaintext watermark is embedded (ablation);  adaptive=False -> constant threshold T0 (ablation);
       do_adv=True    -> also report NC for an adversary without K_TF (adv_no_key) and one who knows the block map (adv_knows_blocks).
       Localization here is a quick screening protocol (splice + JPEG70, loc_alpha / loc_min_rate as given);
       the held-out localization results in Phase 6 use the fixed LOC_ALPHA / LOC_MIN_RATE."""
    wm_id = wm_id or WATERMARK_ID
    attacks = attacks or ATTACKS_MEM
    R = int(params["R"]); rng = np.random.default_rng(seed)
    dec = (lambda v, c: ctr_decrypt_bits(v, c)) if encrypt else (lambda v, c: v)
    n_bits = int(np.asarray(W_plain).size)
    jp = ATTACKS_MEM["JPEG_Q70"]
    out = dict(psnr=[], ssim=[], clean_nc=[], clean_ber=[], adv_nokey=[], adv_blocks=[], tp=0, fp=0, fn=0, flag=0, n=0)
    atk_nc = {k: [] for k in attacks}; atk_ber = {k: [] for k in attacks}
    for n_i, rgb in enumerate(rgb_list):
        ident = f"{id_prefix}-{n_i}"
        crypt_id = f"{ident}-{wm_id}"                      # per-image nonce / AAD
        ctx = derive_wm_crypto(K_bytes, crypt_id)
        enc_bits = gcm_encrypt_bits(W_plain, ctx)[0] if encrypt else np.asarray(W_plain, dtype=np.uint8)
        bitsR = np.repeat(enc_bits, R)
        adv_ctx = derive_wm_crypto(adv_key_bytes, crypt_id)
        blocks, _ = select_blocks_tfqkd(K_bytes, ident, 512, BLOCK_SIZE, n_bits * R)
        rows, cols = blocks_rc(blocks)
        sem = sem_fn(rgb, blocks) if sem_fn is not None else np.full(len(blocks), 0.5)
        th = image_thresholds(rgb, rows, cols, sem, params["T0"]) if adaptive else np.full(len(blocks), float(params["T0"]))
        wm = watermark_image(rgb, bitsR, rows, cols, th, params["margin"], params["cap"])
        out["psnr"].append(psnr_rgb(rgb, wm)); out["ssim"].append(ssim_rgb(rgb, wm))
        def nc_ber(img):
            pt = dec(majority_vote_bits(extract_raw_bits(img, rows, cols), R), ctx)
            return ncc_bits(pt, W_plain), ber_bits(pt, W_plain)
        nc0, b0 = nc_ber(wm); out["clean_nc"].append(nc0); out["clean_ber"].append(b0)
        for k, fn in attacks.items():
            nc, b = nc_ber(fn(wm, rng)); atk_nc[k].append(nc); atk_ber[k].append(b)
        if do_adv:
            ab, _ = select_blocks_tfqkd(adv_key_bytes, ident, 512, BLOCK_SIZE, n_bits * R)
            ar, ac = blocks_rc(ab)
            v = majority_vote_bits(extract_raw_bits(wm, ar, ac), R)
            out["adv_nokey"].append(ncc_bits(dec(v, adv_ctx), W_plain))
            v = majority_vote_bits(extract_raw_bits(wm, rows, cols), R)
            out["adv_blocks"].append(ncc_bits(dec(v, adv_ctx), W_plain))
        if do_loc:
            src = rgb_list[(n_i + 1) % len(rgb_list)]
            spl, mask = image_splicing(wm, src, rng)
            for img, m in ((jp(spl, rng), mask), (jp(wm, rng), np.zeros_like(mask))):
                cells, _, _, _ = localize_tampering(blocks, extract_raw_bits(img, rows, cols), bitsR,
                                                    alpha=loc_alpha, min_rate=loc_min_rate)
                if m.any():
                    r = prf_iou(cells, region_ground_truth(m)); out["tp"] += r["tp"]; out["fp"] += r["fp"]; out["fn"] += r["fn"]
                else:
                    out["flag"] += int(cells.any()); out["n"] += 1
    P = out["tp"] / max(out["tp"] + out["fp"], 1); Rc = out["tp"] / max(out["tp"] + out["fn"], 1)
    res = dict(psnr=float(np.mean(out["psnr"])), ssim=float(np.mean(out["ssim"])),
               clean_nc=float(np.mean(out["clean_nc"])), clean_ber=float(np.mean(out["clean_ber"])),
               attacked_nc=float(np.mean([np.mean(v) for v in atk_nc.values()])),
               attacked_ber=float(np.mean([np.mean(v) for v in atk_ber.values()])),
               min_attack_nc=float(min(np.mean(v) for v in atk_nc.values())))
    if do_loc:
        res.update(loc_precision=P, loc_recall=Rc, loc_f1=(2 * P * Rc / (P + Rc) if P + Rc else 0.0),
                   benign_false_flag=out["flag"] / max(out["n"], 1))
    if do_adv:
        res.update(adv_no_key_nc=float(np.mean(out["adv_nokey"])), adv_knows_blocks_nc=float(np.mean(out["adv_blocks"])))
    res.update({f"nc_{k}": float(np.mean(v)) for k, v in atk_nc.items()})
    return res

## 3.8 Parameter sweep (optional, exploratory)
The constants reported in the manuscript are fixed in CONFIG. This sweep is not needed to reproduce the paper and does not change them.
For every combination of base threshold `T0`, margin `m` and cap (and redundancy `R`), on **validation** images, the sweep measures PSNR, SSIM, clean NC, NC after six attacks, and region-level splice localization (precision/recall/F1 under JPEG 70) with the false-flag rate on un-tampered images.
It freezes the combination that reaches `TARGET_PSNR` with clean NC ≥ `MIN_CLEAN_NC`, false-flag rate ≤ `MAX_BENIGN_FALSE_FLAG` and the best mean attacked NC. If no combination reaches the target, it freezes the highest-PSNR combination that keeps attacked NC ≥ `MIN_ATTACKED_NC`, and the paper reports the PSNR actually obtained.
The semantic term is a constant 0.5 inside the sweep (the trained encoder is not run at every grid point); Stage A uses the real encoder.

In [ ]:
# =============================================================================
# 3.8 PARAMETER SWEEP  (validation images only)
# =============================================================================
import itertools
SWEEP_ROOT = RESULTS / "PHASE3_SWEEP"; SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
RUN_SWEEP     = False   # optional exploratory sweep on validation images; it does not change the constants fixed in CONFIG
SWEEP_IMAGES  = 16
SWEEP_GRID = dict(T0=[6, 10, 14, 20, 28], margin=[0, 5, 10, 15], cap=[0, 9, 18])
SWEEP_R_LIST  = [1, 2, 3]           # e.g. [1, 3] to include redundancy in the sweep

if RUN_SWEEP:
    val_paths = sorted((PREPARED / "val" / "authentic").glob("*.png"))
    if len(val_paths) < SWEEP_IMAGES:
        raise ValueError(f"Only {len(val_paths)} validation images found")
    sweep_paths = random.Random(GLOBAL_SEED).sample(val_paths, SWEEP_IMAGES)
    sweep_imgs = [load_host_image(p, 512) for p in sweep_paths]
    rows_out = []
    combos = list(itertools.product(SWEEP_R_LIST, SWEEP_GRID["T0"], SWEEP_GRID["margin"], SWEEP_GRID["cap"]))
    for R_, T0_, m_, cap_ in tqdm(combos, desc="sweep"):
        prm = dict(T0=T0_, margin=m_, cap=cap_, R=R_)
        r = evaluate_embedding(sweep_imgs, prm, WM_PLAIN_BITS, K_TF_bytes, id_prefix="sweep")
        rows_out.append({**prm, **r})
    sweep_df = pd.DataFrame(rows_out)
    sweep_df.to_csv(SWEEP_ROOT / "phase3_parameter_sweep.csv", index=False)

    feas = sweep_df[(sweep_df.psnr >= TARGET_PSNR) & (sweep_df.clean_nc >= MIN_CLEAN_NC) &
                    (sweep_df.benign_false_flag <= MAX_BENIGN_FALSE_FLAG)]
    if len(feas):
        best = feas.sort_values(["attacked_nc", "loc_f1"], ascending=False).iloc[0]; reached = True
    else:
        ok = sweep_df[(sweep_df.attacked_nc >= MIN_ATTACKED_NC) & (sweep_df.clean_nc >= MIN_CLEAN_NC) &
                      (sweep_df.benign_false_flag <= MAX_BENIGN_FALSE_FLAG)]
        pool = ok if len(ok) else sweep_df
        best = pool.sort_values("psnr", ascending=False).iloc[0]; reached = False
    frozen = {k: (int(best[k]) if k == "R" else float(best[k])) for k in ("T0", "margin", "cap", "R")}
    frozen.update(target_psnr=TARGET_PSNR, target_reached=bool(reached),
                  metrics={k: float(best[k]) for k in ("psnr", "ssim", "clean_nc", "attacked_nc", "min_attack_nc",
                                                       "loc_precision", "loc_recall", "loc_f1", "benign_false_flag")},
                  sweep_images=SWEEP_IMAGES, selected_from="validation images")
    json.dump(frozen, open(FROZEN_PARAMS_FILE, "w"), indent=2)

    print("=" * 78)
    print("TARGET PSNR %.1f dB %s" % (TARGET_PSNR, "REACHED" if reached else "NOT REACHED with acceptable robustness"))
    print("Frozen:", {k: frozen[k] for k in ("T0", "margin", "cap", "R")})
    print("Metrics:", {k: round(v, 4) for k, v in frozen["metrics"].items()})
    display(sweep_df.sort_values("psnr", ascending=False).head(12).round(4))

    fig, ax = plt.subplots(figsize=(7, 4.5))
    sc = ax.scatter(sweep_df.psnr, sweep_df.attacked_nc, c=sweep_df.loc_f1, cmap="viridis", s=45)
    ax.axvline(TARGET_PSNR, ls="--", c="gray"); ax.axhline(MIN_ATTACKED_NC, ls=":", c="gray")
    ax.set_xlabel("PSNR (dB)"); ax.set_ylabel("mean attacked NC"); plt.colorbar(sc, label="splice localization F1")
    ax.set_title("Imperceptibility vs robustness trade-off (validation images)")
    fig.tight_layout(); fig.savefig(SWEEP_ROOT / "sweep_tradeoff.png", dpi=300); plt.show()
else:
    print("RUN_SWEEP = False -> using the existing frozen file (if any).")

In [ ]:
# =============================================================================
# Fix the constants: from here on, Stage A / Stage B / every later phase use the CONFIG constants
# =============================================================================
FROZEN_PARAMS_FILE.parent.mkdir(parents=True, exist_ok=True)
json.dump(dict(T0=float(EMBEDDING_THRESHOLD), margin=float(EMBED_MARGIN), cap=float(DIFF_CAP), R=int(REDUNDANCY),
               note="fixed values reported in the manuscript"),
          open(FROZEN_PARAMS_FILE, "w"), indent=2)
_fz = json.load(open(FROZEN_PARAMS_FILE))
EMBEDDING_THRESHOLD, EMBED_MARGIN, DIFF_CAP, REDUNDANCY = _fz["T0"], _fz["margin"], _fz["cap"], int(_fz["R"])
assert W_enc.size * REDUNDANCY <= 4096, "R too large: 1024 bits x R must not exceed 4096 blocks"
print("Constants in use -> T0=%.3g  margin=%.3g  cap=%.3g  R=%d" % (EMBEDDING_THRESHOLD, EMBED_MARGIN, DIFF_CAP, REDUNDANCY))
print("Target reached:", _fz.get("target_reached"), "| sweep metrics:", {k: round(v, 4) for k, v in _fz.get("metrics", {}).items()})

## Stage A — content analysis and embedding-map generation

In [ ]:
# =============================================================================
# STAGE A : Content Analysis & Embedding-Map Generation
# =============================================================================

embedding_maps = {}
stage_a_skipped = []
id_registry = []


for idx, img_path in enumerate(image_paths, start=1):

    try:

        print(f"[Stage A] {idx}/{len(image_paths)} : {img_path.name}")

        # ============================================================
        # Load Host Image
        # ============================================================

        I_rgb = load_host_image(
            img_path,
            size=512
        )

        I_ycbcr = rgb_to_ycbcr_np(I_rgb)

        Y = I_ycbcr[:, :, 0].astype(np.float32)

        # ============================================================
        # Generate Persistent Image ID
        # ============================================================

        image_id = str(uuid.uuid4())

        id_registry.append({

            "image_id": image_id,

            "original_filename": img_path.name,

            "original_stem": img_path.stem

        })

        # per-image AES-256-GCM encryption: nonce and associated data are bound to this image ID
        enc_bits_img, tag_img, _ = encrypt_watermark_for_image(WM_PLAIN_BITS, K_TF_bytes, image_id)
        id_registry[-1]["gcm_tag_hex"] = tag_img.hex()      # transmitted with the image ID as metadata

        # ============================================================
        # TF-QKD Block Selection
        # ============================================================

        selected_blocks, seed_bytes = select_blocks_tfqkd(

            K_TF_bytes=K_TF_bytes,

            image_identifier=image_id,

            image_size=512,

            block_size=BLOCK_SIZE,

            n_blocks=W_enc.size * REDUNDANCY

        )

        # ============================================================
        # Semantic Feature Extraction (ConvNeXt)
        # ============================================================

        spatial_map = extract_spatial_map(I_rgb)

        blocks_per_row = 512 // BLOCK_SIZE

        semantic_scores = compute_semantic_block_scores(
            spatial_map,
            blocks_per_row
        )

        sem_min = semantic_scores.min()

        sem_max = semantic_scores.max()

        sem_norm_all = (
            semantic_scores - sem_min
        ) / (
            sem_max - sem_min + 1e-8
        )

        # ============================================================
        # Adaptive Threshold Calculation
        # ============================================================

        adaptive_thresholds = []

        for b in selected_blocks:

            block = Y[
                b["y"]: b["y"] + BLOCK_SIZE,
                b["x"]: b["x"] + BLOCK_SIZE
            ]

            C = dct2(block)

            sem_norm = float(
                sem_norm_all[b["index"]]
            )

            threshold = compute_adaptive_threshold(

                block=block,

                C=C,

                semantic_norm=sem_norm,

                base_threshold=EMBEDDING_THRESHOLD

            )

            adaptive_thresholds.append(
                float(threshold)
            )

        # ============================================================
        # Store Embedding Map
        # ============================================================

        embedding_maps[img_path.stem] = {

            "image_path": str(img_path),

            "image_id": image_id,

            "original_filename": img_path.name,

            "embedding_locations": selected_blocks,

            "adaptive_thresholds": adaptive_thresholds,

            "embedding_seed_hex": seed_bytes.hex(),

            "enc_bits": enc_bits_img.astype(np.uint8).tolist(),

            "gcm_tag_hex": tag_img.hex(),

            "block_selection_source":
                "K_TF + Persistent Image ID",

            "n_blocks_selected": len(selected_blocks)

        }

    except Exception as e:

        print(
            "  SKIPPED:",
            img_path.name,
            "|",
            e
        )

        stage_a_skipped.append({

            "image": img_path.name,

            "reason": str(e)

        })


# =============================================================================
# Save Embedding Maps
# =============================================================================

embedding_map_file = (
    embedding_map_dir /
    "phase3_embedding_maps.npy"
)

np.save(
    embedding_map_file,
    embedding_maps,
    allow_pickle=True
)


# =============================================================================
# Save Skipped Images
# =============================================================================

stage_a_skipped_file = (
    embedding_map_dir /
    "phase3_stage_a_skipped.csv"
)

pd.DataFrame(stage_a_skipped).to_csv(
    stage_a_skipped_file,
    index=False
)


# =============================================================================
# Save Image ID Registry
# =============================================================================

image_id_registry_file = (
    embedding_map_dir /
    "image_id_registry.csv"
)

pd.DataFrame(id_registry).to_csv(
    image_id_registry_file,
    index=False
)


# =============================================================================
# Stage A Summary
# =============================================================================

print("=" * 80)
print("STAGE A COMPLETED")
print("=" * 80)

print(f"Embedding Maps Generated : {len(embedding_maps)}")
print(f"Skipped Images           : {len(stage_a_skipped)}")

print("\nSaved Files")
print(f"Embedding Maps : {embedding_map_file}")
print(f"Skipped CSV    : {stage_a_skipped_file}")
print(f"Image Registry : {image_id_registry_file}")

print("=" * 80)

## Stage B — DCT embedding

In [ ]:
# =============================================================================
# STAGE B : DCT Embedding using Stage A's stored embedding maps
# =============================================================================

results = []
sample_outputs = []

for idx, (img_stem, emap) in enumerate(embedding_maps.items(), start=1):
    try:
        print(f"[Stage B] {idx}/{len(embedding_maps)}: {img_stem}")
        img_path = Path(emap["image_path"])
        I_rgb = load_host_image(img_path, size=512)
        Y = rgb_to_ycbcr_np(I_rgb)[:, :, 0].astype(np.float64)
        W_enc_img = np.asarray(emap["enc_bits"], dtype=np.uint8)      # this image's own ciphertext

        selected_blocks = emap["embedding_locations"]
        adaptive_thresholds = emap["adaptive_thresholds"]

        Y_w = embed_encrypted_watermark_in_y_channel(
            Y=Y, W_enc=W_enc_img, selected_blocks=selected_blocks,
            adaptive_thresholds=adaptive_thresholds, block_size=BLOCK_SIZE,
            pos1=POS1, pos2=POS2,
            margin=EMBED_MARGIN, redundancy=REDUNDANCY, diff_cap=DIFF_CAP,
        )
        I_w_rgb = apply_luma_delta(I_rgb, Y, Y_w)                     # chroma untouched, one rounding step

        mse_val = compute_mse(I_rgb, I_w_rgb)
        psnr_val = compute_psnr(I_rgb, I_w_rgb)
        ssim_val = compute_ssim_rgb(I_rgb, I_w_rgb)
        ent_orig = compute_entropy_gray(I_rgb)
        ent_wm = compute_entropy_gray(I_w_rgb)

        out_path = watermarked_dir / (img_stem + "_watermarked.png")
        Image.fromarray(I_w_rgb).save(out_path)

        diff = np.abs(I_rgb.astype(np.int16) - I_w_rgb.astype(np.int16)).astype(np.uint8)
        diff_enhanced = np.clip(diff * 10, 0, 255).astype(np.uint8)
        Image.fromarray(diff_enhanced).save(difference_dir / (img_stem + "_diff_x10.png"))

        if idx <= 5:
            vis = Image.fromarray(I_rgb.copy())
            draw = ImageDraw.Draw(vis)
            for b in selected_blocks:
                draw.rectangle([b["x"], b["y"], b["x"] + 8, b["y"] + 8], outline=(255, 0, 0), width=1)
            vis.save(selected_block_dir / (img_stem + "_selected_blocks.png"))

        results.append({
            "image_name": img_stem, "watermarked_image": str(out_path),
            "mse": mse_val, "psnr_db": psnr_val, "ssim": ssim_val,
            "entropy_original_gray": ent_orig, "entropy_watermarked_gray": ent_wm,
            "entropy_diff_gray": abs(ent_orig - ent_wm),
            "embedding_threshold": EMBEDDING_THRESHOLD, "embed_margin": EMBED_MARGIN, "diff_cap": DIFF_CAP,
            "selected_blocks": len(selected_blocks),
            "embedding_seed_hex": emap["embedding_seed_hex"],
            "payload_bits": int(W_enc.size),
        })

        if len(sample_outputs) < 3:
            sample_outputs.append((I_rgb, I_w_rgb, diff_enhanced, img_stem))

        print(f"  MSE={mse_val:.4f}  PSNR={psnr_val:.4f} dB  SSIM={ssim_val:.6f}")

    except Exception as e:
        print("  SKIPPED due to error:", img_stem, "|", e)

df_results = pd.DataFrame(results)
csv_path = folder_result_dir / "folder_level_quality_metrics.csv"
df_results.to_csv(csv_path, index=False)

if len(df_results) > 0:
    avg_metrics = {
        "total_images_processed": len(df_results),
        "avg_mse": df_results["mse"].mean(), "avg_psnr_db": df_results["psnr_db"].mean(),
        "avg_ssim": df_results["ssim"].mean(),
        "avg_entropy_diff_gray": df_results["entropy_diff_gray"].mean(),
        "embedding_threshold": EMBEDDING_THRESHOLD,
    }
    pd.DataFrame([avg_metrics]).to_csv(folder_result_dir / "average_quality_metrics.csv", index=False)
    print("\nAverage metrics:")
    for k, v in avg_metrics.items():
        print(f"  {k}: {v}")

for orig, wm, diff_img, name in sample_outputs:
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1); plt.imshow(orig); plt.title(f"Original\n{name}"); plt.axis("off")
    plt.subplot(1, 3, 2); plt.imshow(wm); plt.title("Watermarked"); plt.axis("off")
    plt.subplot(1, 3, 3); plt.imshow(diff_img); plt.title("Difference x10"); plt.axis("off")
    plt.tight_layout(); plt.show()

print("\nPHASE 3 COMPLETE.")
print("  Stage A embedding maps:", embedding_map_file)
print("  Stage B watermarked images:", watermarked_dir)
print("  Stage B metrics CSV:", csv_path)

## Validation — block selection is attack-immune by construction

In [ ]:
# =============================================================================
# VALIDATION : block selection is attack-immune by construction
# =============================================================================

print("Validating synchronization using the persistent Image ID "
      "(not filename, not pixel content)...")
all_match = True
for img_stem, emap in list(embedding_maps.items())[:5]:
    persistent_id = emap["image_id"]
    original_name = emap["original_filename"]

    blocks_1, seed_1 = select_blocks_tfqkd(K_TF_bytes, persistent_id, 512, BLOCK_SIZE, W_enc.size * REDUNDANCY)
    blocks_2, seed_2 = select_blocks_tfqkd(K_TF_bytes, persistent_id, 512, BLOCK_SIZE, W_enc.size * REDUNDANCY)
    match = (seed_1 == seed_2) and (blocks_1 == blocks_2)
    all_match = all_match and match
    print(f"  {original_name} (id={persistent_id[:8]}...): reproducible from Image ID = {match}")

    # Pixel content changes (attacks) don't
    # matter, because selection never looks at pixel content at all.
    fake_attacked_pixels = np.random.default_rng(999).integers(0, 255, (512, 512, 3), dtype=np.uint8)
    blocks_attacked, seed_attacked = select_blocks_tfqkd(K_TF_bytes, persistent_id, 512, BLOCK_SIZE, W_enc.size * REDUNDANCY)
    attack_match = (seed_attacked == seed_1) and (blocks_attacked == blocks_1)
    all_match = all_match and attack_match
    print(f"    under simulated arbitrary pixel content, still identical = {attack_match}")

    # A RENAMED file still resolves to the
    # SAME seed, because the seed never used the filename at all -- only
    # the persistent Image ID, which a rename cannot touch.
    renamed_file_lookup = persistent_id  # receiver looks this up via the
                                          # id registry, regardless of
                                          # whatever the file got renamed to
    blocks_renamed, seed_renamed = select_blocks_tfqkd(K_TF_bytes, renamed_file_lookup, 512, BLOCK_SIZE, W_enc.size * REDUNDANCY)
    rename_match = (seed_renamed == seed_1) and (blocks_renamed == blocks_1)
    all_match = all_match and rename_match
    print(f"    under simulated filename change, still identical = {rename_match}")

print()
if all_match:
    print("PASS: block selection is provably identical regardless of pixel content "
          "AND regardless of filename changes, using only K_TF + the persistent Image ID.")
else:
    print("FAIL: investigate -- this should be impossible given the K_TF-only, "
          "Image-ID-based design.")

# PHASE 4 — Attack simulation
19 benign/geometric attacks and 4 forgeries; the forgery masks are saved under `Masks/<attack>/`.

In [ ]:
# =============================================================================
# PHASE 4 : ATTACK SIMULATION
# Setup, Imports and Folder Configuration
# =============================================================================
# 19 benign/geometric attack folders + 4 malicious content-editing attack folders
# (Copy_Move_Forgery, Image_Splicing, Object_Removal, Region_Replacement),
# plus a Masks/ subfolder per forgery that stores the ground-truth tamper mask.
# =============================================================================

import os
import cv2
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# ------------------------------------------------------------------
# CROSS-PHASE CONVENTION (extraction)
# ------------------------------------------------------------------
# Phase 3 embedded the watermark using images loaded via PIL as RGB
# arrays (Image.open(...).convert("RGB")); its Y/Cb/Cr formula assumes
# img[:,:,0] = R. This Phase 4 notebook loads/writes images via cv2
# (BGR) -- self-consistent internally, and the image FILES saved to
# disk are correct standard images either way.
#
# However: the extraction phase MUST load these attacked images the
# SAME way Phase 3 embedded them -- via PIL as RGB, NOT cv2.imread
# directly -- or R and B will be silently swapped in the recovered
# luminance channel for every attacked image, corrupting every
# extracted watermark bit. If Phase 6 needs cv2 for any reason,
# convert immediately after loading:
#     img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------
RNG_SEED = 42
random.seed(RNG_SEED)
rng = np.random.default_rng(RNG_SEED)

# ------------------------------------------------------------------
# Root Directory
# ------------------------------------------------------------------

RESULT_ROOT = RESULTS

# ------------------------------------------------------------------
# Input Images (Generated in Phase 3)
# ------------------------------------------------------------------

INPUT_FOLDER = (
    RESULT_ROOT
    / "folder_level_embedding_authentic"
    / "watermarked_images"
)

if not INPUT_FOLDER.exists():
    raise FileNotFoundError(
        f"Watermarked images not found:\n{INPUT_FOLDER}"
    )

# ------------------------------------------------------------------
# Phase 4 Output Directory
# ------------------------------------------------------------------

PHASE4_ROOT = RESULT_ROOT / "PHASE4_ATTACK_SIMULATION"

PHASE4_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Attack Output Folders (19 benign/geometric + 4 malicious)
# ------------------------------------------------------------------

ATTACK_FOLDERS = {
    "JPEG": PHASE4_ROOT / "JPEG",
    "Gaussian_Noise": PHASE4_ROOT / "Gaussian_Noise",
    "Salt_Pepper": PHASE4_ROOT / "Salt_Pepper",
    "Speckle_Noise": PHASE4_ROOT / "Speckle_Noise",
    "Gaussian_Blur": PHASE4_ROOT / "Gaussian_Blur",
    "Median_Filter": PHASE4_ROOT / "Median_Filter",
    "Mean_Filter": PHASE4_ROOT / "Mean_Filter",
    "Bilateral_Filter": PHASE4_ROOT / "Bilateral_Filter",
    "Sharpening": PHASE4_ROOT / "Sharpening",
    "Unsharp_Masking": PHASE4_ROOT / "Unsharp_Masking",
    "Histogram_Equalization": PHASE4_ROOT / "Histogram_Equalization",
    "CLAHE": PHASE4_ROOT / "CLAHE",
    "Rotation": PHASE4_ROOT / "Rotation",
    "Cropping": PHASE4_ROOT / "Cropping",
    "Resize": PHASE4_ROOT / "Resize",
    "Brightness": PHASE4_ROOT / "Brightness",
    "Contrast": PHASE4_ROOT / "Contrast",
    "Gamma": PHASE4_ROOT / "Gamma",
    "Motion_Blur": PHASE4_ROOT / "Motion_Blur",

    # -------- malicious content-editing attacks --------
    "Copy_Move_Forgery": PHASE4_ROOT / "Copy_Move_Forgery",
    "Image_Splicing": PHASE4_ROOT / "Image_Splicing",
    "Object_Removal": PHASE4_ROOT / "Object_Removal",
    "Region_Replacement": PHASE4_ROOT / "Region_Replacement",
}

# Attack types that are ground-truth TAMPERED (used later by Phase 7).
# Saved to disk below so Phase 7 reads this list rather than hardcoding
# it a second time.
MALICIOUS_ATTACKS = [
    "Copy_Move_Forgery",
    "Image_Splicing",
    "Object_Removal",
    "Region_Replacement",
]

for folder in ATTACK_FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)

# Mask output folders -- one per malicious attack, storing a binary
# tamper mask (255 = edited pixel, 0 = untouched) alongside each
# attacked image. Not required for Phase 7's classification-level
# decision, but kept for future pixel-level localization work.
MASK_FOLDERS = {
    name: PHASE4_ROOT / "Masks" / name
    for name in MALICIOUS_ATTACKS
}

for folder in MASK_FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Image List
# ------------------------------------------------------------------

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

image_paths = sorted([
    p for p in INPUT_FOLDER.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
])

if len(image_paths) == 0:
    raise ValueError("No watermarked images found.")

if len(image_paths) < 2:
    raise ValueError(
        "Image_Splicing and Region_Replacement each need a second, "
        "different source image -- at least 2 watermarked images "
        "are required in the input pool."
    )

print("=" * 70)
print("PHASE 4 : ATTACK SIMULATION")
print("=" * 70)
print(f"Input Folder        : {INPUT_FOLDER}")
print(f"Output Folder       : {PHASE4_ROOT}")
print(f"Total Images        : {len(image_paths)}")
print(f"Total Attacks       : {len(ATTACK_FOLDERS)}")
print(f"  Benign/Geometric  : {len(ATTACK_FOLDERS) - len(MALICIOUS_ATTACKS)}")
print(f"  Malicious         : {len(MALICIOUS_ATTACKS)}")
print("=" * 70)

In [ ]:
# =============================================================================
# PHASE 4 : ATTACK SIMULATION
# Apply All Attacks (19 benign/geometric + 4 malicious)
# =============================================================================

attack_summary = []
skipped_images = []

# Preload all images once for splicing/region-replacement source
# sampling, so a different image can be picked per attack without
# re-reading from disk inside the loop.
_image_cache = {}
for p in image_paths:
    im = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if im is not None:
        _image_cache[p.name] = im

for img_path in tqdm(image_paths, desc="Applying Attacks"):

    img = _image_cache.get(img_path.name)

    if img is None:
        print(f"Skipping: {img_path.name}")
        skipped_images.append({"Image": img_path.name, "Reason": "cv2.imread returned None (unreadable)"})
        continue

    # Pick a different image as the splicing/replacement source
    other_names = [n for n in _image_cache if n != img_path.name]
    source_name = other_names[int(rng.integers(0, len(other_names)))]
    source_img = _image_cache[source_name]

    attack_images = {

        "JPEG": jpeg_compression(img, quality=50),

        "Gaussian_Noise": gaussian_noise(img, sigma=10, rng=rng),

        "Salt_Pepper": salt_pepper_noise(img, amount=0.02, rng=rng),

        "Speckle_Noise": speckle_noise(img, rng=rng),

        "Gaussian_Blur": gaussian_blur(img),

        "Median_Filter": median_filter(img),

        "Mean_Filter": mean_filter(img),

        "Bilateral_Filter": bilateral_filter(img),

        "Sharpening": sharpening(img),

        "Unsharp_Masking": unsharp_mask(img),

        "Histogram_Equalization": histogram_equalization(img),

        "CLAHE": clahe(img),

        "Rotation": rotation(img, angle=5),

        "Cropping": cropping(img, crop_ratio=0.10),

        "Resize": resize_attack(img, scale=0.75),

        "Brightness": brightness(img, beta=25),

        "Contrast": contrast(img, alpha=1.25),

        "Gamma": gamma_correction(img, gamma=1.2),

        "Motion_Blur": motion_blur(img, kernel_size=9)

    }

    for attack_name, attacked_img in attack_images.items():

        save_path = ATTACK_FOLDERS[attack_name] / img_path.name

        cv2.imwrite(str(save_path), attacked_img)

        attack_summary.append({
            "Image": img_path.name,
            "Attack": attack_name,
            "Output_Path": str(save_path),
            "Is_Malicious": False,
            "Source_Image": None,
            "Mask_Path": None,
        })

    # -------- malicious content-editing attacks --------

    malicious_outputs = {
        "Copy_Move_Forgery": copy_move_forgery(img, rng),
        "Image_Splicing": image_splicing(img, source_img, rng),
        "Object_Removal": object_removal(img, rng),
        "Region_Replacement": region_replacement(img, source_img, rng),
    }

    for attack_name, (attacked_img, mask) in malicious_outputs.items():

        save_path = ATTACK_FOLDERS[attack_name] / img_path.name
        cv2.imwrite(str(save_path), attacked_img)

        mask_path = MASK_FOLDERS[attack_name] / img_path.name
        cv2.imwrite(str(mask_path), mask)

        uses_source = attack_name in ("Image_Splicing", "Region_Replacement")

        attack_summary.append({
            "Image": img_path.name,
            "Attack": attack_name,
            "Output_Path": str(save_path),
            "Is_Malicious": True,
            "Source_Image": source_name if uses_source else None,
            "Mask_Path": str(mask_path),
        })

print("=" * 70)
print("Attack Simulation Completed Successfully")
print("=" * 70)
print(f"Images Processed : {len(image_paths)}")
print(f"Total Attacks    : {len(ATTACK_FOLDERS)}")
print(f"Total Outputs    : {len(attack_summary)}")
print(f"Images Skipped   : {len(skipped_images)}")
print("=" * 70)

In [ ]:
# =============================================================================
# PHASE 4 : ATTACK SIMULATION
# Save Metadata and Generate Summary
# =============================================================================

# ------------------------------------------------------------------
# Create Attack Summary DataFrame
# ------------------------------------------------------------------

attack_df = pd.DataFrame(attack_summary)

summary_csv = PHASE4_ROOT / "attack_summary.csv"

attack_df.to_csv(summary_csv, index=False)

print(f"Attack summary saved : {summary_csv}")

# ------------------------------------------------------------------
# Number of images generated per attack
# ------------------------------------------------------------------

attack_statistics = []

for attack_name in ATTACK_FOLDERS:

    total = len(list(ATTACK_FOLDERS[attack_name].glob("*")))

    attack_statistics.append({
        "Attack": attack_name,
        "Images": total,
        "Is_Malicious": attack_name in MALICIOUS_ATTACKS,
    })

stats_df = pd.DataFrame(attack_statistics)

stats_csv = PHASE4_ROOT / "attack_statistics.csv"

stats_df.to_csv(stats_csv, index=False)

print(f"Attack statistics saved : {stats_csv}")

# ------------------------------------------------------------------
# Skipped-image log
# ------------------------------------------------------------------

skipped_df = pd.DataFrame(skipped_images)
skipped_csv = PHASE4_ROOT / "skipped_images.csv"
skipped_df.to_csv(skipped_csv, index=False)

print(f"Skipped-images log saved : {skipped_csv} ({len(skipped_images)} images)")

# ------------------------------------------------------------------
# NEW: save the malicious-attack list for Phase 7 to import directly,
# instead of hardcoding the same set of names a second time
# ------------------------------------------------------------------

malicious_attacks_csv = PHASE4_ROOT / "malicious_attack_types.csv"
pd.DataFrame({"Attack": MALICIOUS_ATTACKS}).to_csv(malicious_attacks_csv, index=False)
print(f"Malicious attack list saved : {malicious_attacks_csv}")

# ------------------------------------------------------------------
# Save complete attack metadata
# ------------------------------------------------------------------

metadata = {
    "Input Folder": str(INPUT_FOLDER),
    "Output Folder": str(PHASE4_ROOT),
    "Total Original Images": len(image_paths),
    "Total Attack Types": len(ATTACK_FOLDERS),
    "Total Generated Images": len(attack_summary),
    "Attack List": list(ATTACK_FOLDERS.keys()),
    "Malicious Attack List": MALICIOUS_ATTACKS,
}

metadata_path = PHASE4_ROOT / "attack_metadata.npy"

np.save(metadata_path, metadata)  # np.save defaults allow_pickle=True, fine for a dict

print(f"Metadata saved : {metadata_path}")
print("  NOTE: load this back with np.load(metadata_path, allow_pickle=True).item() --")
print("  np.load's own default is allow_pickle=False, unlike np.save's.")

# ------------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 4 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"Original Images      : {len(image_paths)}")
print(f"Attack Types         : {len(ATTACK_FOLDERS)}")
print(f"Generated Images     : {len(attack_summary)}")

print("\nAttack Distribution")

display(stats_df)

print("=" * 70)
print("Output Directory")
print(PHASE4_ROOT)
print("=" * 70)

In [ ]:
# =============================================================================
# PHASE 4 : Attack Quality Evaluation
# =============================================================================
# NOTE on interpreting Rotation / Cropping / Resize rows below: these are
# geometric attacks, so the attacked image is pixel-MISALIGNED with the
# un-attacked watermarked image even before considering any real quality
# loss. Their MSE/PSNR/SSIM numbers will look far worse than the other
# (non-geometric) attacks for that reason -- this is expected, not a sign
# the attack itself is unusually destructive, and is consistent with the
# reference-assisted resynchronization already assumed for these attacks
# elsewhere in this scheme. Caveat this explicitly wherever these numbers
# are reported.

from skimage.metrics import peak_signal_noise_ratio
from skimage.metrics import structural_similarity
from skimage.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from scipy.stats import entropy

import cv2
import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# Entropy
# ------------------------------------------------------------

def image_entropy(img):

    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hist = cv2.calcHist([img],[0],None,[256],[0,256])
    hist = hist.ravel()

    hist = hist / np.sum(hist)

    return entropy(hist + 1e-12, base=2)


# ------------------------------------------------------------
# Calculate Metrics
# ------------------------------------------------------------

metric_results = []
wm_cache = {}  # load each watermarked image once, not once per attack

for row in attack_summary:

    img_name = row["Image"]
    if img_name not in wm_cache:
        wm_cache[img_name] = cv2.imread(str(INPUT_FOLDER / img_name))
    wm = wm_cache[img_name]

    atk_path = Path(row["Output_Path"])
    atk = cv2.imread(str(atk_path))

    if wm is None or atk is None:
        continue

    mse = mean_squared_error(wm, atk)
    
    # ------------------------------------------------------------
    # Mean Absolute Error (MAE)
    # ------------------------------------------------------------

    mae = mean_absolute_error(
        wm.reshape(-1),
        atk.reshape(-1)
    )

    psnr = peak_signal_noise_ratio(
        wm,
        atk,
        data_range=255
    )

    # Phase 3's embedding-quality SSIM is FULL-COLOR
    # (channel_axis=-1). Report both here so nothing is lost, but
    # SSIM_Color is the one directly comparable to Phase 3's numbers --
    # the original grayscale-only "SSIM" name is kept as SSIM_Gray so
    # existing downstream code/reports referencing it don't silently break.
    ssim_gray = structural_similarity(
        cv2.cvtColor(wm, cv2.COLOR_BGR2GRAY),
        cv2.cvtColor(atk, cv2.COLOR_BGR2GRAY),
        data_range=255
    )
    ssim_color = structural_similarity(
        wm, atk, channel_axis=-1, data_range=255
    )

    ent = image_entropy(atk)

    metric_results.append({

        "Image": row["Image"],
        "Attack": row["Attack"],

        "MSE": round(mse,6),
        "MAE": round(mae,6),
        "PSNR": round(psnr,4),
        "SSIM_Gray": round(ssim_gray,6),
        "SSIM_Color": round(ssim_color,6),
        "Entropy": round(ent,6)

    })


# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

metrics_df = pd.DataFrame(metric_results)

metrics_csv = PHASE4_ROOT / "attack_metrics.csv"

metrics_df.to_csv(
    metrics_csv,
    index=False
)

print("="*70)
print("Attack Quality Metrics Completed")
print("="*70)
print(metrics_df.head())

print("\nAverage Metrics")

display(
    metrics_df.groupby("Attack")[
        [
            "MSE",
            "MAE",
            "PSNR",
            "SSIM_Gray",
            "SSIM_Color",
            "Entropy"
        ]
    ].mean().round(4)
)
print("\nSaved to:")
print(metrics_csv)

# PHASE 5 — Extraction and decryption
- Blocks are regenerated from `K_TF` and the persistent Image ID; the raw 3072 per-block bits are saved (they feed Phase 6).
- Each image is decrypted with **its own** nonce/AAD (from its image ID) and checked against **its own** tag.
- Rotation and cropping are aligned to the reference watermarked image with `resynchronize_image` (ORB + RANSAC). This step is reference-assisted (non-blind).
- The receiver applies the same steps to every image and does not use the attack type.
- Majority vote over R → decryption without the tag check → NC / BER against the original watermark; `tag_ok` records whether AES-GCM authentication passed.


In [ ]:
# =============================================================================
# Receiver helper: reference-assisted registration (ORB + RANSAC) for rotation and cropping
# =============================================================================
IMAGE_SIZE = 512
GEOMETRIC_ATTACK_NAMES = {"Cropping", "Rotation"}     # reference-assisted resynchronization (non-blind)

def resynchronize_image(
    original_reference,
    attacked_rgb
):
    """
    Reference-assisted ORB + RANSAC affine registration.

    Used only for:
        Cropping
        Rotation
    """

    reference_gray = cv2.cvtColor(
        original_reference,
        cv2.COLOR_RGB2GRAY
    )

    attacked_gray = cv2.cvtColor(
        attacked_rgb,
        cv2.COLOR_RGB2GRAY
    )


    # -------------------------------------------------------------------------
    # ORB
    # -------------------------------------------------------------------------

    orb = cv2.ORB_create(
        nfeatures=5000,
        scaleFactor=1.2,
        nlevels=8,
        edgeThreshold=31,
        patchSize=31,
        fastThreshold=10
    )


    kp_reference, des_reference = (
        orb.detectAndCompute(
            reference_gray,
            None
        )
    )


    kp_attacked, des_attacked = (
        orb.detectAndCompute(
            attacked_gray,
            None
        )
    )


    if des_reference is None:

        raise RuntimeError(
            "No ORB descriptors found in reference image."
        )


    if des_attacked is None:

        raise RuntimeError(
            "No ORB descriptors found in attacked image."
        )


    # -------------------------------------------------------------------------
    # MATCHING
    # -------------------------------------------------------------------------

    matcher = cv2.BFMatcher(
        cv2.NORM_HAMMING,
        crossCheck=False
    )


    knn_matches = matcher.knnMatch(

        des_attacked,

        des_reference,

        k=2

    )


    good_matches = []


    for pair in knn_matches:

        if len(pair) < 2:
            continue

        m, n = pair

        if m.distance < 0.75 * n.distance:

            good_matches.append(m)


    if len(good_matches) < 4:

        raise RuntimeError(

            "Insufficient ORB matches for "
            "geometric synchronization. "

            f"Matches = {len(good_matches)}"

        )


    # -------------------------------------------------------------------------
    # CORRESPONDENCE POINTS
    # -------------------------------------------------------------------------

    src_points = np.float32([

        kp_attacked[
            m.queryIdx
        ].pt

        for m in good_matches

    ])


    dst_points = np.float32([

        kp_reference[
            m.trainIdx
        ].pt

        for m in good_matches

    ])


    # -------------------------------------------------------------------------
    # RANSAC AFFINE
    # -------------------------------------------------------------------------

    affine_matrix, inlier_mask = (

        cv2.estimateAffinePartial2D(

            src_points,

            dst_points,

            method=cv2.RANSAC,

            ransacReprojThreshold=5.0,

            maxIters=5000,

            confidence=0.99,

            refineIters=10

        )

    )


    if affine_matrix is None:

        raise RuntimeError(
            "RANSAC affine transformation failed."
        )


    inlier_count = int(
        np.sum(inlier_mask)
    )


    if inlier_count < 4:

        raise RuntimeError(

            "Insufficient RANSAC inliers. "

            f"Inliers = {inlier_count}"

        )


    # -------------------------------------------------------------------------
    # WARP TO ORIGINAL REFERENCE COORDINATES
    # -------------------------------------------------------------------------

    synchronized_rgb = cv2.warpAffine(

        attacked_rgb,

        affine_matrix,

        (
            IMAGE_SIZE,
            IMAGE_SIZE
        ),

        flags=cv2.INTER_LINEAR,

        borderMode=cv2.BORDER_REFLECT

    )


    return (
        synchronized_rgb,
        True,
        len(good_matches),
        inlier_count
    )


In [ ]:
# =============================================================================
# PHASE 5 : EXTRACTION + DECRYPTION (all images x all attacks + Clean)
# =============================================================================
P5_ROOT = RESULTS / "PHASE5_EXTRACTION"; P5_ROOT.mkdir(parents=True, exist_ok=True)
RAW_DIR = P5_ROOT / "raw_bits"; RAW_DIR.mkdir(exist_ok=True)
RESUME = True
# stale-cache guard: only resume if the cached results were written by this same run configuration
_sig = dict(run_mode=RUN_MODE, n_stems=len(embedding_maps), T0=float(EMBEDDING_THRESHOLD),
            margin=float(EMBED_MARGIN), cap=float(DIFF_CAP), R=int(REDUNDANCY),
            key_id=__import__("hashlib").sha256(K_TF_bytes).hexdigest()[:16],
            crypto="per-image nonce + AAD (v2)", block_selection="keyed BLAKE2b Fisher-Yates (v2)",
            colour="luma-delta, single rounding (v2)")
_sig_f = P5_ROOT / "run_signature.json"
if RESUME and any(RAW_DIR.glob("*.npz")) and (not _sig_f.exists() or json.load(open(_sig_f)) != _sig):
    print("[resume] cached results come from a different run configuration -> recomputing everything")
    RESUME = False
json.dump(_sig, open(_sig_f, "w"))

def load_rgb512(path):
    im = Image.open(path).convert("RGB")
    if im.size != (512, 512):
        im = im.resize((512, 512), Image.Resampling.LANCZOS)
    return np.array(im, dtype=np.uint8)

# per-image ciphertext / tag / crypto context (each image has its own nonce and tag)
def enc_bits_of(stem): return np.asarray(embedding_maps[stem]["enc_bits"], dtype=np.uint8)
def enc_R(stem):       return np.repeat(enc_bits_of(stem), REDUNDANCY)     # what was embedded, one entry per selected block
def ctx_of(stem):      return derive_wm_crypto(K_TF_bytes, embedding_maps[stem]["image_id"])
def tag_of(stem):      return bytes.fromhex(embedding_maps[stem]["gcm_tag_hex"])
ALL_CONDITIONS = ["Clean"] + list(ATTACK_FOLDERS.keys())
stems = sorted(embedding_maps.keys())
RAW_BITS = {}                                            # RAW_BITS[condition][stem] -> uint8[3072]
P5_rows = []

def received_path(stem, cond):
    base = f"{stem}_watermarked.png"
    return (watermarked_dir / base) if cond == "Clean" else (ATTACK_FOLDERS[cond] / base)

for cond in ALL_CONDITIONS:
    npz_f, csv_f = RAW_DIR / f"{cond}.npz", RAW_DIR / f"{cond}_results.csv"
    if RESUME and npz_f.exists() and csv_f.exists():
        z = np.load(npz_f, allow_pickle=True)
        RAW_BITS[cond] = {s: r for s, r in zip(z["stems"].tolist(), z["raw"])}
        P5_rows.extend(pd.read_csv(csv_f).to_dict("records"))
        print(f"[resume] {cond}: {len(z['stems'])} images")
        continue
    RAW_BITS[cond] = {}; rows_c = []
    for stem in tqdm(stems, desc=f"extract {cond}"):
        try:
            emap = embedding_maps[stem]
            blocks = emap["embedding_locations"]; rows, cols = blocks_rc(blocks)
            recv = load_rgb512(received_path(stem, cond))
            ref = load_rgb512(watermarked_dir / f"{stem}_watermarked.png")
            ok, matches, inl = 1, 0, 0
            if cond in GEOMETRIC_ATTACK_NAMES:
                recv, ok, matches, inl = resynchronize_image(ref, recv)
            raw = extract_raw_bits(recv, rows, cols)
            voted = majority_vote_bits(raw, REDUNDANCY)
            ctx_s = ctx_of(stem)
            pt = ctr_decrypt_bits(voted, ctx_s)
            try:
                gcm_decrypt_verify(voted, tag_of(stem), ctx_s); tag_ok = 1
            except Exception:
                tag_ok = 0
            RAW_BITS[cond][stem] = raw
            rows_c.append(dict(Image=stem, Attack=cond, Is_Malicious=cond in MALICIOUS_ATTACKS,
                               NC=ncc_bits(pt, WM_PLAIN_BITS), BER=ber_bits(pt, WM_PLAIN_BITS),
                               raw_error_rate=float(np.mean(raw != enc_R(stem))), tag_ok=tag_ok,
                               registration_ok=int(ok), orb_matches=int(matches), ransac_inliers=int(inl)))
        except Exception as e:
            print("  SKIPPED", stem, cond, "|", e)
    np.savez_compressed(npz_f, stems=np.array(list(RAW_BITS[cond].keys())), raw=np.array(list(RAW_BITS[cond].values())))
    pd.DataFrame(rows_c).to_csv(csv_f, index=False)
    P5_rows.extend(rows_c)

P5 = pd.DataFrame(P5_rows)
P5.to_csv(P5_ROOT / "extraction_results_all.csv", index=False)
P5_summary = (P5.groupby("Attack").agg(NC_mean=("NC", "mean"), NC_std=("NC", "std"), BER_mean=("BER", "mean"),
                                       tag_ok_rate=("tag_ok", "mean"), n=("NC", "size")).reindex(ALL_CONDITIONS))
P5_summary.to_csv(P5_ROOT / "extraction_summary_by_attack.csv")
display(P5_summary.round(4))
print("Note: tag_ok = 1 only when every encrypted bit survived; it is an integrity flag, not a robustness metric.")

# PHASE 6 — Authentication and tamper localization
**6.1 Siamese evidence** — distance between the reference watermarked image and the received image (deployed encoder from Phase 0).
**6.2 Calibration** — thresholds are chosen on **validation identities only** (seeded 30 % of images) and reported on the **held-out** 70 %.
**6.3 Decisions** — watermark-only, Siamese-only, and fusion (authentic only if both pass). Positive class = *tampered* (the four forgeries); Clean and the 19 benign/geometric attacks are *authentic*. `false_accept_rate` = forged images accepted as authentic; `false_reject_rate` = authentic/benign images rejected.
**6.4 Region-level localization** — scan-statistic test on the raw per-block bits. `alpha = 1e-8` and `min_rate = 0.4` are fixed in CONFIG.

In [ ]:
# =============================================================================
# 6.1  Siamese evidence for every (image, condition)
# =============================================================================
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import convnext_tiny
P6_ROOT = RESULTS / "PHASE6_AUTHENTICATION"; P6_ROOT.mkdir(parents=True, exist_ok=True)

class DeployEncoder(nn.Module):
    def __init__(self, ck):
        super().__init__()
        base = convnext_tiny(weights=None)
        self.backbone, self.pool = base.features, base.avgpool
        d = ck.get("embedding_dim", 128)
        self.projection = nn.Sequential(nn.Linear(768, 256), nn.ReLU(inplace=True), nn.Linear(256, d))
        self.backbone.load_state_dict(ck["encoder_state"]["backbone"])
        self.pool.load_state_dict(ck["encoder_state"]["pool"])
        self.projection.load_state_dict(ck["encoder_state"]["projection"])
    def forward(self, x):
        return F.normalize(self.projection(torch.flatten(self.pool(self.backbone(x)), 1)), p=2, dim=1)

_ck = torch.load(PHASE0_OUT / "deployment" / "authentication_encoder.pt", map_location=device)
deploy_encoder = DeployEncoder(_ck).to(device).eval()
PHASE0_SIAM_THRESHOLD = json.load(open(PHASE0_OUT / "deployment" / "auth_threshold.json"))["siamese_auth_threshold"]

@torch.no_grad()
def embed_paths(paths, bs=16):
    out = []
    for i in range(0, len(paths), bs):
        x = torch.stack([eval_transform(load_rgb512(p)) for p in paths[i:i + bs]]).to(device)
        out.append(deploy_encoder(x).cpu())
    return torch.cat(out)

dist_csv = P6_ROOT / "siamese_distances.csv"
if RESUME and dist_csv.exists():
    DIST = pd.read_csv(dist_csv)
else:
    ref_emb = dict(zip(stems, embed_paths([watermarked_dir / f"{s}_watermarked.png" for s in stems])))
    recs = []
    for cond in ALL_CONDITIONS:
        ps = [received_path(s, cond) for s in stems]
        E = embed_paths(ps)
        for s, e in zip(stems, E):
            r = ref_emb[s]
            recs.append(dict(Image=s, Attack=cond, siam_euclid=float(torch.norm(r - e)), siam_cosine=float(torch.dot(r, e))))
        print("embedded", cond)
    DIST = pd.DataFrame(recs); DIST.to_csv(dist_csv, index=False)

D6 = P5.merge(DIST, on=["Image", "Attack"], how="inner")
D6["tampered"] = D6["Is_Malicious"].astype(int)
print("rows:", len(D6), "| tampered:", int(D6.tampered.sum()), "| authentic/benign:", int((1 - D6.tampered).sum()))

In [ ]:
# =============================================================================
# 6.2 / 6.3  Thresholds on validation identities, decisions on held-out identities
# =============================================================================
from sklearn.metrics import roc_curve, roc_auc_score, matthews_corrcoef

_perm = np.random.default_rng(GLOBAL_SEED).permutation(stems)
_nv = int(round(VAL_FRACTION * len(stems)))
VAL_STEMS, TEST_STEMS = sorted(_perm[:_nv].tolist()), sorted(_perm[_nv:].tolist())
pd.DataFrame({"Image": VAL_STEMS + TEST_STEMS, "split": ["val"] * len(VAL_STEMS) + ["test"] * len(TEST_STEMS)}).to_csv(P6_ROOT / "identity_split.csv", index=False)
V, TS = D6[D6.Image.isin(VAL_STEMS)], D6[D6.Image.isin(TEST_STEMS)]
print(f"validation identities: {len(VAL_STEMS)}   held-out test identities: {len(TEST_STEMS)}")

def youden_threshold(score, tampered):
    """score: higher = more authentic. Returns the finite cut-off (accept if score >= cut) maximising TPR - FPR."""
    fpr, tpr, thr = roc_curve(1 - np.asarray(tampered), score)
    ok = np.isfinite(thr)                      # drop sklearn's infinite first threshold
    fpr, tpr, thr = fpr[ok], tpr[ok], thr[ok]
    return float(thr[int(np.argmax(tpr - fpr))])

T_NC   = youden_threshold(V.NC.values, V.tampered.values)
T_SIAM = -youden_threshold(-V.siam_euclid.values, V.tampered.values)     # accept if distance <= T_SIAM
print(f"calibrated thresholds  ->  NC >= {T_NC:.4f}   |   Siamese distance <= {T_SIAM:.4f}   (Phase 0 EER threshold: {PHASE0_SIAM_THRESHOLD:.4f})")

def cls_metrics(y, p):
    y, p = np.asarray(y).astype(int), np.asarray(p).astype(int)
    tp = int(((p == 1) & (y == 1)).sum()); fp = int(((p == 1) & (y == 0)).sum())
    tn = int(((p == 0) & (y == 0)).sum()); fn = int(((p == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
    return dict(accuracy=(tp + tn) / max(len(y), 1), precision=prec, recall=rec,
                f1=(2 * prec * rec / (prec + rec) if prec + rec else 0.0),
                mcc=float(matthews_corrcoef(y, p)) if len(set(y)) > 1 and len(set(p)) > 1 else 0.0,
                false_accept_rate=fn / max(tp + fn, 1), false_reject_rate=fp / max(fp + tn, 1),
                balanced_accuracy=0.5 * (rec + tn / max(tn + fp, 1)), tp=tp, fp=fp, tn=tn, fn=fn)

def decisions(df):
    wm_t = (df.NC.values < T_NC)                         # watermark says tampered
    si_t = (df.siam_euclid.values > T_SIAM)              # Siamese (validation-calibrated) says tampered
    p0_t = (df.siam_euclid.values > PHASE0_SIAM_THRESHOLD)
    return {"watermark only": wm_t, "Siamese only (val-calibrated)": si_t, "Siamese only (Phase 0 threshold)": p0_t,
            "fusion: watermark OR Siamese (val-calibrated)": wm_t | si_t,
            "fusion: watermark OR Siamese (Phase 0 threshold)": wm_t | p0_t}

rows = []
for name, pred in decisions(TS).items():
    rows.append(dict(method=name, **cls_metrics(TS.tampered.values, pred)))
AUTH_TABLE = pd.DataFrame(rows); AUTH_TABLE.to_csv(P6_ROOT / "authentication_metrics_test.csv", index=False)
display(AUTH_TABLE.round(4))
AUC_WM  = roc_auc_score(TS.tampered, -TS.NC); AUC_SIAM = roc_auc_score(TS.tampered, TS.siam_euclid)
print(f"ROC-AUC on held-out identities  ->  watermark NC: {AUC_WM:.4f}   Siamese distance: {AUC_SIAM:.4f}")

# benign vs malicious breakdown + attack-wise rejection rates (watermark-only and fusion)
d_wm, d_fu = decisions(TS)["watermark only"], decisions(TS)["fusion: watermark OR Siamese (val-calibrated)"]
TS2 = TS.assign(rej_watermark=d_wm.astype(int), rej_fusion=d_fu.astype(int))
ATTACKWISE = TS2.groupby("Attack").agg(NC_mean=("NC", "mean"), BER_mean=("BER", "mean"), siam_dist_mean=("siam_euclid", "mean"),
                                       rejected_watermark=("rej_watermark", "mean"), rejected_fusion=("rej_fusion", "mean"),
                                       n=("NC", "size")).reindex(ALL_CONDITIONS)
ATTACKWISE.to_csv(P6_ROOT / "attackwise_decisions_test.csv"); display(ATTACKWISE.round(4))
json.dump(dict(T_NC=T_NC, T_SIAM=T_SIAM, phase0_siamese_threshold=PHASE0_SIAM_THRESHOLD, val_images=len(VAL_STEMS),
               test_images=len(TEST_STEMS), auc_watermark=AUC_WM, auc_siamese=AUC_SIAM), open(P6_ROOT / "calibrated_thresholds.json", "w"), indent=2)

In [ ]:
# =============================================================================
# 6.4 / 6.5  Region-level tamper localization
# =============================================================================
LOC_ROOT = P6_ROOT / "localization"; LOC_ROOT.mkdir(exist_ok=True)
FORGERIES = list(MALICIOUS_ATTACKS)
BENIGN_CONDS = [c for c in ALL_CONDITIONS if c not in FORGERIES]
GT_MASKS = {}
def gt_cells(stem, cond):
    k = (stem, cond)
    if k not in GT_MASKS:
        m = cv2.imread(str(MASK_FOLDERS[cond] / f"{stem}_watermarked.png"), cv2.IMREAD_GRAYSCALE)
        GT_MASKS[k] = region_ground_truth(m)
    return GT_MASKS[k]

def run_loc(stem_list, alpha, min_rate, conds):
    out = {}
    for c in conds:
        for s in stem_list:
            if s in RAW_BITS[c]:
                cells, _, _, _ = localize_tampering(embedding_maps[s]["embedding_locations"], RAW_BITS[c][s], enc_R(s),
                                                    alpha=alpha, min_rate=min_rate)
                out[(s, c)] = cells
    return out

GEOMETRIC = ("Cropping", "Rotation")   # break block alignment: reported separately, excluded from the false-flag constraint

def score_loc(res):
    tp = fp = fn = 0; ious = []; flagged = {}
    for (s, c), cells in res.items():
        if c in FORGERIES:
            r = prf_iou(cells, gt_cells(s, c)); tp += r["tp"]; fp += r["fp"]; fn += r["fn"]; ious.append(r["iou"])
        else:
            flagged.setdefault(c, []).append(bool(cells.any()))
    P = tp / max(tp + fp, 1); Rc = tp / max(tp + fn, 1)
    allb = [x for c_, v in flagged.items() if c_ not in GEOMETRIC for x in v]
    return dict(precision=P, recall=Rc, f1=(2 * P * Rc / (P + Rc) if P + Rc else 0.0), iou_micro=tp / max(tp + fp + fn, 1),
                iou_mean=float(np.mean(ious)) if ious else 0.0, benign_false_flag=float(np.mean(allb)) if allb else 0.0), flagged

_sc_val, _ = score_loc(run_loc(VAL_STEMS, LOC_ALPHA, LOC_MIN_RATE, ALL_CONDITIONS))
TUNE = pd.DataFrame([dict(alpha=LOC_ALPHA, min_rate=LOC_MIN_RATE, **_sc_val)]); TUNE.to_csv(LOC_ROOT / "localization_validation.csv", index=False)
LOC_CONSTRAINT_MET = bool(_sc_val["benign_false_flag"] <= MAX_BENIGN_FALSE_FLAG)
print(f"localization setting: alpha={LOC_ALPHA:g}  min_rate={LOC_MIN_RATE}  (validation benign false-flag {_sc_val['benign_false_flag']:.2%}; limit {MAX_BENIGN_FALSE_FLAG:.0%} met: {LOC_CONSTRAINT_MET})")

LOC_TEST_RES = run_loc(TEST_STEMS, LOC_ALPHA, LOC_MIN_RATE, ALL_CONDITIONS)
LOC_TEST, flagged = score_loc(LOC_TEST_RES)
per_attack = []
for c in FORGERIES:
    sc, _ = score_loc({k: v for k, v in LOC_TEST_RES.items() if k[1] == c}); per_attack.append(dict(Attack=c, **{k: sc[k] for k in ("precision", "recall", "f1", "iou_mean")}))
for c in BENIGN_CONDS:
    if c in flagged: per_attack.append(dict(Attack=c, false_flag_rate=float(np.mean(flagged[c]))))
LOC_TABLE = pd.DataFrame(per_attack); LOC_TABLE.to_csv(LOC_ROOT / "localization_by_attack_test.csv", index=False)
json.dump(dict(alpha=LOC_ALPHA, min_rate=LOC_MIN_RATE, constraint_met=LOC_CONSTRAINT_MET, test=LOC_TEST), open(LOC_ROOT / "localization_test_summary.json", "w"), indent=2)
print("HELD-OUT localization:", {k: round(v, 4) for k, v in LOC_TEST.items()})
display(LOC_TABLE.round(4))

# operating curve over alpha (min_rate fixed) + example maps
curve = []
for a in (1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8):
    sc, _ = score_loc(run_loc(TEST_STEMS, a, LOC_MIN_RATE, ALL_CONDITIONS)); curve.append(dict(alpha=a, **sc))
CURVE = pd.DataFrame(curve); CURVE.to_csv(LOC_ROOT / "localization_operating_curve_test.csv", index=False)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(CURVE.benign_false_flag, CURVE.recall, "o-")
for _, r in CURVE.iterrows(): ax.annotate(f"{r.alpha:g}", (r.benign_false_flag, r.recall), fontsize=7)
ax.set_xlabel("benign false-flag rate"); ax.set_ylabel("cell-level recall (forgeries)"); ax.set_title("Localization operating curve (varying alpha)")
fig.tight_layout(); fig.savefig(LOC_ROOT / "localization_operating_curve.png", dpi=300); plt.show()

ex = [(s, c) for c in FORGERIES for s in TEST_STEMS[:2] if (s, c) in LOC_TEST_RES]
if ex:
    fig, axs = plt.subplots(len(ex), 4, figsize=(11, 2.7 * len(ex)))
    axs = np.atleast_2d(axs)
    for r, (s, c) in enumerate(ex):
        im = load_rgb512(received_path(s, c)); gt = flagged_to_mask(gt_cells(s, c)); pr = flagged_to_mask(LOC_TEST_RES[(s, c)])
        _, rate, _, _ = localize_tampering(embedding_maps[s]["embedding_locations"], RAW_BITS[c][s], enc_R(s), alpha=LOC_ALPHA, min_rate=LOC_MIN_RATE)
        for a_, (img, t, kw) in zip(axs[r], [(im, f"{c}", {}), (gt, "ground truth", dict(cmap="gray")), (pr, "predicted", dict(cmap="gray")),
                                              (rate, "window error rate", dict(cmap="magma", vmin=0, vmax=0.6))]):
            a_.imshow(img, **kw); a_.set_title(t, fontsize=8); a_.axis("off")
    fig.tight_layout(); fig.savefig(LOC_ROOT / "localization_examples.png", dpi=300); plt.show()

## 6.6 / 6.7 — Decision rule and JPEG-localization robustness
Three-way image-level verdict (Authentic / Localized Tampering / Global Modification), so a benign global attack is not mistaken for a local forgery, plus a check of how localization holds up when a forged image is JPEG-recompressed afterwards.

In [ ]:
# =============================================================================
# 6.6  Image-level decision rule: AUTHENTIC / LOCALIZED_TAMPERING /
#      GLOBAL_MODIFICATION (plus TAMPERING_NOT_LOCALIZED)
# The Siamese verdict decides WHETHER the content was forged; the watermark
# decides WHERE. A benign global attack (JPEG, blur, noise, cropping, ...) can
# disturb the embedded bits, but it is not reported as a local forgery unless
# the Siamese network also finds the content changed.
# Evaluated on the held-out test identities with the fixed
# localization setting (LOC_ALPHA, LOC_MIN_RATE) and Siamese threshold (T_SIAM).
# =============================================================================

DECISION_ROOT = P6_ROOT / "decision_rule"
DECISION_ROOT.mkdir(parents=True, exist_ok=True)

SIAM_LOOKUP = D6.set_index(["Image", "Attack"])["siam_euclid"].to_dict()

decision_rows = []
for cond in ALL_CONDITIONS:
    if cond not in RAW_BITS:
        continue
    for s in TEST_STEMS:
        if s not in RAW_BITS[cond]:
            continue
        emap = embedding_maps[s]
        cells, rate, pval, p0 = localize_tampering(
            emap["embedding_locations"], RAW_BITS[cond][s], enc_R(s),
            alpha=LOC_ALPHA, min_rate=LOC_MIN_RATE)
        wm = classify_tamper_level(emap["embedding_locations"], RAW_BITS[cond][s], enc_R(s), cells)
        siam_tampered = SIAM_LOOKUP[(s, cond)] > T_SIAM
        if siam_tampered:
            level = "LOCALIZED_TAMPERING" if cells.any() else "TAMPERING_NOT_LOCALIZED"
        else:
            level = "AUTHENTIC" if wm["level"] == "AUTHENTIC" else "GLOBAL_MODIFICATION"
        decision_rows.append(dict(
            image=s, condition=cond, any_cell_flagged=bool(cells.any()),
            ground_truth="MALICIOUS" if cond in MALICIOUS_ATTACKS else "BENIGN/CLEAN",
            siamese_tampered=bool(siam_tampered), level=level,
            flagged_fraction=wm["flagged_fraction"], background_error_rate=wm["background_error_rate"]))

DECISIONS = pd.DataFrame(decision_rows)
DECISIONS.to_csv(DECISION_ROOT / "image_level_decisions.csv", index=False)

print("Per-condition decision-level breakdown (counts):")
confusion = (DECISIONS.groupby(["ground_truth", "condition", "level"])
             .size().rename("count").reset_index())
display(confusion)

benign_rows = DECISIONS[DECISIONS.ground_truth == "BENIGN/CLEAN"]
malicious_rows = DECISIONS[DECISIONS.ground_truth == "MALICIOUS"]
false_localized_rate = float((benign_rows.level == "LOCALIZED_TAMPERING").mean())
false_tamper_any = float(benign_rows.level.isin(["LOCALIZED_TAMPERING", "TAMPERING_NOT_LOCALIZED"]).mean())
correctly_localized_rate = float((malicious_rows.level == "LOCALIZED_TAMPERING").mean())
detected_rate = float(malicious_rows.level.isin(["LOCALIZED_TAMPERING", "TAMPERING_NOT_LOCALIZED"]).mean())

print("\nHeld-out test identities:")
print(f"  Benign/clean records reported as LOCALIZED_TAMPERING : {false_localized_rate:.4f}  "
      f"({(benign_rows.level == 'LOCALIZED_TAMPERING').sum()} / {len(benign_rows)})")
print(f"  Benign/clean records reported as any tampering       : {false_tamper_any:.4f}")
print(f"  Forgeries detected (any tampering verdict)           : {detected_rate:.4f}")
print(f"  Forgeries detected AND localized                     : {correctly_localized_rate:.4f}  (n = {len(malicious_rows)})")

decision_summary = dict(
    false_localized_alarm_rate_on_benign=false_localized_rate,
    false_tamper_alarm_rate_on_benign=false_tamper_any,
    forgery_detection_rate=detected_rate,
    correct_localized_rate_on_malicious=correctly_localized_rate,
    n_benign_clean=int(len(benign_rows)), n_malicious=int(len(malicious_rows)),
    siamese_threshold=float(T_SIAM), loc_alpha=float(LOC_ALPHA), loc_min_rate=float(LOC_MIN_RATE))
json.dump(decision_summary, open(DECISION_ROOT / "decision_summary.json", "w"), indent=2)
print("\nSaved:", DECISION_ROOT / "image_level_decisions.csv", "and decision_summary.json")

# -----------------------------------------------------------------------------
# Watermark-side detector for Table 7: "tampered" if at least one 16x16 cell is flagged
# (the NC-only detector above uses a calibrated NC threshold of 1.0, i.e. any bit error -> tampered).
# Reported with and without the geometric attacks, which break block alignment.
# -----------------------------------------------------------------------------
_y = (DECISIONS.ground_truth == "MALICIOUS").astype(int).values
_p = DECISIONS.any_cell_flagged.astype(int).values
_ng = ~DECISIONS.condition.isin(["Cropping", "Rotation"]).values
WM_CELL_ROWS = [dict(method="watermark: >=1 flagged cell (all conditions)", **cls_metrics(_y, _p)),
                dict(method="watermark: >=1 flagged cell (excl. cropping/rotation)", **cls_metrics(_y[_ng], _p[_ng]))]
_s = DECISIONS.siamese_tampered.astype(int).values
WM_CELL_ROWS.append(dict(method="fusion: Siamese OR >=1 flagged cell (excl. cropping/rotation)", **cls_metrics(_y[_ng], (_s | _p)[_ng])))
AUTH_TABLE_EXT = pd.concat([AUTH_TABLE, pd.DataFrame(WM_CELL_ROWS)], ignore_index=True)
AUTH_TABLE_EXT.to_csv(P6_ROOT / "authentication_metrics_test_with_cell_detector.csv", index=False)
display(AUTH_TABLE_EXT.round(4))


In [ ]:
# =============================================================================
# 6.7  Localization robustness
# under JPEG re-compression: a real forgery is almost never left uncompressed
# in the wild. Take each already-generated malicious-attack image, JPEG-
# re-compress it at several qualities, and measure how localization
# precision/recall/F1/IoU degrade as quality drops.
# =============================================================================

JPEG_LOC_ROOT = P6_ROOT / "jpeg_localization_robustness"
JPEG_LOC_ROOT.mkdir(parents=True, exist_ok=True)

N_JPEG_LOC_IMAGES = min(20, len(TEST_STEMS))
JPEG_LOC_QUALITIES = [95, 80, 60, 40]
jpeg_loc_stems = random.Random(GLOBAL_SEED).sample(TEST_STEMS, N_JPEG_LOC_IMAGES)

def jpeg_recompress(rgb, quality):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    ok, enc = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)

jpeg_loc_rows = []
for forgery_cond in FORGERIES:
    for s in tqdm(jpeg_loc_stems, desc=f"JPEG-after-{forgery_cond}"):
        img_path = ATTACK_FOLDERS[forgery_cond] / f"{s}_watermarked.png"
        if not img_path.exists():
            continue
        forged_rgb = load_rgb512(img_path)
        emap = embedding_maps[s]
        rows, cols = blocks_rc(emap["embedding_locations"])
        gt = gt_cells(s, forgery_cond)

        for q in JPEG_LOC_QUALITIES:
            recompressed = jpeg_recompress(forged_rgb, q)
            raw = extract_raw_bits(recompressed, rows, cols)
            cells, _, _, _ = localize_tampering(emap["embedding_locations"], raw, enc_R(s),
                                                alpha=LOC_ALPHA, min_rate=LOC_MIN_RATE)   # same setting as Table 8
            r = prf_iou(cells, gt)
            jpeg_loc_rows.append(dict(image=s, forgery=forgery_cond, jpeg_quality=q, **r))

JPEG_LOC_DF = pd.DataFrame(jpeg_loc_rows)
JPEG_LOC_DF.to_csv(JPEG_LOC_ROOT / "jpeg_localization_robustness.csv", index=False)

jpeg_loc_summary = (JPEG_LOC_DF.groupby(["forgery", "jpeg_quality"])[["precision", "recall", "f1", "iou"]]
                     .mean().reset_index())
jpeg_loc_summary.to_csv(JPEG_LOC_ROOT / "jpeg_localization_robustness_summary.csv", index=False)
print("Localization F1 by forgery type and JPEG quality after re-compression:")
display(jpeg_loc_summary.round(4))

fig, ax = plt.subplots(figsize=(7, 4.5))
for forgery_cond, g in jpeg_loc_summary.groupby("forgery"):
    g_sorted = g.sort_values("jpeg_quality")
    ax.plot(g_sorted.jpeg_quality, g_sorted.f1, marker="o", label=forgery_cond)
ax.set_xlabel("JPEG quality after forgery"); ax.set_ylabel("Localization F1")
ax.set_title("Localization robustness under post-forgery JPEG re-compression")
ax.legend(fontsize=8); ax.invert_xaxis()
fig.tight_layout(); fig.savefig(JPEG_LOC_ROOT / "jpeg_localization_robustness.png", dpi=300)
plt.show()
print("\nSaved:", JPEG_LOC_ROOT / "jpeg_localization_robustness_summary.csv")


# PHASE 7 — Security, ablation, four-watermark test, comparison
Everything here is computed from the run above and written to CSV/JSON. The cell 7.6 collects the headline values into `manuscript_numbers.json`.

In [ ]:
# =============================================================================
# 7.1  Security analysis
# =============================================================================
from math import erfc, sqrt
P7_ROOT = RESULTS / "PHASE7_ANALYSIS"; P7_ROOT.mkdir(parents=True, exist_ok=True)
kb = K_TF.astype(np.uint8); n = kb.size
monobit_p = erfc(abs(2 * int(kb.sum()) - n) / sqrt(2.0 * n))
pi = kb.mean(); runs = 1 + int((kb[1:] != kb[:-1]).sum())
runs_p = erfc(abs(runs - 2 * n * pi * (1 - pi)) / (2 * sqrt(2 * n) * pi * (1 - pi)))
SECURITY = dict(key_bits=n, key_space=f"2^{n}", ones=int(kb.sum()), monobit_p=float(monobit_p), runs_p=float(runs_p),
                min_entropy_per_bit=float(-np.log2(max(pi, 1 - pi))),
                note="K_TF comes from a SIMULATED BB84-style TF-QKD post-processing chain, not a physical TF-QKD link.")
meta_f = RESULTS / "TF_QKD_RESULTS" / "phase1_key_generation_metadata.json"
if meta_f.exists(): SECURITY["phase1_metadata"] = json.load(open(meta_f))

# wrong-key test (one flipped bit of K_TF)
bad = K_TF.copy(); bad[0] ^= 1; bad_bytes = np.packbits(bad).tobytes()
ov, nc_wrongblocks, nc_wrongall, nc_right = [], [], [], []
for s in TEST_STEMS[:20]:
    emap = embedding_maps[s]; rows, cols = blocks_rc(emap["embedding_locations"])
    wb, _ = select_blocks_tfqkd(bad_bytes, emap["image_id"], 512, BLOCK_SIZE, W_enc_bits.size * REDUNDANCY)
    wr, wc = blocks_rc(wb)
    ov.append(len(set(emap["embedding_locations"][i]["index"] for i in range(len(rows))) & set(b["index"] for b in wb)) / len(rows))
    wm = load_rgb512(watermarked_dir / f"{s}_watermarked.png")
    f = lambda r_, c_, ctx: ncc_bits(ctr_decrypt_bits(majority_vote_bits(extract_raw_bits(wm, r_, c_), REDUNDANCY), ctx), WM_PLAIN_BITS)
    ctx_ok, ctx_bad = ctx_of(s), derive_wm_crypto(bad_bytes, emap["image_id"])
    nc_right.append(f(rows, cols, ctx_ok)); nc_wrongblocks.append(f(wr, wc, ctx_ok)); nc_wrongall.append(f(wr, wc, ctx_bad))
SECURITY["wrong_key_one_bit_flip"] = dict(images=len(ov), block_overlap_mean=float(np.mean(ov)), expected_chance_overlap=float(W_enc_bits.size * REDUNDANCY / 4096),
                                          nc_correct_key=float(np.mean(nc_right)), nc_wrong_blocks_right_crypto=float(np.mean(nc_wrongblocks)),
                                          nc_wrong_key_everything=float(np.mean(nc_wrongall)))
SECURITY["limitations"] = ("3072 of 4096 blocks are selected, so any two selections overlap in ~75 % of blocks by chance: the block map hides little. "
                           "Secrecy of the watermark content rests on AES-256-GCM, not on the block map. Extraction here is reference-assisted for geometric attacks. "
                           "With KEY_MODE = 'reproducible' the key is regenerated from a public seed and is a test key only.")
json.dump(SECURITY, open(P7_ROOT / "security_analysis.json", "w"), indent=2, default=str)
display(pd.json_normalize(SECURITY).T)

## 7.1b / 7.1c — Security additions
Multi-session NIST randomness (not only the one final key) and a key-sensitivity (avalanche) check across independent TF-QKD sessions.

In [ ]:
# =============================================================================
# 7.1b  Randomness assessment
# across MANY independent K_TF sessions, not just the one final key above.
# A single 256-bit key passing monobit/runs is weak evidence; this re-runs
# the whole Phase 1 TF-QKD session-generation process (generate_tfqkd_key)
# with many different master seeds and reports the aggregate pass rate.
# =============================================================================

N_RANDOMNESS_SESSIONS = 30
randomness_rows = []
for session_seed in range(1000, 1000 + N_RANDOMNESS_SESSIONS):
    result = generate_tfqkd_key(master_seed=session_seed, target_bits=FINAL_KEY_BITS, max_retries=MAX_RETRIES)
    if not result.get("success"):
        randomness_rows.append(dict(seed=session_seed, success=False))
        continue
    randomness_rows.append(dict(
        seed=session_seed, success=True,
        entropy_bits_per_bit=result["entropy_bits_per_bit"],
        monobit_pass=result["nist_monobit"]["pass"], monobit_p=result["nist_monobit"]["p_value"],
        runs_pass=result["nist_runs"]["pass"], runs_p=result["nist_runs"]["p_value"],
        attempts_used=result["attempts_used"],
    ))

RANDOMNESS_DF = pd.DataFrame(randomness_rows)
RANDOMNESS_DF.to_csv(P7_ROOT / "key_randomness_multisession.csv", index=False)

n_success = int(RANDOMNESS_DF.success.sum())
monobit_pass_rate = float(RANDOMNESS_DF.loc[RANDOMNESS_DF.success, "monobit_pass"].mean()) if n_success else float("nan")
runs_pass_rate = float(RANDOMNESS_DF.loc[RANDOMNESS_DF.success, "runs_pass"].mean()) if n_success else float("nan")

print(f"Generated {n_success} / {N_RANDOMNESS_SESSIONS} independent K_TF sessions successfully.")
print(f"  NIST monobit pass rate : {monobit_pass_rate:.4f}")
print(f"  NIST runs pass rate    : {runs_pass_rate:.4f}")
print("(Same caveat as the single-key test above: this is the monobit + runs subtests")
print(" of NIST SP 800-22 only, not the full 15-test suite, which needs far more bits per key.)")

SECURITY["multisession_randomness"] = dict(
    n_sessions=N_RANDOMNESS_SESSIONS, n_success=n_success,
    monobit_pass_rate=monobit_pass_rate, runs_pass_rate=runs_pass_rate,
)
json.dump(SECURITY, open(P7_ROOT / "security_analysis.json", "w"), indent=2, default=str)
print("\nSaved:", P7_ROOT / "key_randomness_multisession.csv")


In [ ]:
# =============================================================================
# 7.1c  Key sensitivity /
# avalanche check: K_TF depends on the simulated quantum channel's session
# seed through the full sift -> QBER -> reconcile -> privacy-amplify chain.
# This checks that changing only the session seed (i.e. a different
# simulated quantum exchange) produces an essentially UNCORRELATED key --
# about half the bits should differ, with no structural bias either way.
# This is in addition to the wrong-key BLOCK-MAP-overlap test already run
# above; that one tests the block-selection step, this one tests the key-
# generation step itself.
# =============================================================================

N_KEY_SENSITIVITY_PAIRS = 15
hamming_fractions = []
for pair_idx in range(N_KEY_SENSITIVITY_PAIRS):
    seed_a = 5000 + 2 * pair_idx
    seed_b = seed_a + 1  # adjacent seed -> a different simulated quantum session, not a bit flip of the same one
    r_a = generate_tfqkd_key(master_seed=seed_a, target_bits=FINAL_KEY_BITS, max_retries=MAX_RETRIES)
    r_b = generate_tfqkd_key(master_seed=seed_b, target_bits=FINAL_KEY_BITS, max_retries=MAX_RETRIES)
    if not (r_a.get("success") and r_b.get("success")):
        continue
    k_a = np.array(r_a["final_key_bits"], dtype=np.uint8)
    k_b = np.array(r_b["final_key_bits"], dtype=np.uint8)
    hd = float(np.mean(k_a != k_b))
    hamming_fractions.append(dict(seed_a=seed_a, seed_b=seed_b, hamming_fraction=hd))

KEY_SENS_DF = pd.DataFrame(hamming_fractions)
KEY_SENS_DF.to_csv(P7_ROOT / "key_sensitivity.csv", index=False)

mean_hd = float(KEY_SENS_DF.hamming_fraction.mean()) if len(KEY_SENS_DF) else float("nan")
print(f"Key-sensitivity (avalanche) across {len(KEY_SENS_DF)} independent session pairs:")
print(f"  mean Hamming distance fraction = {mean_hd:.4f}  (ideal, uncorrelated keys -> 0.5000)")
display(KEY_SENS_DF.round(4))

SECURITY["key_sensitivity"] = dict(n_pairs=len(KEY_SENS_DF), mean_hamming_fraction=mean_hd)
json.dump(SECURITY, open(P7_ROOT / "security_analysis.json", "w"), indent=2, default=str)
print("\nSaved:", P7_ROOT / "key_sensitivity.csv")


In [ ]:
# =============================================================================
# 7.2  Ablation  (held-out test images, in-memory pipeline)
# =============================================================================
ABL_IMAGES = 30
abl_paths = sorted((PREPARED / "test" / "authentic").glob("*.png"))
abl_paths = random.Random(GLOBAL_SEED).sample(abl_paths, min(ABL_IMAGES, len(abl_paths)))
abl_imgs = [load_host_image(p, 512) for p in abl_paths]
FZ = dict(T0=EMBEDDING_THRESHOLD, margin=EMBED_MARGIN, cap=DIFF_CAP, R=REDUNDANCY)
semfn = lambda rgb, blocks: semantic_norm_selected(rgb, blocks)
DEFAULT_KEY = b"\x00" * 32                                          # a publicly known placeholder key
variants = {
    "Full proposed":               dict(params=FZ, K=K_TF_bytes, sem_fn=semfn),
    "No TF-QKD (public key)":      dict(params=FZ, K=DEFAULT_KEY, sem_fn=semfn),
    "No encryption":               dict(params=FZ, K=K_TF_bytes, sem_fn=semfn, encrypt=False),
    "No adaptive threshold":       dict(params=FZ, K=K_TF_bytes, sem_fn=semfn, adaptive=False),
    "No Siamese guidance":         dict(params=FZ, K=K_TF_bytes, sem_fn=None),
}
for r_ in (1, 2, 3, 4):
    variants[f"R = {r_}"] = dict(params={**FZ, "R": r_}, K=K_TF_bytes, sem_fn=semfn)
abl_rows = []
for name, v in tqdm(variants.items(), desc="ablation"):
    res = evaluate_embedding(abl_imgs, v["params"], WM_PLAIN_BITS, v["K"], sem_fn=v["sem_fn"], encrypt=v.get("encrypt", True),
                             adaptive=v.get("adaptive", True), do_adv=True, id_prefix="abl")
    abl_rows.append(dict(variant=name, **{k: res[k] for k in res if not k.startswith("nc_")}))
ABLATION = pd.DataFrame(abl_rows); ABLATION.to_csv(P7_ROOT / "ablation.csv", index=False)
display(ABLATION.round(4))
print("adv_no_key_nc: NC of an adversary who tries the public placeholder key; adv_knows_blocks_nc: adversary who knows the block map but not K_TF.")

## 7.3b — Three watermarks on 30 test images
Main watermark, second watermark (`WATERMARK_PATH_2`, or a synthetic pattern if the file is missing) and a random-bit watermark, each embedded in the same 30 held-out test images with the fixed constants; reports PSNR/SSIM, clean and attacked NC/BER and splice-localization precision/recall/F1.


In [ ]:
# =============================================================================
# 7.3b  Three watermarks on 30 held-out test images (same fixed constants, same images)
#       Main watermark, second watermark and random bits
# =============================================================================
def to_bits_32(arr):
    a = np.asarray(arr)
    if a.ndim == 3: a = a.mean(axis=2)
    if a.shape != (32, 32): a = np.array(Image.fromarray(a.astype(np.uint8)).resize((32, 32), Image.Resampling.NEAREST))
    return (a > 127).astype(np.uint8).ravel() if a.max() > 1 else (a > 0).astype(np.uint8).ravel()

if WATERMARK_PATH_2.exists():
    WM2 = to_bits_32(np.array(Image.open(WATERMARK_PATH_2).convert("L"))); wm2_src = str(WATERMARK_PATH_2)
else:
    yy, xx = np.mgrid[0:32, 0:32]
    WM2 = (((yy // 4 + xx // 4) % 2 == 0) ^ (np.abs(yy - xx) < 3)).astype(np.uint8).ravel(); wm2_src = "synthetic pattern (no second watermark file found)"
WM_RAND = np.random.default_rng(GLOBAL_SEED + 1).integers(0, 2, 1024).astype(np.uint8)
rows_ = []
for name, bits in (("main watermark", WM_PLAIN_BITS), ("second watermark", WM2), ("random-bit watermark", WM_RAND)):
    res = evaluate_embedding(abl_imgs, FZ, bits, K_TF_bytes, wm_id=name.replace(" ", "_"), sem_fn=semfn, do_loc=True, id_prefix="wm2")
    rows_.append(dict(watermark=name, ones_fraction=float(bits.mean()), **{k: res[k] for k in res if not k.startswith("nc_")}))
WM2_TABLE = pd.DataFrame(rows_); WM2_TABLE.to_csv(P7_ROOT / "second_watermark.csv", index=False)
print("second watermark source:", wm2_src); display(WM2_TABLE.round(4))

## 7.3 — Four kinds of watermark on one image
Eagle logo, ASCII text, texture and random bits are embedded one at a time in the same host image (lion, Image 1) with the same pipeline; only the 1024 plaintext bits differ. Writes 16 panel images and two CSV files for the figure.

In [ ]:
# =============================================================================
# 7.3  Four kinds of watermark on ONE image (lion = Image 1 = Au_ani_00030) 
#      Uses the existing pipeline (fixed constants, K_TF, AES-GCM, block selection, attacks, extraction);
#      only the 1024 plaintext bits differ:  eagle logo | ASCII text | texture | random bits.
#      Output: 16 individual images (4 watermarks x 4 rows) + 2 CSVs (PSNR / NC / BER, and a detailed one); the table is built in LaTeX
# =============================================================================
import numpy as np, matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

LION_STEM    = "Au_ani_00030"
TEXT_MESSAGE = "Digital image authentication 2026"          # <- the text watermark (ASCII, max 128 characters)
FZ = dict(T0=EMBEDDING_THRESHOLD, margin=EMBED_MARGIN, cap=DIFF_CAP, R=REDUNDANCY)       # fixed constants
R_ = int(FZ["R"])
_b2r = lambda a: np.ascontiguousarray(a[:, :, ::-1])                                       # RGB <-> BGR (Phase-4 attack functions use BGR)

# ---- 1) the four plaintext bit-vectors (1 = white pixel, same convention as the main watermark) ----------------------
def text_to_bits(msg, n=1024):
    b = np.unpackbits(np.frombuffer(msg.encode("ascii"), dtype=np.uint8))
    assert b.size <= n, "text too long: max 128 ASCII characters"
    return np.concatenate([b, np.zeros(n - b.size, np.uint8)]).astype(np.uint8)

def bits_to_text(bits, n_chars):
    by = np.packbits(np.asarray(bits[:8 * n_chars], dtype=np.uint8))
    return "".join(chr(c) if 32 <= c < 127 else "?" for c in by)

def make_texture(seed=GLOBAL_SEED + 2, sigma=1.5):                                          # organic texture: smooth noise, thresholded
    f = gaussian_filter(np.random.default_rng(seed).standard_normal((32, 32)), sigma=sigma, mode="wrap")
    return (f > np.median(f)).astype(np.uint8).ravel()

KINDS = {                                              # name -> (bits, attack condition name as in Phase 4)
    "Eagle logo":  (np.asarray(WM_PLAIN_BITS, dtype=np.uint8),                           "JPEG"),
    "Text":        (text_to_bits(TEXT_MESSAGE),                                           "JPEG"),
    "Texture":     (make_texture(),                                                       "Median_Filter"),
    "Random bits": (np.random.default_rng(GLOBAL_SEED + 1).integers(0, 2, 1024).astype(np.uint8), "Contrast"),
}
ATTACK_FN = {                                          # manuscript parameters (Tables 3 and 4)
    "Salt_Pepper":   lambda im, rng: _b2r(salt_pepper_noise(_b2r(im), amount=0.02, rng=rng)),
    "JPEG":          lambda im, rng: _b2r(jpeg_compression(_b2r(im), quality=50)),
    "Median_Filter": lambda im, rng: _b2r(median_filter(_b2r(im))),
    "Contrast":      lambda im, rng: _b2r(contrast(_b2r(im), alpha=1.25)),
}
ATTACK_LABEL = {"Salt_Pepper": "Salt & pepper (density 0.02)", "JPEG": "JPEG (quality 50)",
                "Median_Filter": "Median filter (3x3)", "Contrast": "Contrast (alpha = 1.25)"}

# ---- 2) the lion image (the held-out test image used as Image 1 in the paper) ------------------------------------------
_cand = sorted(PREPARED.rglob(f"{LION_STEM}.png"))
assert _cand, f"{LION_STEM}.png not found under {PREPARED}"
rgb = load_host_image(_cand[0], 512)

# ---- 3) embed -> attack -> extract -> decrypt, once per watermark -------------------------------------------
mw_results = {}
for k_i, (name, (bits, cond)) in enumerate(KINDS.items()):
    ctx = derive_wm_crypto(K_TF_bytes, f"{LION_STEM}-multi_{k_i}")    # own nonce per watermark
    enc_bits = gcm_encrypt_bits(bits, ctx)[0]
    bitsR = np.repeat(enc_bits, R_)
    blocks, _ = select_blocks_tfqkd(K_TF_bytes, LION_STEM, 512, BLOCK_SIZE, enc_bits.size * R_)
    rows, cols = blocks_rc(blocks)
    sem = semantic_norm_selected(rgb, blocks)
    th = image_thresholds(rgb, rows, cols, sem, FZ["T0"])
    wm_img = watermark_image(rgb, bitsR, rows, cols, th, FZ["margin"], FZ["cap"])
    att = ATTACK_FN[cond](wm_img, np.random.default_rng(GLOBAL_SEED))
    recv = att                                                                              # same receiver rule as Phase 5: no attack-specific step
    pt = ctr_decrypt_bits(majority_vote_bits(extract_raw_bits(recv, rows, cols), R_), ctx)
    mw_results[name] = dict(bits=bits, cond=cond, wm_img=wm_img, att=att, rec_bits=np.asarray(pt, dtype=np.uint8),
                         psnr=psnr_rgb(rgb, wm_img), nc=float(ncc_bits(pt, bits)), ber=float(ber_bits(pt, bits)))

TAB = pd.DataFrame([dict(watermark=n, attack=ATTACK_LABEL[r["cond"]], PSNR_dB=r["psnr"], NC=r["nc"], BER=r["ber"]) for n, r in mw_results.items()])
TAB.to_csv(P7_ROOT / "multi_watermark_lion.csv", index=False); display(TAB.round(4))

# ---- 3b) second CSV: detailed per-watermark results (incl. the image file names used in the LaTeX table) -------------
KEY = {"Eagle logo": "eagle", "Text": "text", "Texture": "texture", "Random bits": "random"}
DET = []
for n, r in mw_results.items():
    n_err = int(np.sum(np.asarray(r["bits"], dtype=np.uint8) != r["rec_bits"]))
    DET.append(dict(
        image=LION_STEM, watermark=n, attack=ATTACK_LABEL[r["cond"]],
        PSNR_dB=round(r["psnr"], 4), NC=round(r["nc"], 4), BER=round(r["ber"], 4),
        bit_errors=n_err, total_bits=int(len(r["bits"])), exact_recovery=(n_err == 0),
        recovered_text=bits_to_text(r["rec_bits"], len(TEXT_MESSAGE)) if n == "Text" else "",
        file_original=f"{KEY[n]}_1_original.png", file_watermarked=f"{KEY[n]}_2_watermarked.png",
        file_attacked=f"{KEY[n]}_3_attacked.png", file_extracted=f"{KEY[n]}_4_extracted.png"))
DET = pd.DataFrame(DET)
DET.to_csv(P7_ROOT / "multi_watermark_lion_detailed.csv", index=False); display(DET)

# ---- 4) save every panel of the figure as its own image ---------------------------------------------------------------
from PIL import Image
IMG_DIR = P7_ROOT / "multi_watermark_images"; IMG_DIR.mkdir(parents=True, exist_ok=True)

def text_layout(msg, width=14):                           # word-wrap once on the ORIGINAL text -> (line, col) per character index
    pos, line, col, words, i = {}, 0, 0, msg.split(" "), 0
    for w_i, w in enumerate(words):
        need = len(w) + (1 if w_i < len(words) - 1 else 0)
        if col > 0 and col + len(w) > width: line, col = line + 1, 0
        for ch_i in range(len(w)): pos[i + ch_i] = (line, col + ch_i)
        i += len(w) + 1; col += need
    return pos

def save_text(msg, ref, path, width=14, fs=16):           # square monospaced text panel; wrong characters in red
    f, ax = plt.subplots(figsize=(2.2, 2.2)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal"); ax.axis("off")
    pos = text_layout(TEXT_MESSAGE, width); n_lines = max(l for l, _ in pos.values()) + 1
    for i, (li, ci) in pos.items():
        bad = ref is not None and ref[i] != msg[i]
        ax.text(0.5 + (ci - (width - 1) / 2) / (width + 1.2), 0.5 + (n_lines - 1) / 2 * 0.2 - li * 0.2, msg[i], family="monospace",
                fontsize=fs, fontweight="bold", ha="center", va="center", color="red" if bad else "black")
    f.subplots_adjust(0, 0, 1, 1); f.savefig(path, dpi=300); plt.close(f)

def save_bits(bits, path, scale=16):                       # 32x32 binary watermark -> 512x512 PNG, hard pixels
    a = (np.asarray(bits, dtype=np.uint8).reshape(32, 32) * 255).astype(np.uint8)
    Image.fromarray(a).resize((32 * scale, 32 * scale), Image.NEAREST).save(path)

def save_rgb(img, path): Image.fromarray(np.asarray(img).astype(np.uint8)).save(path)

for n, r in mw_results.items():
    k = KEY[n]
    if n == "Text":
        save_text(TEXT_MESSAGE, None, IMG_DIR / f"{k}_1_original.png")
        save_text(bits_to_text(r["rec_bits"], len(TEXT_MESSAGE)), TEXT_MESSAGE, IMG_DIR / f"{k}_4_extracted.png")
    else:
        save_bits(r["bits"],     IMG_DIR / f"{k}_1_original.png")
        save_bits(r["rec_bits"], IMG_DIR / f"{k}_4_extracted.png")
    save_rgb(r["wm_img"], IMG_DIR / f"{k}_2_watermarked.png")
    save_rgb(r["att"],    IMG_DIR / f"{k}_3_attacked.png")

print("images saved in:", IMG_DIR, f"({len(list(IMG_DIR.glob('*.png')))} files)")
print(*sorted(p.name for p in IMG_DIR.glob("*.png")), sep="\n")

In [ ]:
# =============================================================================
# 7.4  Image-to-dataset mapping ("Image 1-10" in the manuscript)  +  7.5 comparison  +  7.6 manuscript numbers
# =============================================================================
MANUSCRIPT_IMAGES = ["Au_ani_00030", "Au_arc_00009", "Au_art_00029", "Au_cha_10127", "Au_ind_00002",
                     "Au_nat_00025", "Au_pla_00085", "Au_sec_00029", "Au_txt_00001", "Au_ani_00092"]
mp = []
for i, st in enumerate(MANUSCRIPT_IMAGES, 1):
    hit = sorted(DATASET_DIR.rglob(f"{st}.*"))
    prep = sorted(PREPARED.rglob(f"{st}*"))
    mp.append(dict(manuscript_label=f"Image {i}", casia_file=st, casia_path=str(hit[0]) if hit else "NOT FOUND",
                   in_prepared_split=str(prep[0].parent.parent.name) if prep else "not in prepared set"))
pd.DataFrame(mp).to_csv(P7_ROOT / "image_to_dataset_mapping.csv", index=False)
display(pd.DataFrame(mp))

# ---- 7.5 comparison ----
q = pd.read_csv(folder_result_dir / "folder_level_quality_metrics.csv")
ben = P5[(~P5.Is_Malicious) & (P5.Attack != "Clean")]
ours = dict(Method="Proposed (this run)", PSNR=q.psnr_db.mean(), SSIM=q.ssim.mean(), NC=ben.NC.mean(), BER=ben.BER.mean(), Capacity_bits=int(W_enc_bits.size))
lit_f = ROOT / "literature_comparison.csv"
if lit_f.exists():
    COMP = pd.concat([pd.read_csv(lit_f), pd.DataFrame([ours])], ignore_index=True)
else:
    COMP = pd.DataFrame([ours]); print("No literature_comparison.csv in ROOT -> table holds only this run. Add one (columns: Method,PSNR,SSIM,NC,BER,Capacity_bits) with values taken from the cited papers.")
COMP.to_csv(P7_ROOT / "comparison_table.csv", index=False); display(COMP.round(4))

# ---- robustness figure ----
fig, ax = plt.subplots(figsize=(11, 4))
s_ = P5_summary.drop(index=FORGERIES, errors="ignore")
ax.bar(range(len(s_)), s_.NC_mean, yerr=s_.NC_std.fillna(0), capsize=2)
ax.set_xticks(range(len(s_))); ax.set_xticklabels(s_.index, rotation=70, ha="right", fontsize=8); ax.set_ylim(0, 1.05)
ax.set_ylabel("mean NC"); ax.set_title("Watermark robustness per attack (decrypted NC)")
fig.tight_layout(); fig.savefig(P7_ROOT / "robustness_per_attack.png", dpi=300); plt.show()

# ---- 7.6 headline numbers for the manuscript ----
fz = json.load(open(FROZEN_PARAMS_FILE))
best_fusion = AUTH_TABLE.set_index("method")
NUMBERS = dict(
    embedding=dict(T0=EMBEDDING_THRESHOLD, margin=EMBED_MARGIN, cap=DIFF_CAP, R=REDUNDANCY, target_psnr=TARGET_PSNR, target_reached=fz.get("target_reached")),
    imperceptibility=dict(psnr_mean=float(q.psnr_db.mean()), psnr_std=float(q.psnr_db.std()), ssim_mean=float(q.ssim.mean()), mse_mean=float(q.mse.mean()), n_images=int(len(q))),
    clean_extraction=dict(nc_mean=float(P5[P5.Attack == "Clean"].NC.mean()), ber_mean=float(P5[P5.Attack == "Clean"].BER.mean()), tag_ok_rate=float(P5[P5.Attack == "Clean"].tag_ok.mean())),
    benign_attacks=dict(nc_mean=float(ben.NC.mean()), ber_mean=float(ben.BER.mean()), per_attack_nc=P5_summary.NC_mean.drop(index=FORGERIES, errors="ignore").round(4).to_dict()),
    authentication=dict(thresholds=dict(T_NC=T_NC, T_SIAM=T_SIAM, phase0=PHASE0_SIAM_THRESHOLD), auc_watermark=AUC_WM, auc_siamese=AUC_SIAM,
                        metrics=best_fusion.round(4).to_dict(orient="index")),
    localization=dict(alpha=LOC_ALPHA, min_rate=LOC_MIN_RATE, constraint_met=LOC_CONSTRAINT_MET, held_out=LOC_TEST),
    security=dict(monobit_p=SECURITY["monobit_p"], runs_p=SECURITY["runs_p"], wrong_key=SECURITY["wrong_key_one_bit_flip"]),
    split=dict(validation_images=len(VAL_STEMS), test_images=len(TEST_STEMS)),
)
json.dump(NUMBERS, open(P7_ROOT / "manuscript_numbers.json", "w"), indent=2, default=float)
print(json.dumps({k: NUMBERS[k] for k in ("embedding", "imperceptibility", "clean_extraction")}, indent=2, default=float))
print("\nAll Phase 7 outputs ->", P7_ROOT)

## 7.7 — Figure exports
Images for the imperceptibility figure (ten manuscript images: host, watermarked, difference ×10).

In [ ]:
# =============================================================================
# 7.7  Export the images used in the imperceptibility figure (Figure 11):
#      for each of the ten manuscript images -> host, watermarked, and |difference| x10
# =============================================================================
FIG11_DIR = P7_ROOT / "imperceptibility_10_export"; FIG11_DIR.mkdir(parents=True, exist_ok=True)
rows_f11 = []
for i, st in enumerate(MANUSCRIPT_IMAGES, 1):
    if st not in embedding_maps:
        print(f"Image {i} ({st}): not in the pipeline results -> skipped"); continue
    host = load_host_image(embedding_maps[st]["image_path"], 512)
    wm_p = watermarked_dir / f"{st}_watermarked.png"
    wm = np.array(Image.open(wm_p).convert("RGB"))
    diff = np.clip(np.abs(host.astype(np.int16) - wm.astype(np.int16)) * 10, 0, 255).astype(np.uint8)
    Image.fromarray(host).save(FIG11_DIR / f"{st}_host.png")
    Image.fromarray(wm).save(FIG11_DIR / f"{st}_watermarked.png")
    Image.fromarray(diff).save(FIG11_DIR / f"{st}_diff_x10.png")
    rows_f11.append(dict(manuscript_label=f"Image {i}", image=st, PSNR_dB=psnr_rgb(host, wm), SSIM=ssim_rgb(host, wm)))
FIG11_TAB = pd.DataFrame(rows_f11); FIG11_TAB.to_csv(P7_ROOT / "imperceptibility_10_images.csv", index=False)
display(FIG11_TAB.round(4))
print("images saved in:", FIG11_DIR)

In [ ]:
# =============================================================================
# 7.8  Export localization panels for the manuscript images (Image 1 and Image 2)
# =============================================================================
PANEL_DIR = LOC_ROOT / "panels"; PANEL_DIR.mkdir(exist_ok=True)

def save_panel(arr, path, cmap=None, vmin=None, vmax=None):
    a = np.asarray(arr)
    if a.ndim == 3:                                   # colour image
        if a.dtype != np.uint8:
            a = (np.clip(a, 0, 1) * 255).astype(np.uint8) if a.max() <= 1.0 else a.astype(np.uint8)
        cv2.imwrite(str(path), cv2.cvtColor(a, cv2.COLOR_RGB2BGR))
    else:                                             # mask -> 512x512, crisp cells
        a = cv2.resize(a.astype(np.float32), (512, 512), interpolation=cv2.INTER_NEAREST)
        plt.imsave(str(path), a, cmap=cmap, vmin=vmin, vmax=vmax)

short = {"Copy_Move_Forgery": "copymove", "Image_Splicing": "splicing",
         "Object_Removal": "objremoval", "Region_Replacement": "regionrepl"}
SELECTED = [("image1", "Au_ani_00030"), ("image2", "Au_arc_00009")]   # Image 1 and Image 2 of the paper

for tag, s in SELECTED:
    if s not in embedding_maps:
        print(f"{tag}: {s} not found in the pipeline results -> skipped"); continue
    print(f"{tag}: {s}  ({'held-out test' if s in TEST_STEMS else 'validation'} identity)")
    for c in FORGERIES:
        if s not in RAW_BITS[c]:
            print(f"   {c}: no extracted bits -> skipped"); continue
        cells = run_loc([s], LOC_ALPHA, LOC_MIN_RATE, [c])[(s, c)]
        im = load_rgb512(received_path(s, c))
        gt = flagged_to_mask(gt_cells(s, c))
        pr = flagged_to_mask(cells)
        base = PANEL_DIR / f"loc_{short[c]}_{tag}"
        save_panel(im, f"{base}_forged.png")
        save_panel(gt, f"{base}_groundtruth.png", cmap="gray", vmin=0, vmax=1)
        save_panel(pr, f"{base}_predicted.png", cmap="gray", vmin=0, vmax=1)
print("done ->", PANEL_DIR)

## Done
Everything for the paper is in `PROPOSED_SCHEME_RESULTS/PHASE7_ANALYSIS/` (`manuscript_numbers.json`, the CSV tables and the figure images).